In [ ]:
library("scDblFinder")
library("scds")
library("Seurat")
library("tidyverse")
library("viridis")
library("cowplot")
library('ggpubr')
library('harmony')
library('msigdbr')
library('GSVA')
library('RColorBrewer')
library('fgsea')

# Define global variable

In [ ]:
tumor_dataset_collection <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection"
health_dataset_collection <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/nat_med_2023_Lisa"
figures <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/figures"
obj <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects"

In [ ]:
age_group_color <- c('Young' = '#E5368E', 'Old' = '#3C7BB0')

In [ ]:
# ltc_palettes package
main_celltype_colors <- c('T_Cell' = '#9b2226', 'NK' = '#D55D4C', 'ILC' = '#EA6175', 'B_Cell' = '#ca6702', 'Plasma' = '#ee9b00',
                          'Macro' = '#005F73', 'Mono' = '#0a9396', 'DC' = '#94d2bd', 'Mast' = '#66679C', 'Neutrophil' = '#e9d8a6',
                          'Endo' = '#C2C1E0', 'Tumor' = '#B2CAEE', 'Mural' = '#D17C7D', 'Fibro' = '#F3BAA5', 'Epi' = '#67ADB7', 'Imm' = '#C17F9E')

#'NK' = '#f897a1'

In [ ]:
imm_subcelltype_colors <- c(Memory_B = "#2d6037", Naive_B = "#64AE59", Plasma_B = "#839098", CCR7_CD4_Tnaive = "#ECA8A9",
                            GZMK_CD8_Tem = "#74AED4", ZNF683_CD8_Trm = "#67ADB7",CXCR6_CD4_Trm = "#E4A6BD", ISG15_Teffector = '#B3B2B3',
                            MAIT = "#F3D8E1", FOXP3_CD4_Treg = "#009170", CXCL13_CD4_Tex = "#78A040", IFITM3_CD8_Teffector = '#839098',
                            GZMB_CD8_Teffector = "#2E75AB", CXCL13_CD8_Tex = "#009393",FGFBP2_NK = "#B06E3C", XCL1_NK = "#5AA2DA",
                            NKT = "#FBD8A2", ILC = "#B84848", Mast = "#6567A0", Undetermined = "#87C3EC",
                            pDC = "#BDE6FA", cDC1 = "#D7EFFB", cDC2 = "#6EB1DE", LAMP3_DC = "#92C2DD", Neutrophil = "#4A94C6",
                            PPARG_Mono = "#FADED2", CD16_Mono = "#FAC7B3", PPARG_Macro = "#F0A29B", SPP1_Macro = "#B389B9",
                            LGMN_Macro = "#CC7892", FABP4_Macro = "#E2A2B3", CD14_Mono = "#F3C6C1", MIF_Macro = "#89558D")

In [ ]:
transparent_bg <- theme(panel.background = element_rect(fill = NA, colour = NA),
                        plot.background = element_rect(fill = NA, colour = NA),
                        legend.box.background = element_rect(fill = NA, colour = NA),
                        legend.background = element_rect(fill = NA, colour = NA))

In [ ]:
# function for volcano plot
volcano_plot <- function (DEGs_df, col_names_pval, col_names_LFC, pval_cutoff, title, label_logFC_cutoff, label_pval_cutoff, logFC_cutoff = 1){
pvals <- DEGs_df[[col_names_pval]]
logFC <- DEGs_df[[col_names_LFC]]
row_names <- rownames(DEGs_df)

# Create a data frame with p-values and log2-fold changes

df <- data.frame(pvals, logFC, row.names = row_names)

#ranmodly set the -log(pvals) if the original p value is 0
df <- mutate(df, random = runif(n(), min = 1.5, max = 3), log_pvals = ifelse(pvals == 0, random*100, -log10(pvals)))

    
# Set significance threshold and log2-fold change threshold
pval_cutoff <- pval_cutoff
logFC_cutoff <- logFC_cutoff

# Add columns indicating the significance and direction of change
df$significant <- df$pvals < pval_cutoff & abs(df$logFC) > logFC_cutoff
df$direction <- ifelse(df$logFC > 0, "Up", "Down")
df$label <- ifelse(abs(df$logFC) > label_logFC_cutoff & df$pvals < label_pval_cutoff, rownames(df), "")
df$sig_dir <- ifelse(df$significant, df$direction, "NonSignificant")
df$sig_dir <- factor(df$sig_dir, levels = c('Up', 'NonSignificant', 'Down'))

# Create the plot
ggplot(df, aes(x=logFC, y=log_pvals, color=sig_dir, label= label)) +
    geom_point(size=4) +
    ggrepel::geom_text_repel(max.overlaps = 700, nudge_y = 0.1, size = 6, show.legend = F) +
    scale_color_manual(values=c('Down' = '#3C7BB0', 'NonSignificant' = "#D1D4D3", 'Up' = '#E5368E')) +
#    scale_color_manual(values=c('Down' = '#76A6CE', 'NonSignificant' = "#D1D4D3", 'Up' = '#DA7096')) +
    labs(x="Log2-Fold Change", y="-Log10(Padj)", title=title) +
    geom_hline(yintercept = -log10(pval_cutoff), linetype = "dashed") + 
    geom_vline(xintercept = c(-logFC_cutoff, logFC_cutoff), linetype = "dashed")  +
    guides(color = guide_legend(title = "Log2-FoldChange", direction = 'horizontal', title.theme = element_text(size = 20))) +
    theme_classic(base_size = 25) +
    theme(plot.title = element_text(hjust = 0.5),
          legend.position = 'top',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))
}
#4A94C6

# Samples_all_sc_analysis

## Data load and preprocess

### Data load: Health 

In [ ]:
# RenameGenesSeurat  ------------------------------------------------------------------------------------
RenameGenesSeurat <- function(obj, newnames) { # Replace gene names in different slots of a Seurat object. Run this before integration. Run this before integration. It only changes obj@assays$RNA@counts, @data and @scale.data.
  print("Run this before integration. It only changes obj@assays$RNA@counts, @data and @scale.data.")
  RNA <- obj@assays$RNA

  if (nrow(RNA) == length(newnames)) {
    if (length(RNA@counts)) RNA@counts@Dimnames[[1]]            <- newnames
    if (length(RNA@data)) RNA@data@Dimnames[[1]]                <- newnames
#    if (length(RNA@scale.data)) RNA@scale.data@Dimnames[[1]]    <- newnames
  } else {"Unequal gene sets: nrow(RNA) != nrow(newnames)"}
  obj@assays$RNA <- RNA
  return(obj)
}
# RenameGenesSeurat(obj = SeuratObj, newnames = HGNC.updated.genes)

In [ ]:
# health_samples_YO from HLCA_core
health_samples_YO <- schard::h5ad2seurat(filename = paste0(health_dataset_collection, '/objects/HLCA_core_YO_health_selected.h5ad'), use.raw =T, load.obsm = F)
health_samples_YO$orig.ident <- health_samples_YO$sample

In [ ]:
# renameGene
health_samples_YO <- RenameGenesSeurat(obj = health_samples_YO,
                                       newnames = health_samples_YO@assays$RNA@meta.features$feature_name)

### Data load: Tumor

In [ ]:
##load the data from raw fastq based cellranger of emm_2022_taojiang dataset
emm_2022_taojiang_dir <- paste0(tumor_dataset_collection, "/emm_2022_taojiang/cellranger_outs")
emm_2022_taojiang_sample <-c("TD3" = "SRR17008553", "TD6" = "SRR17008556", 
                             "TD7" = "SRR17008557", "TD8" = "SRR17008558", "TD9" = "SRR17008559")

#
for ( i in seq_along(emm_2022_taojiang_sample)) {
    print(paste0(emm_2022_taojiang_dir, "/", emm_2022_taojiang_sample[i], "/", emm_2022_taojiang_sample[i], "/outs/filtered_feature_bc_matrix"))
    
    assign(paste0("emm_2022_taojiang_", names(emm_2022_taojiang_sample[i]), "_matrix"), 
           Read10X(data.dir = paste0(emm_2022_taojiang_dir, "/", emm_2022_taojiang_sample[i], "/", emm_2022_taojiang_sample[i], "/outs/filtered_feature_bc_matrix")))
    
    assign(paste0("emm_2022_taojiang_", names(emm_2022_taojiang_sample[i])), 
           CreateSeuratObject(get(paste0("emm_2022_taojiang_", names(emm_2022_taojiang_sample[i]), "_matrix")), project = paste0("emm_2022_taojiang_", names(emm_2022_taojiang_sample[i])), min.cells = 20))
}

In [ ]:
##load the data from raw fastq based cellranger of stm_2022_yuxin dataset
stm_2022_yuxin_dir <- paste0(tumor_dataset_collection, "/stm_2022_yuxin_yin/cellranger_outs")
stm_2022_yuxin_sample <-c("LC1" = "HRR059414", "LC17" = "HRR059417", "LC26" = "HRR059418")

#
for ( i in seq_along(stm_2022_yuxin_sample)) {
    print(paste0(stm_2022_yuxin_dir, "/", stm_2022_yuxin_sample[i], "/", stm_2022_yuxin_sample[i], "/outs/filtered_feature_bc_matrix"))
    
    assign(paste0("stm_2022_yuxin_", names(stm_2022_yuxin_sample[i]), "_matrix"), 
           Read10X(data.dir = paste0(stm_2022_yuxin_dir, "/", stm_2022_yuxin_sample[i], "/", stm_2022_yuxin_sample[i], "/outs/filtered_feature_bc_matrix")))
    
    assign(paste0("stm_2022_yuxin_", names(stm_2022_yuxin_sample[i])), 
           CreateSeuratObject(get(paste0("stm_2022_yuxin_", names(stm_2022_yuxin_sample[i]), "_matrix")), project = paste0("stm_2022_yuxin_", names(stm_2022_yuxin_sample[i])), min.cells = 20))
}

In [ ]:
##load the data from expression matrix based Rdata object of sttt_2022_Weimin_li
sttt_2022_Weimin_sample <- c("PA09", "PA13", "PA18")

#the name of this seurat object is main_tiss
load("/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/sttt_2022_Weimin_Li/objects/01_unfiltered.RData")  

for ( i in sttt_2022_Weimin_sample) {
    assign(paste0("sttt_2022_Weimin_", i), subset(main_tiss, PatientID == i & Tissue == "Tumor" & Batch == 1))
}

Project(sttt_2022_Weimin_PA09) <- "sttt_2022_Weimin_PA09"
Project(sttt_2022_Weimin_PA13) <- "sttt_2022_Weimin_PA13"
Project(sttt_2022_Weimin_PA18) <- "sttt_2022_Weimin_PA18"

In [ ]:
#load in-house data from cellranger of ylac_23k_dataset and olac
ylac_23k_counts <- Read10X(paste0(tumor_dataset_collection, '/YLAC/Cellranger/YLAC1-23K/outs/filtered_feature_bc_matrix'))
ylac_23k <- CreateSeuratObject(counts = ylac_23k_counts, project = "ylac_23k_t", min.cells = 20)

olac_dir <- paste0(tumor_dataset_collection, "/OLAC/cellranger")
olac_samples <- c("olac2" = "SCRNA2", "olac4" = "SCRNA4")
for ( i in seq_along(olac_samples)) {
    print(paste0(olac_dir, "/", olac_samples[i], "/outs/filtered_feature_bc_matrix"))
    
    assign(paste0("OLAC_", names(olac_samples[i]), "_matrix"), 
           Read10X(data.dir = paste0(olac_dir, "/", olac_samples[i], "/outs/filtered_feature_bc_matrix")))
    
    assign(paste0("OLAC_", names(olac_samples[i])), 
           CreateSeuratObject(get(paste0("OLAC_", names(olac_samples[i]), "_matrix")), project = paste0("OLAC_", names(olac_samples[i])), min.cells = 20))
}


In [ ]:
##load the data from raw fastq based cellranger of nc_2021_zhoufeng dataset
nc_2021_zhoufeng_dir <- paste0(tumor_dataset_collection, "/nc_2021_zhoufeng/cellranger_outs/cellranger_count_samples_with_expected_cells")
nc_2021_zhoufeng_sample <-c("P4T", 'P7T1', 'P7T2', 'P8T1', 'P8T2', 'P17T', 'P18T')

#
for ( i in seq_along(nc_2021_zhoufeng_sample)) {
    print(paste0(nc_2021_zhoufeng_dir, "/", nc_2021_zhoufeng_sample[i], "/outs/filtered_feature_bc_matrix"))
    
    assign(paste0("nc_2021_zhoufeng_", nc_2021_zhoufeng_sample[i], "_matrix"), 
           Read10X(data.dir = paste0(nc_2021_zhoufeng_dir, "/", nc_2021_zhoufeng_sample[i], "/outs/filtered_feature_bc_matrix")))
    
    assign(paste0("nc_2021_zhoufeng_", nc_2021_zhoufeng_sample[i]), 
           CreateSeuratObject(get(paste0("nc_2021_zhoufeng_", nc_2021_zhoufeng_sample[i], "_matrix")), project = paste0("nc_2021_zhoufeng_", nc_2021_zhoufeng_sample[i]), min.cells = 20))
}

In [ ]:
tumor_samples_list <- list("emm_2022_taojiang_TD3" = emm_2022_taojiang_TD3, "emm_2022_taojiang_TD6" = emm_2022_taojiang_TD6, 
                           "emm_2022_taojiang_TD7" = emm_2022_taojiang_TD7, "emm_2022_taojiang_TD8" = emm_2022_taojiang_TD8, "emm_2022_taojiang_TD9" = emm_2022_taojiang_TD9, 
                           "stm_2022_yuxin_LC1" = stm_2022_yuxin_LC1, "stm_2022_yuxin_LC17" = stm_2022_yuxin_LC17, "stm_2022_yuxin_LC26" = stm_2022_yuxin_LC26, 
                           "sttt_2022_Weimin_PA09" = sttt_2022_Weimin_PA09, "sttt_2022_Weimin_PA13" = sttt_2022_Weimin_PA13, "sttt_2022_Weimin_PA18" = sttt_2022_Weimin_PA18,
                           "nc_2021_zhoufeng_P4T" = nc_2021_zhoufeng_P4T, "nc_2021_zhoufeng_P7T1" = nc_2021_zhoufeng_P7T1, "nc_2021_zhoufeng_P7T2" = nc_2021_zhoufeng_P7T2, 
                           "nc_2021_zhoufeng_P8T1" = nc_2021_zhoufeng_P8T1, "nc_2021_zhoufeng_P8T2" = nc_2021_zhoufeng_P8T2, "nc_2021_zhoufeng_P17T" = nc_2021_zhoufeng_P17T, "nc_2021_zhoufeng_P18T" = nc_2021_zhoufeng_P18T, 
                           "ylac_23k" = ylac_23k, "OLAC_olac2" = OLAC_olac2, "OLAC_olac4" = OLAC_olac4)

In [ ]:
##set the orig.ident and project names for sttt_2022_Weimin dataset
for ( i in c("sttt_2022_Weimin_PA09", "sttt_2022_Weimin_PA13", "sttt_2022_Weimin_PA18")) {
    tumor_samples_list[[i]]$"orig.ident" <- as.factor(i)
    Project(tumor_samples_list[[i]]) <- i
}

In [ ]:
#the summary of nFeature_RNA
for ( i in seq_along(tumor_samples_list) ) {
    print(paste0("The sample is ", names(tumor_samples_list[i])))
    print(summary(tumor_samples_list[[i]]$nFeature_RNA))
}

In [ ]:
#the summary of nCount_RNA
for ( i in seq_along(tumor_samples_list) ) {
    print(paste0("The sample is ", names(tumor_samples_list[i])))
    print(summary(tumor_samples_list[[i]]$nCount_RNA))
}

In [ ]:
n_cells <- 0
for ( i in tumor_samples_list) {
    n_cells <- nrow(i[[]]) + n_cells
}

n_cells

### Delete doublet cell
- only for tumor samples; the health samples from HLCA core undergo doublet detection

In [ ]:
### determine the multiplets using scDblFinder (the dataset does not contain any empty drops, but hasn't been further filtered; be necessary to remove cells with a very low coverage (e.g. <200 reads) to avoid errors)

### determine the multiplets using scds (identify doublets in two complementary ways: cxds and bcds)

tumor_samples_sce_list <- lapply(X = tumor_samples_list, FUN = function (x) {
    
    x <- as.SingleCellExperiment( x, assay = "RNA")
    
    set.seed(1234)
    x <- scDblFinder(x)
    
    set.seed(1234)
    x <- cxds_bcds_hybrid(x, estNdbl = TRUE)
})

In [ ]:
### subset the singlet
tumor_samples_singlet_list <- lapply(X = tumor_samples_sce_list, FUN = function (x) {
    
    print(table(x$scDblFinder.class, x$hybrid_call))
    x <- intersect(rownames(subset(colData(x), scDblFinder.class == "singlet")), rownames(subset(colData(x), !hybrid_call)))
    
})

### subset the singlet
for ( i in seq_along(tumor_samples_list)) {
    tumor_samples_list[[i]] <- subset(tumor_samples_list[[i]], cells = tumor_samples_singlet_list[[names(tumor_samples_list[i])]])
}

In [ ]:
saveRDS(tumor_samples_list, paste0(obj, "/", "tumor_samples_list_afterDoublet.rds"))

In [ ]:
print("done_20260827")

In [ ]:
n_cells <- 0
for ( i in tumor_samples_list) {
    n_cells <- nrow(i[[]]) + n_cells
}

n_cells

### Quality control

In [ ]:
### qc plot functions
qc_std_plot_helper <- function(x) {x + 
    scale_color_viridis() +
    geom_point(size = 0.01, alpha = 0.3)
}

qc_std_plot <- function(seu_obj) {
  qc_data <- as_tibble(FetchData(seu_obj, c("nCount_RNA", "nFeature_RNA", "pMT", "pHB")))
  plot_grid(
    
    qc_std_plot_helper(ggplot(qc_data, aes(log2(nCount_RNA), log2(nFeature_RNA), color = pMT))) + 
      geom_hline(yintercept = log2(nFeature_lower), color = "red", linetype = 2) +
      geom_hline(yintercept = log2(nFeature_upper), color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nCount_lower), color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nCount_upper), color = "red", linetype = 2),
    qc_std_plot_helper(ggplot(qc_data, aes(log2(nCount_RNA), log2(nFeature_RNA), color = pHB))) + 
      geom_hline(yintercept = log2(nFeature_lower), color = "red", linetype = 2) +
      geom_hline(yintercept = log2(nFeature_upper), color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nCount_lower), color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nCount_upper), color = "red", linetype = 2),
    
    qc_std_plot_helper(ggplot(qc_data, aes(log2(nCount_RNA), pMT, color = nFeature_RNA))) + 
      geom_hline(yintercept = pMT_lower, color = "red", linetype = 2) +
      geom_hline(yintercept = pMT_upper, color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nCount_lower), color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nCount_upper), color = "red", linetype = 2),
    
    qc_std_plot_helper(ggplot(qc_data, aes(log2(nFeature_RNA), pMT, color = nCount_RNA))) + 
      geom_hline(yintercept = pMT_lower, color = "red", linetype = 2) +
      geom_hline(yintercept = pMT_upper, color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nFeature_lower), color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nFeature_upper), color = "red", linetype = 2),
      
    qc_std_plot_helper(ggplot(qc_data, aes(log2(nCount_RNA), pHB, color = nFeature_RNA))) + 
      geom_hline(yintercept = pHB_lower, color = "red", linetype = 2) +
      geom_hline(yintercept = pHB_upper, color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nCount_lower), color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nCount_upper), color = "red", linetype = 2),
      
    qc_std_plot_helper(ggplot(qc_data, aes(log2(nFeature_RNA), pHB, color = nCount_RNA))) + 
      geom_hline(yintercept = pHB_lower, color = "red", linetype = 2) +
      geom_hline(yintercept = pHB_upper, color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nFeature_lower), color = "red", linetype = 2) +
      geom_vline(xintercept = log2(nFeature_upper), color = "red", linetype = 2),

    
    ncol = 2, align = "hv"
  )
}

In [ ]:
# health_tumor_samples_list for further quality control
Project(health_samples_YO) <- 'health_samples_YO'
tumor_samples_list[['health_samples_YO']] = health_samples_YO
health_tumor_samples_list <- tumor_samples_list

In [ ]:
# data qc: calculate pMT, pHB
health_tumor_samples_list <- lapply(X = health_tumor_samples_list, FUN = function (x) {
    x <- PercentageFeatureSet(x, pattern = "^MT-", col.name = "pMT")
    x <- PercentageFeatureSet(x, pattern = "^HBA|^HBB", col.name = "pHB")
})

In [ ]:
####ploting before qc # the lower bound is set as 300

#qc parameters
nFeature_lower <- 300        # the lower bound is set as 300
nFeature_upper <- 6500
nCount_lower <- 1000
nCount_upper <- Inf
pMT_lower <- 0
pMT_upper <- 12
pHB_lower <- 0
pHB_upper <- 5

In [ ]:
# filter based on n_feature, n_count, pMT, pHB
for (i in seq_along(health_tumor_samples_list)) {
    health_tumor_samples_list[[i]] <- subset(health_tumor_samples_list[[i]], subset = nFeature_RNA > nFeature_lower & nFeature_RNA < nFeature_upper & nCount_RNA > nCount_lower & nCount_RNA < nCount_upper & pMT < pMT_upper & pHB < pHB_upper)
}

In [ ]:
n_cells <- 0
for ( i in health_tumor_samples_list) {
    n_cells <- nrow(i[[]]) + n_cells
}

n_cells

### Preprocess

In [ ]:
#cell_cycle_score
s.genes <- cc.genes.updated.2019$s.genes
g2m.genes <- cc.genes.updated.2019$g2m.genes

for (i in seq_along(health_tumor_samples_list)) {
    health_tumor_samples_list[[i]] <- CellCycleScoring(object = health_tumor_samples_list[[i]], s.features = s.genes, g2m.features = g2m.genes, set.ident = FALSE)
    health_tumor_samples_list[[i]]$CC.Difference <- health_tumor_samples_list[[i]]$S.Score - health_tumor_samples_list[[i]]$G2M.Score
}

In [ ]:
#remove mitochondrial and ribosome genes
for ( i in seq_along(health_tumor_samples_list)) {
    health_tumor_samples_list[[i]] <- health_tumor_samples_list[[i]][!grepl(pattern = "^MT-", x = rownames(health_tumor_samples_list[[i]])),]
    health_tumor_samples_list[[i]] <- health_tumor_samples_list[[i]][!grepl(pattern = "^RP([0-9]+-|[LS])", x = rownames(health_tumor_samples_list[[i]])),]
}

In [ ]:
# getting the union genes among all samples
union_genes <- rownames(health_tumor_samples_list[[1]])
for (sample in health_tumor_samples_list) {
    union_genes <- union(union_genes, rownames(sample))
}

length(union_genes)

In [ ]:
# get the shared genes among all samples
shared_genes <- rownames(health_tumor_samples_list[[1]])
for (sample in health_tumor_samples_list) {
    #cat(Project(sample), nrow(sample), '\n')
    shared_genes <- intersect(shared_genes, rownames(sample))
}

length(shared_genes)

In [ ]:
# calculate the gene frequency in all samples
gene_collection <- list()

for (i in seq_along(health_tumor_samples_list)) {
    gene_collection[[i]] <- rownames(health_tumor_samples_list[[i]])
}

gene_collection <- unlist(gene_collection)

gene_freq <- as.data.frame(table(gene_collection))


In [ ]:
health_tumor_samples_list %>% length()

In [ ]:
#retain the genes present in more than half of objects (>=11)
filtered_genes <- gene_freq[gene_freq$Freq >=11,][['gene_collection']]
length(filtered_genes)

for ( i in seq_along(health_tumor_samples_list)) {
    health_tumor_samples_list[[i]] <- health_tumor_samples_list[[i]][rownames(health_tumor_samples_list[[i]]) %in% filtered_genes,]
}

In [ ]:
# after the subset of features, meta.feature df have NA value
health_tumor_samples_list[['health_samples_YO']][['RNA']]@meta.features <- data.frame(row.names = rownames(health_tumor_samples_list[['health_samples_YO']][["RNA"]]))


In [ ]:
# merge all health_tumor_samples into one object
samples_all_merged <- merge(x = health_tumor_samples_list[[1]], y = health_tumor_samples_list[c(2:22)])

In [ ]:
# harmonize the metadata
samples_all_merged$SampleID <- NULL
samples_all_merged$Batch <- NULL
samples_all_merged$Tissue <- NULL
samples_all_merged$Cells <- NULL
samples_all_merged$PatientID <- NULL
samples_all_merged$Gender <- NULL
samples_all_merged$Disease <- NULL
samples_all_merged$Age <- NULL
samples_all_merged$T <- NULL
samples_all_merged$N <- NULL
samples_all_merged$M <- NULL
samples_all_merged$CellName <- NULL
samples_all_merged$X_index <- NULL
samples_all_merged$sample <- samples_all_merged$orig.ident
samples_all_merged$tissue <- 'lung parenchyma'
samples_all_merged$cause_of_death <- NULL
samples_all_merged$sequencing_platform <- NULL
samples_all_merged$age_range <- NULL

In [ ]:
samples_all_merged$sample %>% unique()

In [ ]:
# harmonize the metadata
samples_all_metadata <- samples_all_merged[[]] %>% 
    mutate(Stage = case_match(sample,
                              c("emm_2022_taojiang_TD7", "emm_2022_taojiang_TD8", "nc_2021_zhoufeng_P7T1", "nc_2021_zhoufeng_P8T2") ~ "AIS",
                              c("emm_2022_taojiang_TD3", "emm_2022_taojiang_TD6", "nc_2021_zhoufeng_P4T", "nc_2021_zhoufeng_P7T2",
                                "stm_2022_yuxin_LC1", "stm_2022_yuxin_LC17", "stm_2022_yuxin_LC26") ~ "MIA",
                              c("emm_2022_taojiang_TD9", "nc_2021_zhoufeng_P8T1", "nc_2021_zhoufeng_P17T", "nc_2021_zhoufeng_P18T",
                                "sttt_2022_Weimin_PA09", "sttt_2022_Weimin_PA13", "sttt_2022_Weimin_PA18", "ylac_23k_t", "OLAC_olac2", "OLAC_olac4") ~ "IAC",
                              .default = "Healthy")) %>%
    mutate(donor_id = case_when(sample %in% c('nc_2021_zhoufeng_P7T1', 'nc_2021_zhoufeng_P7T2') ~ 'nc_2021_zhoufeng_P7T',
                                sample %in% c('nc_2021_zhoufeng_P8T1', 'nc_2021_zhoufeng_P8T2') ~ 'nc_2021_zhoufeng_P8T',
                                !str_starts(sample, pattern = "emm|stm|sttt|nc|ylac|OLAC") ~ donor_id,
                                .default = sample)) %>%
    mutate(lung_condition = case_when(str_starts(sample, pattern = "emm|stm|sttt|nc|ylac|OLAC") ~ "Tumor",
                                      .default = lung_condition)) %>%
    mutate(disease = case_when(str_starts(sample, pattern = "emm|stm|sttt|nc|ylac|OLAC") ~ "LUAD",
                               .default = disease)) %>%
    mutate(age_or_mean_of_age_range = case_match(sample,
                                                 "emm_2022_taojiang_TD3" ~ 37, "emm_2022_taojiang_TD6" ~ 56,
                                                 "emm_2022_taojiang_TD7" ~ 40, "emm_2022_taojiang_TD8" ~ 69,
                                                 "emm_2022_taojiang_TD9" ~ 62, "stm_2022_yuxin_LC1" ~ 64,
                                                 "stm_2022_yuxin_LC17" ~ 80, "stm_2022_yuxin_LC26" ~ 37,
                                                 "sttt_2022_Weimin_PA09" ~ 78, "sttt_2022_Weimin_PA13" ~ 73,
                                                 "sttt_2022_Weimin_PA18" ~ 38, "nc_2021_zhoufeng_P4T" ~ 60,
                                                 c("nc_2021_zhoufeng_P7T1", "nc_2021_zhoufeng_P7T2") ~ 39,
                                                 c("nc_2021_zhoufeng_P8T1", "nc_2021_zhoufeng_P8T2") ~ 58,
                                                 "nc_2021_zhoufeng_P17T" ~ 38, "nc_2021_zhoufeng_P18T" ~ 61,
                                                 "ylac_23k_t" ~ 38, "OLAC_olac2" ~ 68, "OLAC_olac4" ~ 66,.default = age_or_mean_of_age_range)) %>%
    mutate(smoking_status = case_match(sample,
                                       c("sttt_2022_Weimin_PA09", 'sttt_2022_Weimin_PA13', 'sttt_2022_Weimin_PA18') ~ NA,
                                       c("emm_2022_taojiang_TD3", "emm_2022_taojiang_TD6", "emm_2022_taojiang_TD7",
                                         "emm_2022_taojiang_TD9", "stm_2022_yuxin_LC17", "stm_2022_yuxin_LC26",
                                         "nc_2021_zhoufeng_P8T1", "nc_2021_zhoufeng_P8T2", 'nc_2021_zhoufeng_P17T',
                                         "ylac_23k_t", "OLAC_olac2") ~ "never",
                                       c("emm_2022_taojiang_TD8", "stm_2022_yuxin_LC1") ~ "former",
                                       c("nc_2021_zhoufeng_P4T", "nc_2021_zhoufeng_P7T1", "nc_2021_zhoufeng_P7T2",
                                         "nc_2021_zhoufeng_P18T", "OLAC_olac4") ~ "active",
                                       .default = smoking_status)) %>%
    mutate(sex = case_match(sample,
                            c('sttt_2022_Weimin_PA13', 'emm_2022_taojiang_TD3', 'emm_2022_taojiang_TD8',
                              'stm_2022_yuxin_LC1', 'stm_2022_yuxin_LC17', 'nc_2021_zhoufeng_P4T',
                              'nc_2021_zhoufeng_P18T', 'ylac_23k_t', 'OLAC_olac4') ~ 'male',
                            c('sttt_2022_Weimin_PA09', 'sttt_2022_Weimin_PA18', 'emm_2022_taojiang_TD6',
                              'emm_2022_taojiang_TD7', 'emm_2022_taojiang_TD9', 'stm_2022_yuxin_LC26',
                              'nc_2021_zhoufeng_P7T1', 'nc_2021_zhoufeng_P7T2', 'nc_2021_zhoufeng_P8T1',
                              'nc_2021_zhoufeng_P8T2', 'nc_2021_zhoufeng_P17T', 'OLAC_olac2') ~ 'female',
                            .default = sex)) %>%
    mutate(subject_type = case_when(str_starts(sample, pattern = "emm|stm|sttt|nc|ylac|OLAC") ~ "surgery",
                               .default = subject_type)) %>%
    mutate(study = case_when(str_starts(sample, pattern = "emm") ~ 'emm_2022_taojiang',
                             str_starts(sample, pattern = "stm") ~ 'stm_2022_yuxin',
                             str_starts(sample, pattern = "nc") ~ 'nc_2021_zhoufeng',
                             str_starts(sample, pattern = "sttt") ~ 'sttt_2022_Weimin',
                             str_starts(sample, pattern = "ylac") ~ 'own_resource',
                             str_starts(sample, pattern = "OLAC") ~ 'own_resource',
                             .default = study)) %>%
    mutate(dataset = case_when(str_starts(sample, pattern = "emm") ~ 'emm_2022_taojiang',
                             str_starts(sample, pattern = "stm") ~ 'stm_2022_yuxin',
                             str_starts(sample, pattern = "nc") ~ 'nc_2021_zhoufeng',
                             str_starts(sample, pattern = "sttt") ~ 'sttt_2022_Weimin',
                             str_starts(sample, pattern = "ylac") ~ 'own_resource',
                             str_starts(sample, pattern = "OLAC") ~ 'own_resource',
                             .default = dataset)) %>%
    mutate(Age_type = ifelse(age_or_mean_of_age_range <= 40, 'Young', 'Old'))

samples_all_merged <- AddMetaData(object = samples_all_merged, metadata = samples_all_metadata)

In [ ]:
saveRDS(samples_all_merged, file = paste0(obj, "/", "samples_all_merged.rds"))

In [ ]:
print("done")

In [ ]:
samples_all_merged <- readRDS(paste0(obj, "/", "samples_all_merged.rds"))

In [ ]:
samples_all_merged[[]]

## Samples_all_analysis(Integration:Harmony)

### Harmony integration

In [ ]:
#normalization
samples_all_merged <- samples_all_merged %>%
    NormalizeData(verbose = FALSE)

In [ ]:
# find variable genes in each sample
fvf_collection <- split(row.names(samples_all_merged@meta.data), samples_all_merged@meta.data$sample) %>%
    lapply(function(cells_use) {
    samples_all_merged[,cells_use] %>%
        FindVariableFeatures(selection.method = "vst", nfeatures = 2000) %>% 
        VariableFeatures()
    }) %>% unlist

fvf_genes <- table(fvf_collection) %>% as.data.frame() %>% slice_max(order_by = Freq, n = 3000)
fvf_genes <- fvf_genes[['fvf_collection']]

VariableFeatures(samples_all_merged) <- fvf_genes

In [ ]:
samples_all_merged

In [ ]:
#run scale_data
samples_all_merged <- ScaleData(samples_all_merged, verbose = FALSE)

In [ ]:
# run pca
samples_all_merged <- RunPCA(object = samples_all_merged, features = VariableFeatures(samples_all_merged), npcs = 50, verbose = FALSE)

In [ ]:
# run harmony
samples_all_integrated <- RunHarmony(object = samples_all_merged, group.by.vars = 'sample', plot_convergence = T)

### Clustring and annotation analysis

In [ ]:
# UMAP
samples_all_integrated <- RunUMAP(samples_all_integrated, dims = 1:50, reduction = "harmony", verbose = F )


In [ ]:
# FindNeighbors
samples_all_integrated <- FindNeighbors(samples_all_integrated, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  samples_all_integrated <- FindClusters(samples_all_integrated, resolution = i, verbose = F)
  print(DimPlot(samples_all_integrated, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
print("done")

In [ ]:
#save object
saveRDS(samples_all_integrated, paste0(obj, "/", "samples_all_integrated.rds"))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
FeaturePlot(samples_all_integrated, features = c('EPCAM'), raster=T, order = T)

In [ ]:
# get the multiple_subcluster for c9_res0.4
Idents(samples_all_integrated) <- samples_all_integrated$`RNA_snn_res.0.4`
samples_all_integrated <- FindSubCluster(object = samples_all_integrated, cluster = 9, subcluster.name = 'RNA_snn_res.0.4_subclus', graph.name = 'RNA_snn', resolution = 0.1)

In [ ]:
# get the multiple_subcluster for c10_res0.4
Idents(samples_all_integrated) <- samples_all_integrated$`RNA_snn_res.0.4_subclus`
samples_all_integrated <- FindSubCluster(object = samples_all_integrated, cluster = 10, subcluster.name = 'RNA_snn_res.0.4_subclus', graph.name = 'RNA_snn', resolution = 0.1)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
DimPlot(samples_all_integrated, group.by = "RNA_snn_res.0.4_subclus", pt.size = 1, label = T, label.size = 5, raster=T)

In [ ]:
#mainmarkers_expression_dotplot_res_0.4
mainmarkers <- c("PECAM1", "VWF", "ACTA2", "LUM", "CCL21", 'PDGFRA', 'PDGFRB',"JCHAIN","IGKC","MS4A1", "PTPRC", "CD68", "KIT","CD3E","NKG7","S100A6","CD79A", "EPCAM", "CDH1",
                 "KRT7", "KRT19", "SFTPB","SFTPC", "AGER", "FOXJ1","SCGB1A1", "SCGB3A2","CD34","FN1","TGFBI","COL1A1", "CD14", "SPP1", 'SPI1', "TREM2","MKI67")


options(repr.plot.width = 15, repr.plot.height = 10)
DotPlot(samples_all_integrated, features = mainmarkers, group.by = "RNA_snn_res.0.4_subclus") *
    theme(axis.text = element_text(size = 20, face = "bold"), axis.text.x = element_text(angle = 270)) +
    coord_flip()

In [ ]:
#find MainMarkers_all_res0.4_presto with wilcox test in presto
DefaultAssay(samples_all_integrated) <- "RNA"
MainMarkers_all_res0.4_presto <- presto::wilcoxauc(X = samples_all_integrated, group_by = 'RNA_snn_res.0.4_subclus')

In [ ]:
MainMarkers_all_res0.4_presto %>%
    filter(group == '24') %>% slice_max(n = 50, order_by = auc)

In [ ]:
## main cell type annotation.

Idents(samples_all_integrated) <- samples_all_integrated$'RNA_snn_res.0.4_subclus'

main_type_anno <- c("0" = "Imm", "1" = "Imm", "2" = "Epi", "3" = "Imm", "4" = "Imm", "5" = "Endo", "6" = "Imm", "7" = "Epi", "8" = "Imm",
                    "9_0" = "Fibro",  "9_1" = "Fibro",  "9_2" = "Fibro",  "9_3" = "Contamination", "9_4" = "Contamination", "10_0" = "Imm", "10_1" = "Imm", "10_2" = "Contamination",
                    "11" = "Imm", "12" = "Imm", "13" = "Epi", "14" = "Epi", "15" = "Prolif", "16" = "Epi", "17" = "Imm", "18" = "Imm",
                    "19" = "Endo", "20" = "Epi", "21" = "Epi", "22" = "Contamination", "23" = "Epi", "24" = "Imm", "25" = "Imm")



#names(main_type_anno) <- levels(all_21samples_integrated)
samples_all_integrated <- RenameIdents(samples_all_integrated, main_type_anno)
samples_all_integrated$main_cell_type_res_0.4 <- Idents(samples_all_integrated)

DimPlot(samples_all_integrated, label = T, pt.size = 0.5, label.size = 7, raster = FALSE, repel = T) +
    theme(plot.title = element_text(size = 30), legend.text = element_text(size = 20), legend.key.size = unit(0.5, "inches")) +
    guides(colour = guide_legend(override.aes = list(size = 5))) +
    theme(panel.background = element_rect(fill = 'transparent'),
          plot.background = element_rect(fill = 'transparent'),
          legend.box.background = element_rect(fill = 'transparent'),
          legend.background = element_rect(fill = 'transparent'))

ggsave(paste0(figures, '/', "main_cell_type_annotation_res_0.4.pdf"), width = 15, height = 15, bg = 'transparent')

In [ ]:
#filter Contanimation cells
samples_all_integrated <- subset(samples_all_integrated, subset = main_cell_type_res_0.4 != "Contanimation")

In [ ]:
#save object
saveRDS(samples_all_integrated, paste0(obj, "/", "samples_all_integrated.rds"))

In [ ]:
samples_all_integrated <- readRDS(paste0(obj, "/", "samples_all_integrated.rds"))

In [ ]:
samples_all_integrated[[]] %>% colnames()

In [ ]:
samples_all_integrated

In [ ]:
FetchData(samples_all_integrated, vars = c('lung_condition', 'Age_type', 'sample')) %>%
    distinct() %>% 
    group_by(lung_condition, Age_type) %>%
    summarise(n = n())

## Samples_imm_analysis(Integration:Harmony)

### Subset

In [ ]:
Idents(samples_all_integrated) <- samples_all_integrated$`main_cell_type_res_0.4`
levels(samples_all_integrated)

In [ ]:
# subset immune cells
Idents(samples_all_integrated) <- samples_all_integrated$`main_cell_type_res_0.4`
samples_imm_integrated <- subset(samples_all_integrated, idents = c('Imm'))
samples_imm_integrated

### Harmony integration

In [ ]:
#normalization
samples_imm_integrated <- samples_imm_integrated %>%
    NormalizeData(verbose = FALSE)

In [ ]:
# find variable genes in each sample
fvf_collection <- split(row.names(samples_imm_integrated@meta.data), samples_imm_integrated@meta.data$sample) %>%
    lapply(function(cells_use) {
    samples_imm_integrated[,cells_use] %>%
        FindVariableFeatures(selection.method = "vst", nfeatures = 2000) %>% 
        VariableFeatures()
    }) %>% unlist

fvf_genes <- table(fvf_collection) %>% as.data.frame() %>% slice_max(order_by = Freq, n = 3000)
fvf_genes <- fvf_genes[['fvf_collection']]

VariableFeatures(samples_imm_integrated) <- fvf_genes

In [ ]:
samples_imm_integrated

In [ ]:
#run scale_data pca
samples_imm_integrated <- samples_imm_integrated %>% 
    ScaleData(verbose = FALSE) %>% 
    RunPCA(features = VariableFeatures(samples_imm_integrated), npcs = 50, verbose = FALSE)

In [ ]:
# run harmony
options(repr.plot.width = 10, repr.plot.height = 10)
samples_imm_integrated <- RunHarmony(object = samples_imm_integrated, group.by.vars = 'sample', plot_convergence = T)

### Clustring and annotation analysis

In [ ]:
# UMAP
samples_imm_integrated <- RunUMAP(samples_imm_integrated, dims = 1:50, reduction = "harmony", verbose = F, seed.use = 24)


In [ ]:
# FindNeighbors
samples_imm_integrated <- FindNeighbors(samples_imm_integrated, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  samples_imm_integrated <- FindClusters(samples_imm_integrated, resolution = i, verbose = F)
  print(DimPlot(samples_imm_integrated, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
print("done")

In [ ]:
options(repr.plot.height =24, repr.plot.width = 24)

imm_markers <- c('TRAC', 'CD3D',
                 'KLRF1', 'NCR1',
                 'CD79A', 'MS4A1', 'JCHAIN',
                 'CD1C', 'CLEC9A', 'FSCN1', 'LILRA4',
                 'C1QC', 'CD68',
                 'FCN1', 'CD14', 'FCGR3A', 'S100A8',
                 'CSF3R', 'FCGR3B',
                 'KIT')

FeaturePlot(object = samples_imm_integrated, features = imm_markers, order = T)

In [ ]:
# get the multiple_subcluster for c6_res0.9
Idents(samples_imm_integrated) <- samples_imm_integrated$`RNA_snn_res.0.9`
samples_imm_integrated <- FindSubCluster(object = samples_imm_integrated, cluster = 6, subcluster.name = 'RNA_snn_res.0.9_subclus', graph.name = 'RNA_snn', resolution = 0.3)

In [ ]:
options(repr.plot.height =10, repr.plot.width = 10)
DimPlot(samples_imm_integrated, group.by = 'RNA_snn_res.0.9_subclus', label = T)

In [ ]:
#find ImmMarkers_res0.9_presto with wilcox test in presto
ImmMarkers_res0.9_presto <- presto::wilcoxauc(X = samples_imm_integrated, group_by = 'RNA_snn_res.0.9_subclus')

In [ ]:
ImmMarkers_res0.9_presto %>% filter(group == '21') %>% arrange(-logFC)

In [ ]:
ImmMarkers_res0.9_presto %>% filter(group == '27') %>% arrange(-logFC) %>% pull(feature) %>% head(100)

In [ ]:
#imm_marker;
DefaultAssay(samples_imm_integrated) <- "RNA"

#
#naïve B cells (CD20+, CD27−, and CD38−), 主要的基因是 IGHD, FCER2, TCL1A, and IL4R,
#memory B cells (CD20+, CD27+, and CD38–), 主要的基因是 CD27, AIM2, TNFRSF13B
#germinal center (GC) B cells (CD20+, CD27+, CD38+, and CD138−),主要的基因是S1PI2, LRMP, SUGCT, MME, MKI67, and AICDA
imm_markers <- c("TRAC", "CD3E", "CD4", "CD8A", "NCR1", "KLRB1", "KLRD1", "NKG7", "NCAM", "ID2", "IL7R", "GATA3",
                 "LYZ", "CD68", 'CD69', 'LGMN', 'CSF1R', "ITGAX", "MARCO", "FCGR1A", "C1QA", "APOC1",
                 "FCGR3B", "CSF3R", "FCN1", "S100A9", "CD14", "FCER1A", "CD1C", "FCGR3A", "CLEC9A", "LILRA4", "CLEC4C",
                 "CD79A", "MS4A1", "IGHD", "FCER2", 'CD27', 'AIM2', 'TNFRSF13B', 'CD38', 'AICDA', 'LRMP','JCHAIN', 'IGHA1', 'IGHG1',
                 "CPA3", "KIT", "MKI67", "CDK1", "EPCAM")


options(repr.plot.height =12, repr.plot.width = 18)
DotPlot(samples_imm_integrated, features = imm_markers, group.by = "RNA_snn_res.0.9_subclus", scale = T) * theme(axis.text = element_text(size = 20, face = "bold")) +
  coord_flip()


- NK_T_marker
    - Th1: TBX21, IL2, IFNG, TNF;
    - Th2: GATA3, IL4, IL5, IL6, IL10, IL13;
    - Th9: IL9, IL10
    - Th17: RORC, IL17A, IL17F, IL21, IL22, IL26, CCR6
    - Th22: IL22, no IL17
    - Tfh: 'BCL6', 'CXCR5', 'CCR5'
    - NK_T_marker <- c('CD3E', 'CD4', 'CD8A','TRAC', #Tcell
                 'TCF7', 'SELL', 'LEF1', 'CCR7', #naive
                 'IL2', 'IFNG','GZMA', 'GZMB', 'GZMK', 'GNLY', 'PRF1', #effector
                 'LAG3', 'TIGIT', 'PDCD1', 'HAVCR2', 'CTLA4', 'CXCL13', 'TOX', #exhuasted 
                 'CD27', 'CD28', 'ICOS', 'TNFRSF9', 'TNFRSF14', #costimulatory
                 'IL2RA', 'FOXP3', 'IKZF2', # Treg
                 'EOMES', 'HOPX', 'TBX21', 'ZNF683', 'HIF1A', #transcription factor
                 'ID2', 'IL7R', 'GATA3', 'KIT', 'THY1', #ILC
                 'NKG7', 'KLRF1', 'NCR1', 'FGFBP2', 'NCAM1', #NK
                 'TRGV9', 'TRDC', #gdT
                 'SLC4A10', 'IL12RB', 'IL18R1', 'PLZF', 'MR1', #MAIT
                 'ITGAE', S1PI2'ITGA1','CD41RO', 'CD69','IL15RA', 'MBD2', #memory CD41RO
                 'CXCR5', 'CCR5','BCL6', #Tfh
                 'KLRC2', 'KLRG1' #Temra)

- Myeloid_marker
    - naïve B cells (CD20+, CD27−, and CD38−), 主要的基因是 IGHD, FCER2, TCL1A, and IL4R,
    - memory B cells (CD20+, CD27+, and CD38–), 主要的基因是 CD27, AIM2, TNFRSF13B
    - germinal center (GC) B cells (CD20+, CD27+, CD38+, and CD138−),主要的基因是S1PI2, LRMP, SUGCT, MME, MKI67, and AICDA
    - Myeloid_marker <- c("KIT", 'GATA2', 'CPA3',#MAST CELL
                    "CSF3R", 'FCGR3B', #neutrophil
                    "LILRA4", 'TCF4', 'CLEC4C', #pDC
                    "XCR1", "CADM1", 'CLEC9A', 'THBD', #cDC1
                    "CD1A", 'CD1C', 'CD1E', 'CD207', 'FCER1A', #cDC2
                    "FSCN1", #cDC3
                    'CSF1R', 'LYZ', "CD14", 'FCN1', 'S100A8', 'S100A9', 'FCGR3A', 'CDKN1C', 'LILRB2', 'ITGAL', 'LST1',#monocyte
                    "CD68", "CD163",'FABP4', 'MARCO', 'PPARG','C1QA', 'APOC1', 'LGMN') #macrophage

In [ ]:
options(repr.plot.height =10, repr.plot.width = 10)
DimPlot(samples_imm_integrated, group.by = 'RNA_snn_res.0.9_subclus', label = T)

In [ ]:
##imm mainclass annotation_res0.9
Idents(samples_imm_integrated) <- samples_imm_integrated$RNA_snn_res.0.9_subclus
imm_maincluster_anno <- c('0' = 'T_Cell', '1' = 'Macro', '2' = 'B_Cell','3' = 'T_Cell', '4' = 'Macro', '5' = 'NK', 
                          '6_0' = 'Mono', '6_1' = 'Mono', '6_2' = 'Mono', '6_3' = 'Neutrophil', '7' = 'T_Cell',
                          '8' = 'T_Cell', '9' = 'Mast', '10' = 'DC', '11' = 'Mono',
                          '12' = 'Plasma', '13' = 'Macro', '14' = 'Macro', '15' = 'Plasma', '16' = 'NK',
                          '17' = 'Macro', '18' = 'Mono', '19' = 'T_Cell', '20' = 'DC', '21' = 'DC',
                          '22' = 'T_Cell', '23' = 'Plasma', '24' = 'Macro', '25' = 'Unknown', '26' = 'Unknown', '27' = 'Unknown')



#names(imm_maincluster_anno) <- levels(all_22samples_imm_integrated)
samples_imm_integrated <- RenameIdents(samples_imm_integrated, imm_maincluster_anno)
samples_imm_integrated$ImmMaincluster_res0.9 <- Idents(samples_imm_integrated)

options(repr.plot.height =10, repr.plot.width = 12)
DimPlot(samples_imm_integrated, label = T, pt.size = 1, label.size = 7, repel = T) +
    theme(plot.title = element_text(size = 30),
          legend.text = element_text(size = 20),
          legend.key.size = unit(0.5, "inches")) +
    guides(colour = guide_legend(override.aes = list(size = 5)))

ggsave(paste0(figures, '/', "imm_maincluster_annotation_res_0.9.png"), width = 18, height = 18)

In [ ]:
#filter Unknown cells
samples_imm_integrated <- subset(samples_imm_integrated, subset = ImmMaincluster_res0.9 != "Unknown")

In [ ]:
saveRDS(samples_imm_integrated, file = paste0(obj, "/", "samples_imm_integrated.rds"))

In [ ]:
table(samples_imm_integrated$`ImmMaincluster_res0.9`)

In [ ]:
options(repr.plot.height =10, repr.plot.width = 10)
DimPlot(samples_imm_integrated, group.by = 'ImmMaincluster_res0.9', label = F, pt.size = 0.5, raster=FALSE, shuffle=T) +
    scale_color_manual(values = main_celltype_colors) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'none')

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Imm_cell_UMAP.pdf'), device = 'pdf', width = 10, height = 10, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Imm_cell_UMAP.png'), device = 'png', width = 10, height = 10, dpi = 300, bg = 'transparent')

In [ ]:
Imm_cell_type <- c('T_Cell', 'NK', 'B_Cell', 'Plasma', 'DC', 'Macro', 'Mono', 'Neutrophil', 'Mast')

imm_markers <- c('TRAC', 'CD3D', 'KLRF1', 'NCR1', 'CD79A', 'MS4A1', 'JCHAIN',
                 'CD1C', 'CLEC9A', 'C1QC', 'CD68', 'FCN1', 'CD14', 'S100A8', 'CSF3R', 'KIT')

options(repr.plot.height =8, repr.plot.width = 16)
DotPlot(samples_imm_integrated, features = imm_markers, group.by = 'ImmMaincluster_res0.9', scale = T) + 
    scale_y_discrete(limits = Imm_cell_type) +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', guide = guide_colorbar(order = 1), limits = c(-1,1), oob = scales::squish) +
    scale_size_area(max_size = 12, guide = guide_legend(order = 2)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.border = element_rect(linewidth = 1, fill = NA, color = 'black'),
          legend.title = element_text(size = 20),
          legend.position = 'top',
          axis.text = element_text(colour = 'black'),
          axis.line = element_blank(),
          axis.text.x = element_text(angle = 90),
          axis.title.x = element_blank(),
          axis.title.y = element_blank())

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Imm_Cells_markers_Dotplot.pdf'), device = 'pdf', width = 13, height = 8, bg = 'transparent')

In [ ]:
samples_imm_integrated <- readRDS(paste0(obj, "/", "samples_imm_integrated.rds"))

In [ ]:
samples_imm_integrated

### Abundance analysis

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8)

Imm_cell_type <- c('T_Cell', 'NK', 'B_Cell', 'Plasma', 'DC', 'Macro', 'Mono', 'Neutrophil', 'Mast')
FetchData(samples_imm_integrated, vars = c('lung_condition', 'Age_type', 'ImmMaincluster_res0.9')) %>%
    group_by(lung_condition, Age_type, ImmMaincluster_res0.9) %>%
    summarise(n_cell = n()) %>%
    mutate(n_Immcell = sum(n_cell)) %>%
    mutate(proportion = n_cell/n_Immcell) %>%
    mutate(ImmMaincluster_res0.9 = factor(ImmMaincluster_res0.9, levels = Imm_cell_type)) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = Age_type, y = proportion, fill = ImmMaincluster_res0.9)) +
    geom_col(linewidth = 0.3, color = 'black') +
    facet_wrap(facets = ~lung_condition) +
    scale_fill_manual(values = main_celltype_colors,
                      guide = guide_legend(title = 'Cell Type', ncol = 1, title.theme = element_text(size = 20))) +
    xlab(label = NULL) +
    ylab(label = 'Proportion Relative to Imm Cell') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'right',
          text = element_text(face = 'plain'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 25),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Imm_Proportion_BarPlot.pdf'), device = 'pdf', width = 10, height = 10, bg = 'transparent')


In [ ]:
options(repr.plot.width = 16, repr.plot.height = 10)
FetchData(samples_imm_integrated, vars = c('lung_condition', 'Age_type', 'ImmMaincluster_res0.9', 'sample')) %>%
    group_by(lung_condition, Age_type, sample, ImmMaincluster_res0.9) %>%
    summarise(n_cell = n()) %>%
    mutate(n_Immcell = sum(n_cell), Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(n_Immcell > 100) %>%    # filter the samples whose n_Immcell less than 100
    mutate(proportion = n_cell/n_Immcell)  %>%
    filter(!ImmMaincluster_res0.9 %in% c('Neutrophil')) %>%
    mutate(ImmMaincluster_res0.9 = factor(ImmMaincluster_res0.9, levels = Imm_cell_type)) %>%
    ggplot(mapping = aes(x = lung_condition, y = proportion, fill = Age_type)) +
#    stat_boxplot(geom = 'errorbar', width = 0.3, linewidth = 0.8, position = position_dodge(width = 0.75)) +
    geom_boxplot(position = position_dodge(width = 0.75),  width = 0.5, linewidth = 0.8) +
    geom_jitter(size = 3, shape = 21, position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75)) +
    facet_wrap(facets = ~ImmMaincluster_res0.9, scales = 'free_y', nrow = 2) +
    stat_compare_means(mapping = aes(group = lung_condition), label = 'p.signif',
                       method = 'wilcox.test', size = 6, label.x.npc = c(0.5)) +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif',
                       method = 'wilcox.test', size = 6, label.y.npc = c(0.9)) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 2)) +
#    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Proportion Relative to Imm Cell') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Imm_ProportionBoxPlot.pdf'), device = 'pdf', width = 20, height = 10, bg = 'transparent')


In [ ]:
samples_imm_integrated

### DEGs analysis

#### DEGs for Overall

In [ ]:
##split the samples_imm_integrated into healthy and tumor
DefaultAssay(samples_imm_integrated) <- "RNA"
Idents(samples_imm_integrated) <- samples_imm_integrated$`lung_condition`

samples_imm_healthy_integrated <- subset(samples_imm_integrated, idents = 'Healthy')

samples_imm_tumor_integrated <- subset(samples_imm_integrated, idents = 'Tumor')

In [ ]:
unique(samples_imm_healthy_integrated$`ImmMaincluster_res0.9`)

In [ ]:
##find DEGs between imm_healthy_young and imm_healthy_old with wilcox_test
Idents(samples_imm_healthy_integrated) <- samples_imm_healthy_integrated$`ImmMaincluster_res0.9`

for ( i in c('Macro', 'T_Cell', 'Plasma', 'NK', 'Mono', 'Mast', 'B_Cell', 'DC')) {
    print(i)
    assign(x = paste0('DEGs_imm_', i, '_young_old_healthy_wilcox'), 
            value = FindMarkers(object = samples_imm_healthy_integrated, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                            logfc.threshold = 0.25, min.pct = 0.1, group.by = 'Age_type', subset.ident = i))
    
    saveRDS(get(paste0('DEGs_imm_', i, '_young_old_healthy_wilcox')), paste0(obj, '/', 'DEGs_imm_healthy/', paste0('DEGs_imm_', i, '_young_old_healthy_wilcox.rds')))
}

In [ ]:
##find DEGs between imm_tumor_young and imm_tumor_old with wilcox_test
Idents(samples_imm_tumor_integrated) <- samples_imm_tumor_integrated$`ImmMaincluster_res0.9`

for ( i in c('Macro', 'T_Cell', 'Plasma', 'NK', 'Mono', 'Mast', 'B_Cell', 'DC')) {

    assign(x = paste0('DEGs_imm_', i, '_young_old_tumor_wilcox'), 
           value = FindMarkers(object = samples_imm_tumor_integrated, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                          logfc.threshold = 0.25, min.pct = 0.1, group.by = 'Age_type', subset.ident = i))
    
    saveRDS(get(paste0('DEGs_imm_', i, '_young_old_tumor_wilcox')), paste0(obj, '/', 'DEGs_imm_tumor/', paste0('DEGs_imm_', i, '_young_old_tumor_wilcox.rds')))
}

In [ ]:
# imm_healthy_maincluster  DEGs gene numbers

maincluster_vec <- vector(mode = "character")
nDEGs_vec <- vector(mode = "numeric")


for ( i in c('Macro', 'T_Cell', 'Plasma', 'NK', 'Mono', 'Mast', 'B_Cell', 'DC')) {
    DEGs_imm_healthy_df <- readRDS(paste0(obj, '/', 'DEGs_imm_healthy/', 'DEGs_imm_', i, '_young_old_healthy_wilcox.rds'))
    nDEGs <- (DEGs_imm_healthy_df %>% filter(abs(avg_log2FC) > 0.5) %>% filter(p_val_adj < 0.05) %>% dim())[1]
    maincluster_vec <- c(maincluster_vec, i)
    nDEGs_vec <- c(nDEGs_vec, nDEGs)
    nDEGs_imm_healthy_df <- data.frame(maincluster_vec, nDEGs_vec)
    # cat(paste0('DEGs_imm_', i, '_young_old_wilcox', ':', nDEGs, '\n'))
}
nDEGs_imm_healthy_df <- nDEGs_imm_healthy_df %>%
    filter(!maincluster_vec %in% c('test')) %>%
    arrange(desc(nDEGs_vec)) %>%
    mutate(lung_condition = 'Healthy')

In [ ]:
# imm_tumor_maincluster  DEGs gene numbers

maincluster_vec <- vector(mode = "character")
nDEGs_vec <- vector(mode = "numeric")


for ( i in c('Macro', 'T_Cell', 'Plasma', 'NK', 'Mono', 'Mast', 'B_Cell', 'DC')) {
    DEGs_imm_tumor_df <- readRDS(paste0(obj, '/', 'DEGs_imm_tumor/', 'DEGs_imm_', i, '_young_old_tumor_wilcox.rds'))
    nDEGs <- (DEGs_imm_tumor_df %>% filter(abs(avg_log2FC) > 0.5) %>% filter(p_val_adj < 0.05) %>% dim())[1]
    maincluster_vec <- c(maincluster_vec, i)
    nDEGs_vec <- c(nDEGs_vec, nDEGs)
    nDEGs_imm_tumor_df <- data.frame(maincluster_vec, nDEGs_vec)
    # cat(paste0('DEGs_imm_', i, '_young_old_wilcox', ':', nDEGs, '\n'))
}
nDEGs_imm_tumor_df <- nDEGs_imm_tumor_df %>%
    filter(!maincluster_vec %in% c('test')) %>%
    arrange(desc(nDEGs_vec)) %>%
    mutate(lung_condition = 'Diseased')

In [ ]:
nDEGs_imm_tumor_df

In [ ]:
nDEGs_imm_healthy_df

In [ ]:
# Imm_maincluster_df  DEGs gene numbers (plot)
options(repr.plot.width = 10, repr.plot.height = 10)
Imm_cell_type <- c('T_Cell', 'NK', 'B_Cell', 'Plasma', 'DC', 'Macro', 'Mono', 'Neutrophil', 'Mast')
nDEGs_imm_df <- rbind(nDEGs_imm_healthy_df, nDEGs_imm_tumor_df)


ggplot(data = nDEGs_imm_df) +
    geom_col(mapping = aes(x = fct_reorder(maincluster_vec, .x = nDEGs_vec, .fun = max, .desc = T),
                           y = nDEGs_vec, fill = maincluster_vec, linetype = lung_condition),
             color = 'black', position = position_dodge(width = 1), linewidth = 1, width = 0.8) +
    scale_fill_manual(values = main_celltype_colors,
                      limits = Imm_cell_type,
                      guide = guide_legend(title = 'Cell Type',
                                           title.theme = element_text(size = 20),
                                           order = 2)) +
    scale_linetype_manual(values = c(Healthy = 'dashed', Diseased = 'solid'),
                          limits = c('Healthy', 'Diseased'),
                          guide = guide_legend(title = 'Lung Condition',
                                               title.theme = element_text(size = 20),
                                               order = 1)) +
    xlab(label = NULL) +
    ylab(label = 'The Number of Differentially Expressed Genes') +
    scale_y_continuous(expand = expansion(mult = c(0.03))) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.grid = element_blank(),
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 25),
          legend.position = 'right',
          legend.title = element_text(size = 20),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))


ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Imm_Clsuter_nDEGs_BarPlot.pdf'), device = 'pdf', width = 10.5, height = 10, bg = 'transparent')

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
Imm_cell_type <- c('T_Cell', 'NK', 'B_Cell', 'Plasma', 'DC', 'Macro', 'Mono', 'Neutrophil', 'Mast')
nDEGs_imm_df <- rbind(nDEGs_imm_healthy_df, nDEGs_imm_tumor_df)

#add lymphoid and myeloid identity
nDEGs_imm_df <- nDEGs_imm_df %>%
    mutate(lineage = ifelse(maincluster_vec %in% c( 'B_Cell', 'Plasma', 'T_Cell', 'NK'),
                            'Lymphoid', 'Myeloid')) %>%
    mutate(lung_condition = factor(lung_condition, levels = c('Diseased', 'Healthy'))) %>%
    mutate(lineage = factor(lineage, levels = c('Myeloid', 'Lymphoid'))) %>%
    arrange(desc(lineage), desc(nDEGs_vec)) %>%
    mutate(maincluster_vec = factor(maincluster_vec, levels = unique(maincluster_vec)))

In [ ]:
ggplot(data = nDEGs_imm_df, mapping = aes(x = maincluster_vec,
       y = nDEGs_vec,
       fill = lung_condition,
       color = maincluster_vec)) +
    geom_col(position = position_dodge(width = 1), linewidth = 1, width = 0.8) +
    scale_fill_manual(values = c('Diseased' = 'black', 'Healthy' = 'transparent'),
                      #limits = Imm_cell_type,
                      limits = c('Healthy', 'Diseased'),
                      guide = guide_legend(title = 'Lung Condition',
                                           title.theme = element_text(size = 20),
                                           order = 1)) +
    scale_color_manual(values = main_celltype_colors,
                       limits = Imm_cell_type,
                       guide = guide_legend(title = 'Cell Type',
                                            title.theme = element_text(size = 20),
                                            order = 2)) +
    xlab(label = NULL) +
    ylab(label = 'The Number of Differentially Expressed Genes') +
    scale_y_continuous(expand = expansion(mult = c(0.03))) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.grid = element_blank(),
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 25),
          legend.position = 'right',
          legend.title = element_text(size = 20),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Imm_Clsuter_nDEGs_FillNA_BarPlot.pdf'), device = 'pdf', width = 10.5, height = 10, bg = 'transparent')

In [ ]:
nDEGs_imm_df

#### DEGs for T_Cell

In [ ]:
# volcano plot for T_Cell_healthy
DEGs_imm_T_Cell_young_old_healthy_wilcox <- readRDS(paste0(obj, '/', 'DEGs_imm_healthy', '/', 'DEGs_imm_T_Cell_young_old_healthy_wilcox.rds'))

options(repr.plot.width = 10, repr.plot.height = 10)
volcano_plot(DEGs_df = DEGs_imm_T_Cell_young_old_healthy_wilcox, col_names_pval = 'p_val_adj', col_names_LFC = 'avg_log2FC',
             pval_cutoff = 0.05, logFC_cutoff = 0.75, label_logFC_cutoff = 0.75, label_pval_cutoff = 0.05,
             title = NULL)



In [ ]:
# volcano plot for T_Cell_tumor
DEGs_imm_T_Cell_young_old_tumor_wilcox <- readRDS(paste0(obj, '/', 'DEGs_imm_tumor', '/', 'DEGs_imm_T_Cell_young_old_tumor_wilcox.rds'))

options(repr.plot.width = 10, repr.plot.height = 10)
volcano_plot(DEGs_df = DEGs_imm_T_Cell_young_old_tumor_wilcox, col_names_pval = 'p_val_adj', col_names_LFC = 'avg_log2FC',
             pval_cutoff = 0.05, logFC_cutoff = 0.75, label_logFC_cutoff = 0.75, label_pval_cutoff = 0.05,
             title = NULL)

In [ ]:
DEGs_T_healthy_up <- DEGs_imm_T_Cell_young_old_healthy_wilcox %>%
    filter(p_val_adj < 0.05 & avg_log2FC > 0.5) %>%
    arrange(desc(avg_log2FC)) %>%
    mutate(gene = rownames(.))

In [ ]:
DEGs_T_tumor_up <- DEGs_imm_T_Cell_young_old_tumor_wilcox %>%
    filter(p_val_adj < 0.05 & avg_log2FC > 0.5) %>%
    arrange(desc(avg_log2FC)) %>%
    mutate(gene = rownames(.))

In [ ]:
DEGs_T_healthy_down <- DEGs_imm_T_Cell_young_old_healthy_wilcox %>%
    filter(p_val_adj < 0.05 & avg_log2FC < -0.5) %>%
    arrange(desc(avg_log2FC)) %>%
    mutate(gene = rownames(.))

In [ ]:
DEGs_T_tumor_down <- DEGs_imm_T_Cell_young_old_tumor_wilcox %>%
    filter(p_val_adj < 0.05 & avg_log2FC < -0.5) %>%
    arrange(desc(avg_log2FC)) %>%
    mutate(gene = rownames(.))

In [ ]:
length(DEGs_T_healthy_up[['gene']]);length(DEGs_T_healthy_down[['gene']]);length(DEGs_T_tumor_up[['gene']]); length(DEGs_T_tumor_down[['gene']])

In [ ]:
DEGs_T_healthy_down[['gene']]; DEGs_T_tumor_down[['gene']]

In [ ]:
intersect(DEGs_T_healthy_down[['gene']], DEGs_T_tumor_down[['gene']])

In [ ]:
setdiff(DEGs_T_tumor_down[['gene']], DEGs_T_healthy_down[['gene']]) %>% length();
cat('\n')
setdiff(DEGs_T_healthy_down[['gene']], DEGs_T_tumor_down[['gene']])  %>% length()

In [ ]:
setdiff(DEGs_T_tumor_down[['gene']], DEGs_T_healthy_down[['gene']]) %>% str_flatten_comma();
cat('\n')
setdiff(DEGs_T_healthy_down[['gene']], DEGs_T_tumor_down[['gene']]) %>% str_flatten_comma()

In [ ]:
DEGs_T_healthy_up[['gene']]; DEGs_T_tumor_up[['gene']]

In [ ]:
setdiff(DEGs_T_tumor_up[['gene']], DEGs_T_healthy_up[['gene']]) %>% length();
cat('\n')
setdiff(DEGs_T_healthy_up[['gene']], DEGs_T_tumor_up[['gene']]) %>% length()

In [ ]:
setdiff(DEGs_T_tumor_up[['gene']], DEGs_T_healthy_up[['gene']]) %>% str_flatten_comma();
cat('\n')
setdiff(DEGs_T_healthy_up[['gene']], DEGs_T_tumor_up[['gene']]) %>% str_flatten_comma()

#### DEGs for B_Cell

In [ ]:
# volcano plot for B_Cell_healthy
DEGs_imm_B_Cell_young_old_healthy_wilcox <- readRDS(paste0(obj, '/', 'DEGs_imm_healthy', '/', 'DEGs_imm_B_Cell_young_old_healthy_wilcox.rds'))

options(repr.plot.width = 10, repr.plot.height = 10)
volcano_plot(DEGs_df = DEGs_imm_B_Cell_young_old_healthy_wilcox, col_names_pval = 'p_val_adj', col_names_LFC = 'avg_log2FC',
             pval_cutoff = 0.05, logFC_cutoff = 0.75, label_logFC_cutoff = 0.75, label_pval_cutoff = 0.05,
             title = NULL)



In [ ]:
# volcano plot for B_Cell_tumor
DEGs_imm_B_Cell_young_old_tumor_wilcox <- readRDS(paste0(obj, '/', 'DEGs_imm_tumor', '/', 'DEGs_imm_B_Cell_young_old_tumor_wilcox.rds'))

options(repr.plot.width = 10, repr.plot.height = 10)
volcano_plot(DEGs_df = DEGs_imm_B_Cell_young_old_tumor_wilcox, col_names_pval = 'p_val_adj', col_names_LFC = 'avg_log2FC',
             pval_cutoff = 0.05, logFC_cutoff = 0.75, label_logFC_cutoff = 0.75, label_pval_cutoff = 0.05,
             title = NULL)

In [ ]:
DEGs_B_healthy_up <- DEGs_imm_B_Cell_young_old_healthy_wilcox %>%
    filter(p_val_adj < 0.05 & avg_log2FC > 0.5) %>%
    arrange(desc(avg_log2FC)) %>%
    mutate(gene = rownames(.))

In [ ]:
DEGs_B_tumor_up <- DEGs_imm_B_Cell_young_old_tumor_wilcox %>%
    filter(p_val_adj < 0.05 & avg_log2FC > 0.5) %>%
    arrange(desc(avg_log2FC)) %>%
    mutate(gene = rownames(.))

In [ ]:
DEGs_B_healthy_down <- DEGs_imm_B_Cell_young_old_healthy_wilcox %>%
    filter(p_val_adj < 0.05 & avg_log2FC < -0.5) %>%
    arrange(desc(avg_log2FC)) %>%
    mutate(gene = rownames(.))

In [ ]:
DEGs_B_tumor_down <- DEGs_imm_B_Cell_young_old_tumor_wilcox %>%
    filter(p_val_adj < 0.05 & avg_log2FC < -0.5) %>%
    arrange(desc(avg_log2FC)) %>%
    mutate(gene = rownames(.))

In [ ]:
DEGs_B_healthy_down[['gene']]; DEGs_B_tumor_down[['gene']]

In [ ]:
intersect(DEGs_B_healthy_down[['gene']], DEGs_B_tumor_down[['gene']])

In [ ]:
setdiff(DEGs_B_tumor_down[['gene']], DEGs_B_healthy_down[['gene']]) %>% length();
cat('\n')
setdiff(DEGs_B_healthy_down[['gene']], DEGs_B_tumor_down[['gene']]) %>% length()

In [ ]:
setdiff(DEGs_B_tumor_down[['gene']], DEGs_B_healthy_down[['gene']]) %>% str_flatten_comma();
cat('\n')
setdiff(DEGs_B_healthy_down[['gene']], DEGs_B_tumor_down[['gene']]) %>% str_flatten_comma()

In [ ]:
DEGs_B_healthy_up[['gene']]; DEGs_B_tumor_up[['gene']]

In [ ]:
setdiff(DEGs_B_tumor_up[['gene']], DEGs_B_healthy_up[['gene']]) %>% length();
cat('\n')
setdiff(DEGs_B_healthy_up[['gene']], DEGs_B_tumor_up[['gene']]) %>% length()

In [ ]:
setdiff(DEGs_B_tumor_up[['gene']], DEGs_B_healthy_up[['gene']]) %>% str_flatten_comma();
cat('\n')
setdiff(DEGs_B_healthy_up[['gene']], DEGs_B_tumor_up[['gene']]) %>% str_flatten_comma()

## Samples_T_Cell_analysis

### Subset

In [ ]:
Idents(samples_imm_integrated) <- samples_imm_integrated$`ImmMaincluster_res0.9`
levels(samples_imm_integrated)

In [ ]:
# subset T_Cell
Idents(samples_imm_integrated) <- samples_imm_integrated$`ImmMaincluster_res0.9`
samples_T_Cell_integrated <- subset(samples_imm_integrated, idents = c('T_Cell'))
samples_T_Cell_integrated

### Clustring and annotation analysis

In [ ]:
# UMAP
samples_T_Cell_integrated <- RunUMAP(samples_T_Cell_integrated, dims = 1:50, reduction = "harmony", verbose = F)


In [ ]:
# FindNeighbors
samples_T_Cell_integrated <- FindNeighbors(samples_T_Cell_integrated, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
options(repr.plot.width = 8, repr.plot.height = 8)
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  samples_T_Cell_integrated <- FindClusters(samples_T_Cell_integrated, resolution = i, verbose = F)
  print(DimPlot(samples_T_Cell_integrated, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

- NK_T_marker
    - Th1: TBX21, IL2, IFNG, TNF;
    - Th2: GATA3, IL4, IL5, IL6, IL10, IL13;
    - Th9: IL9, IL10
    - Th17: RORC, IL17A, IL17F, IL21, IL22, IL26, CCR6
    - Th22: IL22, no IL17
    - Tfh: 'BCL6', 'CXCR5', 'CCR5'
    - NK_T_marker <- c('CD3E', 'CD4', 'CD8A','TRAC', #Tcell
                 'TCF7', 'SELL', 'LEF1', 'CCR7', #naive
                 'IL2', 'IFNG','GZMA', 'GZMB', 'GZMK', 'GNLY', 'PRF1', #effector
                 'LAG3', 'TIGIT', 'PDCD1', 'HAVCR2', 'CTLA4', 'CXCL13', 'TOX', #exhuasted 
                 'CD27', 'CD28', 'ICOS', 'TNFRSF9', 'TNFRSF14', #costimulatory
                 'IL2RA', 'FOXP3', 'IKZF2', # Treg
                 'EOMES', 'HOPX', 'TBX21', 'ZNF683', 'HIF1A', #transcription factor
                 'ID2', 'IL7R', 'GATA3', 'KIT', 'THY1', #ILC
                 'NKG7', 'KLRF1', 'NCR1', 'FGFBP2', 'NCAM1', #NK
                 'TRGV9', 'TRDC', #gdT
                 'SLC4A10', 'IL12RB', 'IL18R1', 'PLZF', 'MR1', #MAIT
                 'ITGAE', S1PI2'ITGA1','CD41RO', 'CD69','IL15RA', 'MBD2', #memory CD41RO
                 'CXCR5', 'CCR5','BCL6', #Tfh
                 'KLRC2', 'KLRG1' #Temra
                )

- Myeloid_marker
    - naïve B cells (CD20+, CD27−, and CD38−), 主要的基因是 IGHD, FCER2, TCL1A, and IL4R,
    - memory B cells (CD20+, CD27+, and CD38–), 主要的基因是 CD27, AIM2, TNFRSF13B
    - germinal center (GC) B cells (CD20+, CD27+, CD38+, and CD138−),主要的基因是S1PI2, LRMP, SUGCT, MME, MKI67, and AICDA
    - Myeloid_marker <- c("KIT", 'GATA2', 'CPA3',#MAST CELL
                    "CSF3R", 'FCGR3B', #neutrophil
                    "LILRA4", 'TCF4', 'CLEC4C', #pDC
                    "XCR1", "CADM1", 'CLEC9A', 'THBD', #cDC1
                    "CD1A", 'CD1C', 'CD1E', 'CD207', 'FCER1A', #cDC2
                    "FSCN1", #cDC3
                    'CSF1R', 'LYZ', "CD14", 'FCN1', 'S100A8', 'S100A9', 'FCGR3A', 'CDKN1C', 'LILRB2', 'ITGAL', 'LST1',#monocyte
                    "CD68", "CD163",'FABP4', 'MARCO', 'PPARG','C1QA', 'APOC1', 'LGMN') #macrophage

In [ ]:
FeaturePlot(samples_T_Cell_integrated, features = 'IFITM3', order = T)

In [ ]:
options(repr.plot.height =24, repr.plot.width = 24)
T_markers1 <- c('TRAC', 'CD3E', 'CD4', 'CD8A', 'TCF7', 'SELL', 'LEF1', 'CCR7', 'IL7R',
                'CD27', 'CD28', 'MAL', 'KLF2', 'PIK3IP1', 'TRDC', 'TRGC2')

FeaturePlot(object = samples_T_Cell_integrated, features = T_markers1, order = T)

In [ ]:
options(repr.plot.height =48, repr.plot.width = 24)
T_markers2 <- c(
    'CXCR6', 'CD69', 'IL7R', 'KLRB1', 'PTGER4',
    'IFNG', 'GZMA', 'GZMB', 'GZMK', 'GNLY', 'PRF1', 'NKG7',
    'ZNF683', 'ITGAE', 'RBPJ',
    'LAG3', 'TIGIT', 'PDCD1', 'HAVCR2', 'CTLA4', 'CXCL13','TOX',
    'IL2RA', 'FOXP3', 'IKZF2',
    'IFITM3', 'IFI27', 'ISG15', 'ISG20',
    'SLC4A10', 'IL12RB', 'IL18R1', 'PLZF')

FeaturePlot(object = samples_T_Cell_integrated, features = T_markers2, order = T)

In [ ]:
#find TMarkers_res0.5_presto with wilcox test in presto
TMarkers_res0.5_presto <- presto::wilcoxauc(X = samples_T_Cell_integrated, group_by = 'RNA_snn_res.0.5')

In [ ]:
TMarkers_res0.5_presto %>% filter(group == '11') %>% arrange(-logFC) %>% pull(feature) %>% head(100)

In [ ]:
options(repr.plot.height =8, repr.plot.width = 8)
DimPlot(samples_T_Cell_integrated, group.by = 'RNA_snn_res.0.5', label = T)

In [ ]:
##T_Cell subclass annotation_res0.5
options(repr.plot.width = 16, repr.plot.height = 12)
Idents(samples_T_Cell_integrated) <- samples_T_Cell_integrated$`RNA_snn_res.0.5`

T_Cell_subcluster_anno <- c('0' = 'CCR7_CD4_Tnaive', '1' = 'CXCR6_CD4_Trm', '2' = 'GZMK_CD8_Tem', '3' = 'ZNF683_CD8_Trm',
                            '4' = 'GZMB_CD8_Teffector', '5' = 'FOXP3_CD4_Treg', '6' = 'CXCL13_CD8_Tex',  '7' = 'MAIT',
                            '8' = 'CXCL13_CD4_Tex', '9' = 'ISG15_Teffector', '10' = 'GZMK_CD8_Tem', '11' = 'IFITM3_CD8_Teffector')



#names(imm_maincluster_anno) <- levels(all_22samples_imm_integrated)
samples_T_Cell_integrated <- RenameIdents(samples_T_Cell_integrated, T_Cell_subcluster_anno)
samples_T_Cell_integrated$T_subcluster_res_0.5 <- Idents(samples_T_Cell_integrated)

DimPlot(samples_T_Cell_integrated, label = T, pt.size = 1, label.size = 7, repel = T) +
theme(plot.title = element_text(size = 30),
      legend.text = element_text(size = 20),
      legend.key.size = unit(0.5, "inches")) +
guides(colour = guide_legend(override.aes = list(size = 5)))

ggsave(paste0(figures, '/', "T_Cell_subcluster_annotation_res_0.5.png"), width = 18, height = 18)

In [ ]:
saveRDS(samples_T_Cell_integrated, file = paste0(obj, "/", "samples_T_Cell_integrated.rds"))

In [ ]:
T_markers <- c('TRAC', 'CD3E', 'CD4', 'CD8A',
    'TCF7', 'SELL', 'LEF1', 'CCR7',
    'CXCR6', 'CD69', 'IL7R', 'KLRB1', 'PTGER4',
    'IFNG', 'GZMA', 'GZMB', 'GZMK', 'GNLY', 'PRF1', 'NKG7',
    'ZNF683', 'ITGAE', 'RBPJ',
    'LAG3', 'TIGIT', 'PDCD1', 'HAVCR2', 'CTLA4', 'CXCL13','TOX',
    'IL2RA', 'FOXP3', 'IKZF2',
    'IFITM3', 'IFI27', 'ISG15', 'ISG20',
    'SLC4A10', 'IL12RB', 'IL18R1', 'PLZF')

CD8T_markers <- c('TRAC', 'CD3E', 'CD4', 'CD8A',
    'TCF7', 'SELL', 'LEF1', 'CCR7',
    'IL7R', 'IFNG', 'GZMK', 'GZMA', 'GZMB', 'GNLY', 'PRF1', 'NKG7',
    'ZNF683', 'ITGAE', 'HOPX', 'RBPJ',
    'LAG3', 'TIGIT', 'PDCD1', 'HAVCR2', 'CTLA4', 'CXCL13','TOX',
    'IL2RA', 'FOXP3', 'IKZF2',
    'IFITM3', 'IFI27', 'ISG15', 'ISG20',
    'SLC4A10', 'IL12RB', 'IL18R1', 'PLZF')

CD4T_markers <- c('TRAC', 'CD3E', 'CD4', 'CD8A',
    'TCF7', 'LEF1', 'SELL', 'CCR7',
    'CXCR6', 'CD69', 'IL7R', 'KLRB1', 'PTGER4',
    'IFNG', 'GZMK', 'GZMA', 'GZMB', 'GNLY', 'PRF1', 'NKG7',
    'ZNF683', 'ITGAE', 'HOPX', 'RBPJ',
    'LAG3', 'TIGIT', 'PDCD1', 'HAVCR2', 'CTLA4', 'CXCL13','TOX',
    'IL2RA', 'FOXP3', 'IKZF2',
    'BCL6', 'IL17A') # CXCR5

In [ ]:
#T_Cells_markers
options(repr.plot.width = 16, repr.plot.height = 10)
DotPlot(samples_T_Cell_integrated, features = T_markers, group.by = 'T_subcluster_res_0.5', scale = T, col.max = 3) + 
    scale_y_discrete(limits=c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                              'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex',
                              'IFITM3_CD8_Teffector', 'ISG15_Teffector', 'MAIT')) +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', guide = guide_colorbar(order = 1), limits = c(-2,2), oob = scales::squish) +
    scale_size_area(max_size = 10, guide = guide_legend(order = 2)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.border = element_rect(linewidth = 1, fill = NA, color = 'black'),
          legend.title = element_text(size = 20),
          legend.position = 'top',
          axis.text = element_text(colour = 'black'),
          axis.line = element_blank(),
          axis.text.x = element_text(angle = 90),
          axis.title.x = element_blank(),
          axis.title.y = element_blank())

#C23339
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'T_Cells_markers_Dotplot.pdf'), device = 'pdf', width = 17, height = 8, bg = 'transparent')

In [ ]:
#cd4+_T_Cells_markers
Idents(samples_T_Cell_integrated) <- samples_T_Cell_integrated$'T_subcluster_res_0.5'
CD4_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg')

options(repr.plot.width = 16, repr.plot.height = 6)
DotPlot(samples_T_Cell_integrated, features = CD4T_markers, group.by = 'T_subcluster_res_0.5', scale = T, col.max = 3, idents = CD4_subcluster) + 
    scale_y_discrete(limits=c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg')) +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', guide = guide_colorbar(order = 1), limits = c(-2,2), oob = scales::squish) +
    scale_size_area(max_size = 10, guide = guide_legend(order = 2)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.border = element_rect(linewidth = 1, fill = NA, color = 'black'),
          legend.title = element_text(size = 20),
          legend.position = 'top',
          axis.text = element_text(colour = 'black'),
          axis.line = element_blank(),
          axis.text.x = element_text(angle = 90),
          axis.title.x = element_blank(),
          axis.title.y = element_blank())

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CD4_T_Cells_markers_Dotplot.pdf'), device = 'pdf', width = 15, height = 5, bg = 'transparent')

In [ ]:
#cd8+_T_Cells_markers
Idents(samples_T_Cell_integrated) <- samples_T_Cell_integrated$'T_subcluster_res_0.5'
CD8_subcluster <- c('GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex', 'GZMB_CD8_Teffector', 'IFITM3_CD8_Teffector',
                  'ISG15_Teffector', 'MAIT')

options(repr.plot.width = 16, repr.plot.height = 8)
DotPlot(samples_T_Cell_integrated, features = CD8T_markers, group.by = 'T_subcluster_res_0.5', scale = T, col.max = 3, idents = CD8_subcluster) + 
    scale_y_discrete(limits=c('GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex',
                              'IFITM3_CD8_Teffector', 'ISG15_Teffector', 'MAIT')) +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', guide = guide_colorbar(order = 1), limits = c(-2,2), oob = scales::squish) +
    scale_size_area(max_size = 10, guide = guide_legend(order = 2)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.border = element_rect(linewidth = 1, fill = NA, color = 'black'),
          legend.title = element_text(size = 20),
          legend.position = 'top',
          axis.text = element_text(colour = 'black'),
          axis.line = element_blank(),
          axis.text.x = element_text(angle = 90),
          axis.title.x = element_blank(),
          axis.title.y = element_blank())
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CD8_T_Cells_markers_Dotplot.pdf'), device = 'pdf', width = 15, height = 7, bg = 'transparent')

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
T_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex',
                 'IFITM3_CD8_Teffector', 'ISG15_Teffector', 'MAIT')

DimPlot(samples_T_Cell_integrated, group.by = 'T_subcluster_res_0.5', label = F, pt.size = 0.5, raster=FALSE, shuffle=F) +
    scale_color_manual(values = imm_subcelltype_colors, limits = T_subcluster) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'none',
          legend.key.height = unit(x = 0.5, units = 'in'),
          legend.key.size = unit(x = 0.5, units = 'in'))

#ggsave(filename = paste0(figures, '/Figures_raw', '/', 'T_cell_UMAP.pdf'), device = 'pdf', width = 6, height = 6, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'T_cell_UMAP.png'), device = 'png', width = 4, height = 6, dpi = 300, bg = 'transparent')

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

T_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex',
                 'IFITM3_CD8_Teffector', 'ISG15_Teffector', 'MAIT')

DimPlot(samples_T_Cell_integrated, group.by = 'T_subcluster_res_0.5', split.by = 'lung_condition', label = F, pt.size = 0.5, raster=FALSE, shuffle=F) +
    scale_color_manual(values = imm_subcelltype_colors, limits = T_subcluster) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          strip.text = element_blank(),
          legend.position = 'none',
          legend.key.height = unit(x = 0.5, units = 'in'),
          legend.key.size = unit(x = 0.5, units = 'in'))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'T_cell_Split_UMAP.png'), device = 'png', width = 12, height = 6, dpi = 300, bg = 'transparent')

In [ ]:
samples_T_Cell_integrated <- readRDS(file = paste0(obj, "/", "samples_T_Cell_integrated.rds"))

### Abundance analysis 

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 8)

T_selected_subclusters <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                  'IFITM3_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'CXCL13_CD8_Tex')

FetchData(samples_T_Cell_integrated, vars = c('lung_condition', 'Age_type', 'T_subcluster_res_0.5', 'sample')) %>%
    group_by(lung_condition, Age_type, sample, T_subcluster_res_0.5) %>%
    summarise(n_cell = n()) %>%
    mutate(n_Tcell = sum(n_cell)) %>%
    filter(n_Tcell > 200) %>%    # filter the samples whose n_Tcell less than 200
    mutate(proportion = n_cell/n_Tcell)  %>%
    filter(T_subcluster_res_0.5 %in% T_selected_subclusters) %>%
    mutate(T_subcluster_res_0.5 = factor(T_subcluster_res_0.5, levels = T_selected_subclusters)) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = lung_condition, y = proportion, fill = Age_type)) +
    geom_boxplot(position = position_dodge(width = 0.75),  width = 0.5, linewidth = 0.8) +
    geom_jitter(size = 3, shape = 21, position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75)) +
    facet_wrap(facets = ~T_subcluster_res_0.5, scales = 'free_y', nrow = 2) +
    stat_compare_means(mapping = aes(group = lung_condition), label = 'p.signif',
                       method = 'wilcox.test', size = 6, label.x.npc = c(0.5)) +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif',
                       method = 'wilcox.test', size = 6, label.y.npc = c(0.9)) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 2)) +
    scale_x_discrete(labels = c('Healthy' = 'Healthy', 'Tumor' = 'Diseased')) +
    scale_y_continuous(limits = c(0, NA)) +
    xlab(label = NULL) +
    ylab(label = 'Proportion Relative to T Cell') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'T_Cell_ProportionBoxPlot.pdf'), device = 'pdf', width = 20, height = 10, bg = 'transparent')


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
T_selected_subclusters <- c('ZNF683_CD8_Trm', 'ISG15_Teffector')

FetchData(samples_T_Cell_integrated, vars = c('lung_condition', 'Age_type', 'T_subcluster_res_0.5', 'sample')) %>%
    group_by(lung_condition, Age_type, sample, T_subcluster_res_0.5) %>%
    summarise(n_cell = n()) %>%
    mutate(n_Tcell = sum(n_cell)) %>%
    filter(n_Tcell > 200) %>%    # filter the samples whose n_Tcell less than 200
    mutate(proportion = n_cell/n_Tcell)  %>%
    filter(T_subcluster_res_0.5 %in% T_selected_subclusters) %>%
    mutate(T_subcluster_res_0.5 = factor(T_subcluster_res_0.5, levels = T_selected_subclusters)) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = lung_condition, y = proportion, fill = Age_type)) +
    geom_boxplot(position = position_dodge(width = 0.75),  width = 0.5, linewidth = 0.8) +
    geom_jitter(size = 3, shape = 21, position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75)) +
    facet_wrap(facets = ~T_subcluster_res_0.5, scales = 'free_y', nrow = 1) +
    stat_compare_means(mapping = aes(group = lung_condition), label = 'p.signif',
                       method = 'wilcox.test', size = 6, label.x.npc = c(0.5)) +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif',
                       method = 'wilcox.test', size = 6, label.y.npc = c(0.9)) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 2)) +
    scale_x_discrete(labels = c('Healthy' = 'Healthy', 'Tumor' = 'Diseased')) +
    scale_y_continuous(limits = c(0, NA)) +
    xlab(label = NULL) +
    ylab(label = 'Proportion Relative to T Cell') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'T_Cell_ProportionBoxPlot_Part2.pdf'), device = 'pdf', width = 10, height = 5, bg = 'transparent')

### DEGs analysis

In [ ]:
##split the samples_T_Cell_integrated into healthy and tumor
DefaultAssay(samples_T_Cell_integrated) <- "RNA"
Idents(samples_T_Cell_integrated) <- samples_T_Cell_integrated$`lung_condition`

samples_T_Cell_healthy_integrated <- subset(samples_T_Cell_integrated, idents = 'Healthy')

samples_T_Cell_tumor_integrated <- subset(samples_T_Cell_integrated, idents = 'Tumor')

In [ ]:
unique(samples_T_Cell_integrated$`T_subcluster_res_0.5`)

In [ ]:
##find DEGs between T_Cell_healthy_young and T_Cell_healthy_old with wilcox_test
Idents(samples_T_Cell_healthy_integrated) <- samples_T_Cell_healthy_integrated$`T_subcluster_res_0.5`
T_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex',
                 'IFITM3_CD8_Teffector', 'ISG15_Teffector', 'MAIT')

for ( i in T_subcluster) {
    print(i)
    assign(x = paste0('DEGs_T_Cell_', i, '_young_old_healthy_wilcox'), 
            value = FindMarkers(object = samples_T_Cell_healthy_integrated, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                            logfc.threshold = 0, min.pct = 0.1, group.by = 'Age_type', subset.ident = i))
    
    saveRDS(get(paste0('DEGs_T_Cell_', i, '_young_old_healthy_wilcox')), paste0(obj, '/', 'DEGs_T_Cell_healthy/', paste0('DEGs_T_Cell_', i, '_young_old_healthy_wilcox.rds')))
}

In [ ]:
##find DEGs between T_Cell_tumor_young and T_Cell_tumor_old with wilcox_test
Idents(samples_T_Cell_tumor_integrated) <- samples_T_Cell_tumor_integrated$`T_subcluster_res_0.5`
T_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex',
                 'IFITM3_CD8_Teffector', 'ISG15_Teffector', 'MAIT')

for ( i in T_subcluster) {
    print(i)
    assign(x = paste0('DEGs_T_Cell_', i, '_young_old_tumor_wilcox'), 
           value = FindMarkers(object = samples_T_Cell_tumor_integrated, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                          logfc.threshold = 0., min.pct = 0.2, group.by = 'Age_type', subset.ident = i))
    
    saveRDS(get(paste0('DEGs_T_Cell_', i, '_young_old_tumor_wilcox')), paste0(obj, '/', 'DEGs_T_Cell_tumor/', paste0('DEGs_T_Cell_', i, '_young_old_tumor_wilcox.rds')))
}
    

In [ ]:
print("done")

In [ ]:
# T_cell_healthy_subcluster DEGs gene numbers
T_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex', 'IFITM3_CD8_Teffector', 'ISG15_Teffector', 'MAIT')

subcluster_vec <- vector(mode = "character")
nDEGs_vec <- vector(mode = "numeric")

for ( i in T_subcluster) {
    DEGs_T_Cell_healthy_df <- readRDS(paste0(obj, '/', 'DEGs_T_Cell_healthy/', 'DEGs_T_Cell_', i, '_young_old_healthy_wilcox.rds'))
    nDEGs <- (DEGs_T_Cell_healthy_df %>% filter(abs(avg_log2FC) > 0.5) %>% filter(p_val_adj < 0.05) %>% dim())[1]
    subcluster_vec <- c(subcluster_vec, i)
    nDEGs_vec <- c(nDEGs_vec, nDEGs)
    nDEGs_T_Cell_healthy_df <- data.frame(subcluster_vec, nDEGs_vec)
    # cat(paste0('DEGs_imm_', i, '_young_old_wilcox', ':', nDEGs, '\n'))
}
nDEGs_T_Cell_healthy_df <- nDEGs_T_Cell_healthy_df %>% 
    filter(!subcluster_vec %in% c('MAIT')) %>% 
    arrange(desc(nDEGs_vec)) %>%
    mutate(lung_condition = 'Healthy')

In [ ]:
nDEGs_T_Cell_healthy_df

In [ ]:
49 * 4

In [ ]:
# T_cell_tumor_subcluster  DEGs gene numbers
T_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex', 'IFITM3_CD8_Teffector', 'ISG15_Teffector', 'MAIT')

subcluster_vec <- vector(mode = "character")
nDEGs_vec <- vector(mode = "numeric")


for ( i in T_subcluster) {
    DEGs_T_Cell_tumor_df <- readRDS(paste0(obj, '/', 'DEGs_T_Cell_tumor/', 'DEGs_T_Cell_', i, '_young_old_tumor_wilcox.rds'))
    nDEGs <- (DEGs_T_Cell_tumor_df %>% filter(abs(avg_log2FC) > 0.5) %>% filter(p_val_adj < 0.05) %>% dim())[1]
    subcluster_vec <- c(subcluster_vec, i)
    nDEGs_vec <- c(nDEGs_vec, nDEGs)
    nDEGs_T_Cell_tumor_df <- data.frame(subcluster_vec, nDEGs_vec)
    # cat(paste0('DEGs_imm_', i, '_young_old_wilcox', ':', nDEGs, '\n'))
}
nDEGs_T_Cell_tumor_df <- nDEGs_T_Cell_tumor_df %>%
    filter(!subcluster_vec %in% c('MAIT')) %>%
    arrange(desc(nDEGs_vec)) %>%
    mutate(lung_condition = 'Diseased')

In [ ]:
nDEGs_T_Cell_tumor_df

In [ ]:
# T_cell_subcluster_df  DEGs gene numbers (plot)
options(repr.plot.width = 10, repr.plot.height = 10)
nDEGs_T_Cell_df <- rbind(nDEGs_T_Cell_healthy_df, nDEGs_T_Cell_tumor_df) %>%
    mutate(lung_condition = factor(lung_condition, levels = c('Healthy', 'Diseased')))



ggplot(data = nDEGs_T_Cell_df) +
    geom_col(mapping = aes(x = fct_reorder(subcluster_vec, .x = nDEGs_vec, .fun = max, .desc = F),
                           y = nDEGs_vec, fill = subcluster_vec, linetype = lung_condition),
             color = 'black', position = position_dodge(width = 1), linewidth = 1, width = 0.8) +
    scale_fill_manual(values = imm_subcelltype_colors,
                      limits = T_subcluster,
                      guide = guide_legend(title = 'Cell Type', nrow = 2,
                                           title.theme = element_text(size = 20),
                                           order = 2)) +
    scale_linetype_manual(values = c(Healthy = 'dashed', Diseased = 'solid'),
                          limits = c('Healthy', 'Diseased'),
                          guide = guide_legend(title = 'Lung Condition',
                                               title.theme = element_text(size = 20),
                                               order = 1)) +
    xlab(label = NULL) +
    ylab(label = 'The Number of DEGs') +
    scale_y_continuous(expand = expansion(mult = c(0.03))) +
    coord_flip() +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.grid = element_blank(),
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
#          axis.text.x = element_text(angle = 25),
          legend.position = 'top',
          legend.title = element_text(size = 20),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))


ggsave(filename = paste0(figures, '/Figures_raw', '/', 'T_Subclsuter_nDEGs_BarPlot.pdf'), device = 'pdf', width = 14, height = 10, bg = 'transparent')

### GSEA analysis

#### CXCL13_CD8

In [ ]:
msigdb_c5_BP <- msigdbr(species = 'Homo sapiens', category = "C5",subcategory = "BP")
msigdb_c5_BP_list <- msigdb_c5_BP %>% split(x = .$gene_symbol, f = .$gs_name)

In [ ]:
#DEGs
DEGs_CXCL13_CD8_young_old_tumor_wilcox <- readRDS(paste0(obj, '/DEGs_T_Cell_tumor', '/DEGs_T_Cell_CXCL13_CD8_Tex_young_old_tumor_wilcox.rds'))
DEGs_CXCL13_CD8_young_old_healthy_wilcox <-  readRDS(paste0(obj, '/DEGs_T_Cell_healthy', '/DEGs_T_Cell_CXCL13_CD8_Tex_young_old_healthy_wilcox.rds'))

In [ ]:
OrderGene_GSEA <- function(DEGs_wilcox_df) {
    
    DEGs_wilcox_df_list<-DEGs_wilcox_df[['avg_log2FC']] # 提取排序值 only about thousands genes (some genes are filtered due to low expression pct in wilcox test)
    names(DEGs_wilcox_df_list)<-rownames(DEGs_wilcox_df) #列名定义为基因名
    DEGs_wilcox_df_list <- sort(DEGs_wilcox_df_list, decreasing = T) #按排序值进行排序，制作基因列表

    return(DEGs_wilcox_df_list)
}

In [ ]:
fgsea_GOBP_CXCL13_CD8_young_old_tumor = fgsea(pathways = msigdb_c5_BP_list, stats = OrderGene_GSEA(DEGs_wilcox_df = DEGs_CXCL13_CD8_young_old_tumor_wilcox))

In [ ]:
fgsea_GOBP_CXCL13_CD8_young_old_tumor %>% filter(padj < 0.05) %>% arrange(NES)

In [ ]:
# Visulization GSEA using aPEAR
options(repr.plot.width = 14, repr.plot.height = 14)

set.seed(3)

GSEA_aPEAR_df <- fgsea_GOBP_CXCL13_CD8_young_old_tumor %>% 
    mutate(pathway = str_remove(pathway, '^GOBP_'),
           pathway = gsub(x = pathway, pattern = '_', replacement = ' '),
           pathway = str_to_title(pathway),) %>%
    filter(padj < 0.05) %>%
    select(c('pathway', 'padj', 'NES', 'size', 'leadingEdge')) %>%
    rename(Description = pathway, pathwayGenes = leadingEdge) %>%
    mutate(pathwayGenes = as.character(pathwayGenes)) %>%
    mutate(pathwayGenes = gsub(x = pathwayGenes, pattern = 'c|\\(|\\)|"|', replacement = "")) %>%
    mutate(pathwayGenes = str_replace_all(string = pathwayGenes, pattern = ', ', replacement = '/'))

aPEAR::enrichmentNetwork(enrichment = GSEA_aPEAR_df, colorBy = 'NES', nodeSize = 'size', verbose = F,
                  fontSize = 4, minClusterSize = 4, cluster = 'hier', clusterNameColumn = 'padj', clusterName = 'pval') +
    scale_color_gradient2(low = "#2164AA", mid = "white", high = "#DC2829", midpoint = 0, limits = c(-2.5, 2.5))

#enrichmentNetwork(enrichment = GSEA_aPEAR_df, colorBy = 'NES', nodeSize = 'size', verbose = F,
#                  fontSize = 4, minClusterSize = 4, cluster = 'hier') +
#    scale_color_gradient2(low = "#2164AA", mid = "white", high = "#DC2829", midpoint = 0, limits = c(-2.5, 2.5))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)
GSEA_aPEAR_df %>% 
    mutate(Description_adjust = ifelse(-log10(padj) > 4 | NES >= 2, Description, '')) %>%
    mutate(Description_adjust = str_wrap(Description_adjust, width = 20)) %>%
    ggplot(mapping = aes(x =  -log10(padj), y = NES)) +
    geom_point(mapping = aes(fill = NES, size = -log10(padj)), shape = 21) +
    ggrepel::geom_text_repel(mapping = aes(label = Description_adjust),
                             max.overlaps = 700, nudge_y = 1.4, size =3, min.segment.length = 3) +
    scale_fill_gradient2(low = '#3C7BB0', mid = '#D1D4D3', high = '#E5368E',
                         midpoint = 0, limits = c(-2, 2), oob = scales::squish,
                         guide = guide_colorbar(title.theme = element_text(size = 20),
                                                barwidth = 10, barheight = 2)) +
    scale_size_area(max_size = 10) +
    labs(x = '-Log(Padj)') +
    theme_bw(base_size = 25) +
    transparent_bg +
    theme(panel.grid = element_blank(),
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
#          axis.text.x = element_text(angle = 25),
          legend.position = 'top',
          legend.title = element_text(size = 20),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CXCL13_CD8_T_Cells(only_Diseased)_GSEA_PointPlot.pdf'), device = 'pdf', width = 12, height = 8, bg = 'transparent')

#### CXCL13_CD4

In [ ]:
msigdb_c5_BP <- msigdbr(species = 'Homo sapiens', category = "C5",subcategory = "BP")
msigdb_c5_BP_list <- msigdb_c5_BP %>% split(x = .$gene_symbol, f = .$gs_name)

In [ ]:
#DEGs
DEGs_CXCL13_CD4_young_old_tumor_wilcox <- readRDS(paste0(obj, '/DEGs_T_Cell_tumor', '/DEGs_T_Cell_CXCL13_CD4_Tex_young_old_tumor_wilcox.rds'))
DEGs_CXCL13_CD4_young_old_healthy_wilcox <-  readRDS(paste0(obj, '/DEGs_T_Cell_healthy', '/DEGs_T_Cell_CXCL13_CD4_Tex_young_old_healthy_wilcox.rds'))

In [ ]:
OrderGene_GSEA <- function(DEGs_wilcox_df) {
    
    DEGs_wilcox_df_list<-DEGs_wilcox_df[['avg_log2FC']] # 提取排序值 only about thousands genes (some genes are filtered due to low expression pct in wilcox test)
    names(DEGs_wilcox_df_list)<-rownames(DEGs_wilcox_df) #列名定义为基因名
    DEGs_wilcox_df_list <- sort(DEGs_wilcox_df_list, decreasing = T) #按排序值进行排序，制作基因列表

    return(DEGs_wilcox_df_list)
}

In [ ]:
fgsea_GOBP_CXCL13_CD4_young_old_tumor = fgsea(pathways = msigdb_c5_BP_list, stats = OrderGene_GSEA(DEGs_wilcox_df = DEGs_CXCL13_CD4_young_old_tumor_wilcox))

In [ ]:
fgsea_GOBP_CXCL13_CD4_young_old_tumor %>% filter(padj < 0.05) %>% arrange(NES)

In [ ]:
# Visulization GSEA using aPEAR
options(repr.plot.width = 14, repr.plot.height = 14)

set.seed(3)

GSEA_aPEAR_df <- fgsea_GOBP_CXCL13_CD4_young_old_tumor %>% 
    mutate(pathway = str_remove(pathway, '^GOBP_'),
           pathway = gsub(x = pathway, pattern = '_', replacement = ' '),
           pathway = str_to_title(pathway),) %>%
    filter(padj < 0.05) %>%
    select(c('pathway', 'padj', 'NES', 'size', 'leadingEdge')) %>%
    rename(Description = pathway, pathwayGenes = leadingEdge) %>%
    mutate(pathwayGenes = as.character(pathwayGenes)) %>%
    mutate(pathwayGenes = gsub(x = pathwayGenes, pattern = 'c|\\(|\\)|"|', replacement = "")) %>%
    mutate(pathwayGenes = str_replace_all(string = pathwayGenes, pattern = ', ', replacement = '/'))

aPEAR::enrichmentNetwork(enrichment = GSEA_aPEAR_df, colorBy = 'NES', nodeSize = 'size', verbose = F,
                  fontSize = 4, minClusterSize = 4, cluster = 'hier', clusterNameColumn = 'padj', clusterName = 'pval') +
    scale_color_gradient2(low = "#2164AA", mid = "white", high = "#DC2829", midpoint = 0, limits = c(-2.5, 2.5))

#enrichmentNetwork(enrichment = GSEA_aPEAR_df, colorBy = 'NES', nodeSize = 'size', verbose = F,
#                  fontSize = 4, minClusterSize = 4, cluster = 'hier') +
#    scale_color_gradient2(low = "#2164AA", mid = "white", high = "#DC2829", midpoint = 0, limits = c(-2.5, 2.5))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)
GSEA_aPEAR_df %>% 
    mutate(Description_adjust = ifelse(-log10(padj) > 3.5 | NES >= 2, Description, '')) %>%
    mutate(Description_adjust = str_wrap(Description_adjust, width = 20)) %>%
    ggplot(mapping = aes(x =  -log10(padj), y = NES)) +
    geom_point(mapping = aes(fill = NES, size = -log10(padj)), shape = 21) +
    ggrepel::geom_text_repel(mapping = aes(label = Description_adjust),
                             max.overlaps = 700, nudge_y = 1.4, size =3, min.segment.length = 3) +
    scale_fill_gradient2(low = '#3C7BB0', mid = '#D1D4D3', high = '#E5368E',
                         midpoint = 0, limits = c(-2, 2), oob = scales::squish,
                         guide = guide_colorbar(title.theme = element_text(size = 20),
                                                barwidth = 10, barheight = 2)) +
    scale_size_area(max_size = 10) +
    labs(x = '-Log(Padj)') +
    theme_bw(base_size = 25) +
    transparent_bg +
    theme(panel.grid = element_blank(),
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
#          axis.text.x = element_text(angle = 25),
          legend.position = 'top',
          legend.title = element_text(size = 20),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CXCL13_CD4_T_Cells(only_Diseased)_GSEA_PointPlot.pdf'), device = 'pdf', width = 12, height = 8, bg = 'transparent')

### Signature score analysis

#### Cell level

In [ ]:
Idents(samples_T_Cell_integrated) <- samples_T_Cell_integrated$`T_subcluster_res_0.5`
samples_CD8T_Cell_integrated <- subset(samples_T_Cell_integrated,
                                       idents = c('ZNF683_CD8_Trm', 'GZMK_CD8_Tem', 'GZMB_CD8_Teffector',
                                                  'CXCL13_CD8_Tex', 'IFITM3_CD8_Teffector'))
samples_CD4T_Cell_integrated <- subset(samples_T_Cell_integrated,
                                       idents = c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm',
                                                  'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg'))

In [ ]:
# curated gene set
## cd8_tumor_reactivity_signature ('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
CD8_T_tumor_reactivity_signature <- c('CXCL13', 'ENTPD1', 'BATF', 'GZMB', 'CD27', 'TIGIT', 'PHLDA1',
                                    'CD74', 'HLA-DMA', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'CD3D', 'CD82',
                                    'ARL3', 'HMOX1', 'ALOX5AP', 'DUSP4', 'CARS', 'LSP1', 'CCND2',
                                    'TPI1', 'GAPDH', 'ITM2A', 'HMGN3', 'CHST12', 'NAP1L4')

## cd4_tumor_reactivity_signature ('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
CD4_T_tumor_reactivity_signature <- c('CXCL13', 'NR3C1', 'ADGRG1', 'NMG', 'ITM2A', 'ETV7', 'COTL1', 'B2M', 'IGFL2')


In [ ]:
# module score
samples_CD8T_Cell_integrated <- AddModuleScore(object = samples_CD8T_Cell_integrated, features = list(CD8_T_tumor_reactivity_signature),
                                               seed = 42, name = 'CD8_T_Tumor_Reactivity_Score')

samples_CD4T_Cell_integrated <- AddModuleScore(object = samples_CD4T_Cell_integrated, features = list(CD4_T_tumor_reactivity_signature),
                                               seed = 42, name = 'CD4_T_Tumor_Reactivity_Score')


In [ ]:
#CD8_T_Tumor_Reactivity_Score('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
options(repr.plot.width = 12, repr.plot.height = 10)
FetchData(samples_CD8T_Cell_integrated, vars = c('T_subcluster_res_0.5', 'Age_type', 'sample', 'CD8_T_Tumor_Reactivity_Score1')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = T_subcluster_res_0.5, y = CD8_T_Tumor_Reactivity_Score1, fill = T_subcluster_res_0.5)) +
    geom_violin(linewidth=1, position = position_dodge(0.7), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.7)) +
    stat_compare_means(mapping = aes(x = T_subcluster_res_0.5, y = CD8_T_Tumor_Reactivity_Score1), label = 'p.signif', method = 'wilcox.test',
                       comparisons = list(c('CXCL13_CD8_Tex', 'IFITM3_CD8_Teffector'), c('GZMB_CD8_Teffector', 'CXCL13_CD8_Tex'),
                                          c('GZMK_CD8_Tem', 'CXCL13_CD8_Tex'), c('ZNF683_CD8_Trm', 'CXCL13_CD8_Tex')),
                       label.y = c(1.6, 1.8, 2, 2.2), tip.length = 0.015) +
    scale_fill_manual(values = imm_subcelltype_colors, guide = guide_legend(title = element_blank(), nrow = 2)) +
    scale_x_discrete(limits = c('IFITM3_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex')) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'CD8+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CD8_T_Cells_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

In [ ]:
#CD4_T_Tumor_Reactivity_Score('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')

FetchData(samples_CD4T_Cell_integrated, vars = c('T_subcluster_res_0.5', 'Age_type', 'sample', 'CD4_T_Tumor_Reactivity_Score1')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = T_subcluster_res_0.5, y = CD4_T_Tumor_Reactivity_Score1, fill = T_subcluster_res_0.5)) +
    geom_violin(linewidth=1, position = position_dodge(0.7), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.7)) +
    stat_compare_means(mapping = aes(x = T_subcluster_res_0.5, y = CD4_T_Tumor_Reactivity_Score1), label = 'p.signif', method = 'wilcox.test',
                       comparisons = list(c('CCR7_CD4_Tnaive', 'CXCL13_CD4_Tex'), c('CXCR6_CD4_Trm', 'CXCL13_CD4_Tex'),
                                          c('FOXP3_CD4_Treg', 'CXCL13_CD4_Tex')),
                       label.y = c(2.6, 2.8, 2.6), tip.length = 0.015) +
    scale_fill_manual(values = imm_subcelltype_colors, guide = guide_legend(title = element_blank(), nrow = 2)) +
    scale_x_discrete(limits = c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg')) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'CD4+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CD4_T_Cells_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', width = 10, height = 10, bg = 'transparent')


In [ ]:
#CXCL13_CD8_Tumor_Reactivity_Score(only diseased)('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
FetchData(samples_CD8T_Cell_integrated, vars = c('T_subcluster_res_0.5', 'Age_type', 'lung_condition', 'CD8_T_Tumor_Reactivity_Score1')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(lung_condition == 'Tumor' & T_subcluster_res_0.5 == 'CXCL13_CD8_Tex') %>%
    ggplot(mapping = aes(x = Age_type, y = CD8_T_Tumor_Reactivity_Score1, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.8), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.8)) +
    stat_compare_means(comparisons = list(c('Young', 'Old')), label = 'p.format', method = 'wilcox.test') +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = 'Age Group', nrow = 1)) +
    geom_hline(yintercept = 0, linetype='dashed') +
#    labs(title = 'CXCL13_CD8_Tex') +
    xlab(label = NULL) +
    ylab(label = 'CD8+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CXCL13_CD8_T_Cells(only_Diseased)_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

In [ ]:
#CXCL13_CD4_Tumor_Reactivity_Score(only diseased)('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
FetchData(samples_CD4T_Cell_integrated, vars = c('T_subcluster_res_0.5', 'Age_type', 'lung_condition', 'CD4_T_Tumor_Reactivity_Score1')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(lung_condition == 'Tumor' & T_subcluster_res_0.5 == 'CXCL13_CD4_Tex') %>%
    ggplot(mapping = aes(x = Age_type, y = CD4_T_Tumor_Reactivity_Score1, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.8), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.8)) +
    stat_compare_means(comparisons = list(c('Young', 'Old')), label = 'p.format', method = 'wilcox.test') +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = 'Age Group', nrow = 1)) +
    geom_hline(yintercept = 0, linetype='dashed') +
#    labs(title = 'CXCL13_CD8_Tex') +
    xlab(label = NULL) +
    ylab(label = 'CD4+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CXCL13_CD4_T_Cells(only_Diseased)_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

In [ ]:
#CXCL13_CD8_Tumor_Reactivity_Score('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
FetchData(samples_CD8T_Cell_integrated, vars = c('T_subcluster_res_0.5', 'Age_type', 'lung_condition', 'CD8_T_Tumor_Reactivity_Score1')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(T_subcluster_res_0.5 == 'CXCL13_CD8_Tex') %>%
    ggplot(mapping = aes(x = lung_condition, y = CD8_T_Tumor_Reactivity_Score1, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.8), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.8)) +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif', method = 'wilcox.test') +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = 'Age Group', nrow = 1)) +
    scale_x_discrete(limits = c('Healthy', 'Tumor'), labels = c('Healthy', 'Diseased')) +
    geom_hline(yintercept = 0, linetype='dashed') +
#    labs(title = 'CXCL13_CD8_Tex') +
    xlab(label = NULL) +
    ylab(label = 'CD8+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CXCL13_CD8_T_Cells_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

In [ ]:
#CXCL13_CD4_Tumor_Reactivity_Score('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
FetchData(samples_CD4T_Cell_integrated, vars = c('T_subcluster_res_0.5', 'Age_type', 'lung_condition', 'CD4_T_Tumor_Reactivity_Score1')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(T_subcluster_res_0.5 == 'CXCL13_CD4_Tex') %>%
    ggplot(mapping = aes(x = lung_condition, y = CD4_T_Tumor_Reactivity_Score1, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.8), alpha = 0.9) +
    geom_boxplot(notch=F, width=0.2, linewidth=1, position = position_dodge(0.8)) +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif', method = 'wilcox.test') +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = 'Age Group', nrow = 1)) +
    scale_x_discrete(limits = c('Healthy', 'Tumor'), labels = c('Healthy', 'Diseased')) +
    geom_hline(yintercept = 0, linetype='dashed') +
#    labs(title = 'CXCL13_CD8_Tex') +
    xlab(label = NULL) +
    ylab(label = 'CD4+ T Cell Tumor Reactivity Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CXCL13_CD4_T_Cells_Tumor_Reactivity_Score_VlnPlot.pdf'), device = 'pdf', bg = 'transparent', width = 10, height = 10)

## Samples_B_Cell_analysis

### Subset

In [ ]:
Idents(samples_imm_integrated) <- samples_imm_integrated$`ImmMaincluster_res0.9`
levels(samples_imm_integrated)

In [ ]:
# subset B_Cell
Idents(samples_imm_integrated) <- samples_imm_integrated$`ImmMaincluster_res0.9`
samples_B_Cell_integrated <- subset(samples_imm_integrated, idents = c('B_Cell'))
samples_B_Cell_integrated

### Clustring and annotation analysis

In [ ]:
# UMAP
samples_B_Cell_integrated <- RunUMAP(samples_B_Cell_integrated, dims = 1:50, reduction = "harmony", verbose = F, seed.use = 24 )


In [ ]:
# FindNeighbors
samples_B_Cell_integrated <- FindNeighbors(samples_B_Cell_integrated, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
options(repr.plot.width = 8, repr.plot.height = 8)
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  samples_B_Cell_integrated <- FindClusters(samples_B_Cell_integrated, resolution = i, verbose = F)
  print(DimPlot(samples_B_Cell_integrated, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
options(repr.plot.height =12, repr.plot.width = 12)
B_markers <- c('TCL1A', 'CD27', 'CD83', 'BCL6')

FeaturePlot(object = samples_B_Cell_integrated, features = B_markers, order = T)

In [ ]:
##B_Cell subclass annotation_res0.2
options(repr.plot.width = 12, repr.plot.height = 12)
Idents(samples_B_Cell_integrated) <- samples_B_Cell_integrated$`RNA_snn_res.0.2`


B_Cell_subcluster_anno <- c('0' = 'Memory_B', '1' = 'Memory_B', '2' = 'Naive_B', '3' = 'Memory_B')


#names(imm_maincluster_anno) <- levels(all_22samples_imm_integrated)
samples_B_Cell_integrated <- RenameIdents(samples_B_Cell_integrated, B_Cell_subcluster_anno)
samples_B_Cell_integrated$B_subcluster_res_0.2 <- Idents(samples_B_Cell_integrated)

DimPlot(samples_B_Cell_integrated, label = T, pt.size = 1, label.size = 7, repel = T) +
theme(plot.title = element_text(size = 30),
      legend.text = element_text(size = 20),
      legend.key.size = unit(0.5, "inches")) +
guides(colour = guide_legend(override.aes = list(size = 5)))

ggsave(paste0(figures, '/', "B_Cell_subcluster_annotation_res_0.2.png"), width = 10, height = 10)

In [ ]:
B_subcluster <- c('Naive_B', 'Memory_B')

DimPlot(samples_B_Cell_integrated, group.by = 'B_subcluster_res_0.2', label = F, pt.size = 1.5, raster=FALSE, shuffle=F) +
    scale_color_manual(values = imm_subcelltype_colors, limits = B_subcluster) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'none',
          legend.key.height = unit(x = 0.5, units = 'in'),
          legend.key.size = unit(x = 0.5, units = 'in'))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'B_cell_UMAP.pdf'), device = 'pdf', width = 6, height = 6, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'B_cell_UMAP.png'), device = 'png', width = 6, height = 6, dpi = 300, bg = 'transparent')

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 5)
#'#E2E1EF'
FeaturePlot(samples_B_Cell_integrated, features = c('TCL1A', 'CD27'), pt.size = 1, order = T) *
    scale_color_gradient(low = '#E2E1EF', high = '#C23339') *
    theme_classic(base_size = 12) *
    transparent_bg *
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'right')

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'B_cell_markergene_FeaturePlot.pdf'), device = 'pdf', width = 11, height = 5, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'B_cell_markergene_FeaturePlot.png'), device = 'png', width = 11, height = 5, dpi = 300, bg = 'transparent')

In [ ]:
saveRDS(samples_B_Cell_integrated, paste0(obj, '/', 'samples_B_Cell_integrated.rds'))

In [ ]:
samples_B_Cell_integrated <- readRDS(paste0(obj, '/', 'samples_B_Cell_integrated.rds'))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
FeaturePlot(samples_B_Cell_integrated, features = c('S1PI2', 'LRMP', 'SUGCT', 'MME', 'BCL6', 'AICDA'), order=T, pt.size = 0.6)

### Abundance analysis

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
FetchData(samples_B_Cell_integrated, vars = c('lung_condition', 'Age_type', 'B_subcluster_res_0.2', 'sample')) %>%
    group_by(lung_condition, Age_type, sample, B_subcluster_res_0.2) %>%
    summarise(n_cell = n()) %>%
    mutate(n_Bcell = sum(n_cell)) %>%
    filter(n_Bcell > 20) %>%    # filter the samples whose n_Bcell less than 20
    mutate(proportion = n_cell/n_Bcell) %>%
    mutate(B_subcluster_res_0.2 = factor(B_subcluster_res_0.2, levels = c('Naive_B', 'Memory_B'))) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = lung_condition, y = proportion, fill = Age_type)) +
    geom_boxplot(position = position_dodge(width = 0.75),  width = 0.5, linewidth = 0.8) +
    geom_jitter(size = 3, shape = 21, position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75)) +
    facet_wrap(facets = ~B_subcluster_res_0.2, scales = 'free_y', nrow = 1) +
    stat_compare_means(mapping = aes(group = lung_condition), label = 'p.signif',
                       method = 'wilcox.test', size = 6, label.x.npc = c(0.5)) +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif',
                       method = 'wilcox.test', size = 6, label.y.npc = c(0.9)) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 2)) +
    scale_x_discrete(labels = c('Healthy' = 'Healthy', 'Tumor' = 'Diseased')) +
    scale_y_continuous(limits = c(0, NA)) +
    xlab(label = NULL) +
    ylab(label = 'Proportion Relative to B Cell') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'B_Cell_ProportionBoxPlot.pdf'), device = 'pdf', width = 10, height = 5, bg = 'transparent')


### DEGs analysis

In [ ]:
samples_B_Cell_integrated <- readRDS(paste0(obj, '/', 'samples_B_Cell_integrated.rds'))

In [ ]:
##split the samples_B_Cell_integrated into healthy and tumor
DefaultAssay(samples_B_Cell_integrated) <- "RNA"
Idents(samples_B_Cell_integrated) <- samples_B_Cell_integrated$`lung_condition`

samples_B_Cell_healthy_integrated <- subset(samples_B_Cell_integrated, idents = 'Healthy')

samples_B_Cell_tumor_integrated <- subset(samples_B_Cell_integrated, idents = 'Tumor')

In [ ]:
unique(samples_B_Cell_integrated$`B_subcluster_res_0.2`)

In [ ]:
##find DEGs between B_Cell_healthy_young and B_Cell_healthy_old with wilcox_test
Idents(samples_B_Cell_healthy_integrated) <- samples_B_Cell_healthy_integrated$`B_subcluster_res_0.2`
B_subcluster <- c('Naive_B', 'Memory_B')

for ( i in B_subcluster) {
    print(i)
    assign(x = paste0('DEGs_B_Cell_', i, '_young_old_healthy_wilcox'), 
            value = FindMarkers(object = samples_B_Cell_healthy_integrated, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                            logfc.threshold = 0, min.pct = 0.1, group.by = 'Age_type', subset.ident = i))
    
    saveRDS(get(paste0('DEGs_B_Cell_', i, '_young_old_healthy_wilcox')), paste0(obj, '/', 'DEGs_B_Cell_healthy/', paste0('DEGs_B_Cell_', i, '_young_old_healthy_wilcox.rds')))
}

In [ ]:
##find DEGs between B_Cell_tumor_young and B_Cell_tumor_old with wilcox_test
Idents(samples_B_Cell_tumor_integrated) <- samples_B_Cell_tumor_integrated$`B_subcluster_res_0.2`
B_subcluster <- c('Naive_B', 'Memory_B')

for ( i in B_subcluster) {
    print(i)
    assign(x = paste0('DEGs_B_Cell_', i, '_young_old_tumor_wilcox'), 
           value = FindMarkers(object = samples_B_Cell_tumor_integrated, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                          logfc.threshold = 0, min.pct = 0.1, group.by = 'Age_type', subset.ident = i))
    
    saveRDS(get(paste0('DEGs_B_Cell_', i, '_young_old_tumor_wilcox')), paste0(obj, '/', 'DEGs_B_Cell_tumor/', paste0('DEGs_B_Cell_', i, '_young_old_tumor_wilcox.rds')))
}
    

In [ ]:
# MHCII_genes
MHCII_genes <- c('HLA-DRA', 'HLA-DRB5', 'HLA-DRB1', 'HLA-DQA1',
                 'HLA-DQB1', 'HLA-DQA2', 'HLA-DMB', 'HLA-DMA',
                 'HLA-DPA1', 'HLA-DPB1', 'HLA-DPB2', 'HLA-DRB6')

In [ ]:
# B_function_genes
#B_function_genes <- c(MHCII_genes, 'CTSD', 'TAP1', 'B2M', 'PSME1', 'PSME2',
#                      'IGHD','IGHM', 'IGHA1', 'IGHA2', 'IGHG1', 'IGHG2', #IgG
#                      'ISG20', 'IFITM1', 'IFI16', 'IFITM2','IRF8', #IFN
#                      'HSPA1A', 'HSPA1B', 'TUBB4B', 'GAPDH', 'MYC', # proliferating
#                      'KMT2C', 'CREBBP', 'KDM5A', 'JARID2', 'TET2')

# B_function_genes
#B_function_genes <- c(MHCII_genes, 'CTSD', 'TAP1', 'B2M', 'PSME1', 'PSME2',
#                      'IGHD','IGHM', 'IGHA1', 'IGHA2', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4',#IgG
#                      'ISG20', 'IFITM1', 'IFI16', 'IFITM2','IRF8', #IFN
#                      'HSPA1A', 'HSPA1B', 'TUBB4B', 'GAPDH', 'MYC')

# B_function_genes
B_function_genes <- c(MHCII_genes, 'CTSD', 'TAP1', 'B2M', 'PSME1', 'PSME2',
                      'ISG20', 'IFITM1', 'IFI16', 'IFITM2','IRF8', #IFN
                      'HSPA1A', 'HSPA1B', 'TUBB4B', 'GAPDH', 'MYC')

In [ ]:
#add the lung_condition_Age_type_Cell_type idents
samples_B_Cell_integrated <- FetchData(samples_B_Cell_integrated, vars = c('lung_condition', 'Age_type', 'B_subcluster_res_0.2')) %>%
    unite(col = 'lung_condition_Age_type_Cell_type', c('lung_condition', 'Age_type', 'B_subcluster_res_0.2')) %>%
    AddMetaData(object = samples_B_Cell_integrated)

# plot
options(repr.plot.width = 10, repr.plot.height = 10)
DotPlot(samples_B_Cell_integrated, features = B_function_genes, group.by = 'lung_condition_Age_type_Cell_type', dot.scale = 10) +
    scale_y_discrete(limits = c('Healthy_Young_Naive_B', 'Healthy_Young_Memory_B', 'Healthy_Old_Naive_B', 'Healthy_Old_Memory_B',
                                'Tumor_Young_Naive_B', 'Tumor_Young_Memory_B', 'Tumor_Old_Naive_B', 'Tumor_Old_Memory_B'),
                     labels = c('Healthy_Young_Naive_B', 'Healthy_Young_Memory_B', 'Healthy_Old_Naive_B', 'Healthy_Old_Memory_B',
                                'Diseased_Young_Naive_B', 'Diseased_Young_Memory_B', 'Diseased_Old_Naive_B', 'Diseased_Old_Memory_B')) +
#    scale_color_gradient(low = '#E2E1EF', high = '#C23339') +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', limits = c(-1,2), oob = scales::squish) +
    labs(x = NULL, y = NULL) +
    coord_flip() +
    theme_bw(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'right',
          legend.title = element_text(size = 20),
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)),
          panel.grid = element_line(colour = 'gray', linewidth = 0.5, linetype = 'dashed'))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'B_Cells_Function_Genes_DotPlot.pdf'), device = 'pdf', width = 9, height = 12, bg = 'transparent')

In [ ]:
#add the lung_condition_Age_type_Cell_type idents (OnlyDiseased)
samples_B_Cell_integrated <- FetchData(samples_B_Cell_integrated, vars = c('lung_condition', 'Age_type', 'B_subcluster_res_0.2')) %>%
    unite(col = 'lung_condition_Age_type_Cell_type', c('lung_condition', 'Age_type', 'B_subcluster_res_0.2')) %>%
    AddMetaData(object = samples_B_Cell_integrated)

# plot
Idents(samples_B_Cell_integrated) <- samples_B_Cell_integrated$`lung_condition_Age_type_Cell_type`
options(repr.plot.width = 10, repr.plot.height = 10)
DotPlot(samples_B_Cell_integrated, features = B_function_genes, group.by = 'lung_condition_Age_type_Cell_type', dot.scale = 10,
        idents = c('Tumor_Young_Naive_B', 'Tumor_Young_Memory_B', 'Tumor_Old_Naive_B', 'Tumor_Old_Memory_B')) +
    scale_y_discrete(limits = c('Tumor_Young_Naive_B', 'Tumor_Young_Memory_B', 'Tumor_Old_Naive_B', 'Tumor_Old_Memory_B'),
                     labels = c('Diseased_Young_Naive_B', 'Diseased_Young_Memory_B', 'Diseased_Old_Naive_B', 'Diseased_Old_Memory_B')) +
#    scale_color_gradient(low = '#E2E1EF', high = '#C23339') +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', oob = scales::squish) +
    labs(x = NULL, y = NULL) +
    coord_flip() +
    theme_bw(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'right',
          legend.title = element_text(size = 20),
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)),
          panel.grid = element_line(colour = 'gray', linewidth = 0.5, linetype = 'dashed'))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'B_Cells_Function_Genes_OnlyDiseased_DotPlot.pdf'), device = 'pdf', width = 9, height = 12, bg = 'transparent')

In [ ]:
DEGs_B_Cell_Naive_B_young_old_tumor_wilcox %>% 
    filter(p_val_adj < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% B_function_genes)

In [ ]:
DEGs_B_Cell_Memory_B_young_old_tumor_wilcox %>% 
    filter(p_val_adj < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% B_function_genes)

## Samples_APC_analysis

### Subset

In [ ]:
Idents(samples_imm_integrated) <- samples_imm_integrated$`ImmMaincluster_res0.9`
levels(samples_imm_integrated)

In [ ]:
# subset APC_Cell
Idents(samples_imm_integrated) <- samples_imm_integrated$`ImmMaincluster_res0.9`
APC_cell <- c('DC', 'Macro', 'Mono', 'B_Cell')
samples_APC_integrated <- subset(samples_imm_integrated, idents = APC_cell)
samples_APC_integrated

### Clustring and annotation analysis

In [ ]:
# UMAP
samples_APC_integrated <- RunUMAP(samples_APC_integrated, dims = 1:50, reduction = "harmony", verbose = F, seed.use = 13)


In [ ]:
# FindNeighbors
samples_APC_integrated <- FindNeighbors(samples_APC_integrated, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
options(repr.plot.width = 8, repr.plot.height = 8)
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  samples_APC_integrated <- FindClusters(samples_APC_integrated, resolution = i, verbose = F)
  print(DimPlot(samples_APC_integrated, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
options(repr.plot.height =24, repr.plot.width = 24)

imm_markers <- c('CD79A', 'MS4A1', 'JCHAIN',
                 'CD1C', 'CLEC9A', 'FSCN1', 'LILRA4',
                 'C1QC', 'CD68',
                 'FCN1', 'CD14', 'FCGR3A', 'S100A8',
                 'CSF3R', 'FCGR3B',
                 'FABP4', 'PPARG', 'MIF', 'SPP1', 'LGMN')

FeaturePlot(object = samples_APC_integrated, features = imm_markers, order = T)

In [ ]:
# get the multiple_subcluster for c1_res0.4
Idents(samples_APC_integrated) <- samples_APC_integrated$`RNA_snn_res.0.4`
samples_APC_integrated <- FindSubCluster(object = samples_APC_integrated, cluster = 1, subcluster.name = 'RNA_snn_res.0.4_subclus', graph.name = 'RNA_snn', resolution = 0.2)

In [ ]:
# get the multiple_subcluster for c6_res0.4
Idents(samples_APC_integrated) <- samples_APC_integrated$`RNA_snn_res.0.4_subclus`
samples_APC_integrated <- FindSubCluster(object = samples_APC_integrated, cluster = 6, subcluster.name = 'RNA_snn_res.0.4_subclus', graph.name = 'RNA_snn', resolution = 0.1)

In [ ]:
# get the multiple_subcluster for c10_res0.4
Idents(samples_APC_integrated) <- samples_APC_integrated$`RNA_snn_res.0.4_subclus`
samples_APC_integrated <- FindSubCluster(object = samples_APC_integrated, cluster = 10, subcluster.name = 'RNA_snn_res.0.4_subclus', graph.name = 'RNA_snn', resolution = 0.2)

In [ ]:
options(repr.plot.height =12, repr.plot.width = 12)
DimPlot(samples_APC_integrated, group.by = 'RNA_snn_res.0.4_subclus', label = T)

In [ ]:
#find APCMarkers_res0.4_presto with wilcox test in presto
APCMarkers_res0.4_presto <- presto::wilcoxauc(X = samples_APC_integrated, group_by = 'RNA_snn_res.0.4_subclus')

In [ ]:
APCMarkers_res0.4_presto %>% filter(group == '12') %>% arrange(-logFC) %>% head(60)

In [ ]:
##APC subclass annotation_res0.4
options(repr.plot.width = 14, repr.plot.height = 12)
Idents(samples_APC_integrated) <- samples_APC_integrated$RNA_snn_res.0.4_subclus

APC_subcluster_anno <- c('0' = 'FABP4_Macro', '1_0' = 'Memory_B', '1_1' = 'Naive_B', '1_2' = 'Memory_B', '1_3' = 'Memory_B', '2' = 'FABP4_Macro',
                         '3' = 'CD14_Mono', '4' = 'cDC2', '5' = 'CD16_Mono', '6_0' = 'LGMN_Macro', '6_1' = 'SPP1_Macro',
                         '7' = 'MIF_Macro', '8' = 'PPARG_Mono', '9' = 'PPARG_Macro',
                         '10_0' = 'cDC1', '10_1' = 'LAMP3_DC', '11' = 'pDC', '12' = 'SPP1_Macro', '13' = 'Contamination')



samples_APC_integrated <- RenameIdents(samples_APC_integrated, APC_subcluster_anno)
samples_APC_integrated$APC_subcluster_res_0.4 <- Idents(samples_APC_integrated)

DimPlot(samples_APC_integrated, label = T, pt.size = 1, label.size = 7, repel = T) +
theme(plot.title = element_text(size = 30),
      legend.text = element_text(size = 20),
      legend.key.size = unit(0.5, "inches")) +
guides(colour = guide_legend(override.aes = list(size = 5)))

ggsave(paste0(figures, '/', "APC_subcluster_annotation_res_0.4.png"), width = 18, height = 18)

In [ ]:
#filter Contamination cells
samples_APC_integrated <- subset(samples_APC_integrated, subset = APC_subcluster_res_0.4 != "Contamination")

In [ ]:
saveRDS(samples_APC_integrated, file = paste0(obj, "/", "samples_APC_integrated.rds"))

In [ ]:
#MS4A1:CD20; TNFRSF7:CD27; ADP-Ribosyl Cyclase 1：CD38； 
APC_markers <- c('XCR1', 'CLEC9A', 'FLT3', 'IDO1', 'CD1C', 'FCER1A', 'HLA-DPA1', 'LAMP3', 'FSCN1', 'CCR7', 'LILRA4', 'GZMB', 'IL3RA',
                 'FCN1', 'CD14', 'S100A8', 'A100A9','FCGR3A', 'LST1', 'LILRB2', 'PPARG',
                 'FABP4',  'MARCO', 'MRC1', 'MSR1', 'LGMN', 'C1QC', 'C1QA', 'SLCO2B1', 'SIGLEC1', 'APOC1', 'APOE', 'SPP1', 'TREM2', 'NR1H3', 'MIF', 'NME2', 'ISG15', 'IL1B', 'NLRP3',
                 'MS4A1', 'FCER2', 'TCL1A', 'CD27', 'TNFRSF13B')

In [ ]:
#APC_markers
options(repr.plot.width = 14, repr.plot.height = 10)

apc_subclusters <- c('cDC1', 'cDC2', 'LAMP3_DC', 'pDC',
                     'CD14_Mono', 'CD16_Mono', 'PPARG_Mono',
                     'FABP4_Macro', 'PPARG_Macro', 'LGMN_Macro', 'SPP1_Macro', 'MIF_Macro',
                     'Naive_B', 'Memory_B')

DotPlot(samples_APC_integrated, features = APC_markers, group.by = 'APC_subcluster_res_0.4', scale = T, col.max = 3) + 
    scale_y_discrete(limits=apc_subclusters) +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', guide = guide_colorbar(order = 1), limits = c(-2,2), oob = scales::squish) +
    scale_size_area(max_size = 10, guide = guide_legend(order = 2)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.border = element_rect(linewidth = 1, fill = NA, color = 'black'),
          legend.title = element_text(size = 20),
          legend.position = 'top',
          axis.text = element_text(colour = 'black'),
          axis.line = element_blank(),
          axis.text.x = element_text(angle = 90),
          axis.title.x = element_blank(),
          axis.title.y = element_blank())

#C23339
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_Cells_markers_Dotplot.pdf'), device = 'pdf', width = 17, height = 8, bg = 'transparent')

In [ ]:
apc_subclusters <- c('FABP4_Macro', 'PPARG_Macro', 'LGMN_Macro', 'SPP1_Macro', 'MIF_Macro',
                     'CD14_Mono', 'CD16_Mono', 'PPARG_Mono',
                     'cDC1', 'cDC2', 'LAMP3_DC', 'pDC',
                     'Naive_B', 'Memory_B')

DimPlot(samples_APC_integrated, group.by = 'APC_subcluster_res_0.4', label = F, pt.size = 0.5, raster=FALSE, shuffle=T) +
    scale_color_manual(values = imm_subcelltype_colors, limits = apc_subclusters) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'none')

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_cell_UMAP.pdf'), device = 'pdf', width = 10, height = 10, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_cell_UMAP.png'), device = 'png', width = 10, height = 10, dpi = 300, bg = 'transparent')

In [ ]:
samples_APC_integrated <- readRDS(paste0(obj, "/", "samples_APC_integrated.rds"))

### DEGs analysis

In [ ]:
##split the samples_APC_integrated into healthy and tumor
DefaultAssay(samples_APC_integrated) <- "RNA"
Idents(samples_APC_integrated) <- samples_APC_integrated$`lung_condition`

samples_APC_healthy_integrated <- subset(samples_APC_integrated, idents = 'Healthy')

samples_APC_tumor_integrated <- subset(samples_APC_integrated, idents = 'Tumor')

In [ ]:
unique(samples_APC_integrated$`APC_subcluster_res_0.4`)

In [ ]:
##find DEGs between APC_healthy_young and APC_healthy_old with wilcox_test
Idents(samples_APC_healthy_integrated) <- samples_APC_healthy_integrated$`Age_type`

DEGs_APC_young_old_healthy_wilcox <- FindMarkers(object = samples_APC_healthy_integrated, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                                 logfc.threshold = 0.25, min.pct = 0.1)


In [ ]:
##find DEGs between APC_tumor_young and APC_tumor_old with wilcox_test
Idents(samples_APC_tumor_integrated) <- samples_APC_tumor_integrated$`Age_type`

DEGs_APC_young_old_tumor_wilcox <- FindMarkers(object = samples_APC_tumor_integrated, ident.1 = "Young", ident.2 = 'Old', test.use = "wilcox",
                                                 logfc.threshold = 0.25, min.pct = 0.1)
 

In [ ]:
Antigen_process <- c('PSME1', 'PSME2', 'PSME3', 'TAP1', 'TAP2', 'TAPBP',
                     'B2M', 'CALR', 'CANX', 'CIITA', 'CREB1', 'CTSB', 'CTSL', 'CTSS',
                     'LGMN', 'NFYA', 'NFYB', 'NFYC', 'PDIA3', 'RFX5', 'RFXANK', 'RFXAP')


Antigen_presentation <- c('HLA-A', 'HLA-B', 'HLA-C', 'HLA-E', 'HLA-F', 'HLA-G',
                          'HLA-DRA', 'HLA-DRB5', 'HLA-DRB1', 'HLA-DQA1', 'HLA-DQB1',
                          'HLA-DQA2', 'HLA-DQB2', 'HLA-DOB', 'HLA-DMB', 'HLA-DMA',
                          'HLA-DOA', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DPB2', 'HLA-DRB6')

In [ ]:
DEGs_APC_young_old_tumor_wilcox %>% 
    filter(p_val_adj < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% Antigen_process);

DEGs_APC_young_old_tumor_wilcox %>% 
    filter(p_val_adj < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% Antigen_presentation)

In [ ]:
DEGs_APC_young_old_healthy_wilcox %>% 
    filter(p_val_adj < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% Antigen_process);

DEGs_APC_young_old_healthy_wilcox %>% 
    filter(p_val_adj < 0.05) %>% 
    arrange(avg_log2FC) %>%
    mutate(gene = rownames(.)) %>%
    filter(gene %in% Antigen_presentation)

In [ ]:
# Featureplot('HLA-DRB5', 'PSME2')
##add the lung_condition_Age_type idents
samples_APC_integrated <- FetchData(samples_APC_integrated, vars = c('lung_condition', 'Age_type')) %>%
    unite(col = 'lung_condition_Age_type', c('lung_condition', 'Age_type')) %>%
    mutate(lung_condition_Age_type = factor(lung_condition_Age_type,
                                            levels = c('Healthy_Young', 'Healthy_Old',
                                                       'Tumor_Young', 'Tumor_Old'))) %>%
    AddMetaData(object = samples_APC_integrated)

options(repr.plot.width = 20, repr.plot.height = 10)
FeaturePlot(samples_APC_integrated, features = c('HLA-DRB5', 'PSME2'), split.by = 'lung_condition_Age_type') &
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', limits = c(0,4), oob = scales::squish) &
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'right')

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_cell_Ag_Presentation_FeaturePlot.pdf'), device = 'pdf', width = 20, height = 10, bg = 'transparent')


In [ ]:
# Featureplot('HLA-DRB5', 'PSME2')
##add the lung_condition_Age_type idents
samples_APC_integrated <- FetchData(samples_APC_integrated, vars = c('lung_condition', 'Age_type')) %>%
    unite(col = 'lung_condition_Age_type', c('lung_condition', 'Age_type')) %>%
    mutate(lung_condition_Age_type = factor(lung_condition_Age_type,
                                            levels = c('Healthy_Young', 'Healthy_Old',
                                                       'Tumor_Young', 'Tumor_Old'))) %>%
    AddMetaData(object = samples_APC_integrated)

options(repr.plot.width = 20, repr.plot.height = 10)
FeaturePlot(samples_APC_integrated, features = c('HLA-DRB5', 'PSME2'), split.by = 'lung_condition_Age_type') &
    scale_color_gradient2(low = '#001F7F',mid = '#E2E1EF', high = '#C23339', midpoint = 1, limits = c(0,4), oob = scales::squish) &
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'right')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_cell_Ag_Presentation1_FeaturePlot.pdf'), device = 'pdf', width = 20, height = 10, bg = 'transparent')


In [ ]:
samples_APC_integrated <- FetchData(samples_APC_integrated, vars = c('lung_condition', 'Age_type')) %>%
    unite(col = 'lung_condition_Age_type', c('lung_condition', 'Age_type')) %>%
    mutate(lung_condition_Age_type = factor(lung_condition_Age_type,
                                            levels = c('Healthy_Young', 'Healthy_Old',
                                                       'Tumor_Young', 'Tumor_Old'))) %>%
    AddMetaData(object = samples_APC_integrated)

options(repr.plot.width = 20, repr.plot.height = 10)
FeaturePlot(samples_APC_integrated, features = c('HLA-DQA2', 'HLA-DQB1'), split.by = 'lung_condition_Age_type') &
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', limits = c(0,4), oob = scales::squish) &
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.position = 'right')

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_cell_Ag_Presentation2_FeaturePlot.pdf'), device = 'pdf', width = 16, height = 10, bg = 'transparent')


### Signature score analysis

In [ ]:
# Antigen_process_and_presentation_from_KEGG
Antigen_process <- c('PSME1', 'PSME2', 'PSME3', 'TAP1', 'TAP2', 'TAPBP',
                     'B2M', 'CALR', 'CANX', 'CIITA', 'CREB1', 'CTSB', 'CTSL', 'CTSS',
                     'LGMN', 'NFYA', 'NFYB', 'NFYC', 'PDIA3', 'RFX5', 'RFXANK', 'RFXAP')


Antigen_presentation <- c('HLA-A', 'HLA-B', 'HLA-C', 'HLA-E', 'HLA-F', 'HLA-G',
                          'HLA-DRA', 'HLA-DRB5', 'HLA-DRB1', 'HLA-DQA1', 'HLA-DQB1',
                          'HLA-DQA2', 'HLA-DQB2', 'HLA-DOB', 'HLA-DMB', 'HLA-DMA',
                          'HLA-DOA', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DPB2', 'HLA-DRB6')


In [ ]:
#Antigen_presentation_Score; Antigen_processing_Score; respective
samples_APC_integrated <- AddModuleScore(object = samples_APC_integrated,
                                         features = list(Antigen_presentation, Antigen_process),
                                         name = c('Antigen_presentation_Score', 'Antigen_process_Score'))



In [ ]:
Antigen_presentation

In [ ]:
Antigen_process

#### Antigen_presentation_Score plot

In [ ]:
samples_APC_integrated[[]] %>% colnames()

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10)
FetchData(samples_APC_integrated, vars = c('APC_subcluster_res_0.4', 'Age_type', 'sample', 'lung_condition', 'Antigen_presentation_Score1')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(APC_subcluster_res_0.4 %in% c('FABP4_Macro', 'PPARG_Macro', 'LGMN_Macro', 'SPP1_Macro', 'MIF_Macro')) %>%
    mutate(APC_subcluster_res_0.4 = factor(APC_subcluster_res_0.4,
                                           levels = c('FABP4_Macro', 'PPARG_Macro', 'LGMN_Macro', 'SPP1_Macro', 'MIF_Macro'))) %>%
    ggplot(mapping = aes(x = lung_condition, y = Antigen_presentation_Score1, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    facet_wrap(~APC_subcluster_res_0.4, nrow = 1) +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif', method = 'wilcox.test', size = 8) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 1)) +
    scale_x_discrete(labels = c('Healthy' = 'Healthy', 'Tumor' = 'Diseased')) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Antigen Presentation Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_Macro_AntigenPresentScore_Boxplot.pdf'), device = 'pdf', width = 18, height = 7, bg = 'transparent')

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 14)
apc_subclusters_selected <- c('CD14_Mono', 'CD16_Mono', 'PPARG_Mono', 'cDC1', 'cDC2', 'LAMP3_DC', 'pDC', 'Naive_B', 'Memory_B')

FetchData(samples_APC_integrated, vars = c('APC_subcluster_res_0.4', 'Age_type', 'sample', 'lung_condition', 'Antigen_presentation_Score1')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(APC_subcluster_res_0.4 %in% apc_subclusters_selected) %>%
    mutate(APC_subcluster_res_0.4 = factor(APC_subcluster_res_0.4, levels = apc_subclusters_selected)) %>%
    ggplot(mapping = aes(x = lung_condition, y = Antigen_presentation_Score1, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=F, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    facet_wrap(~APC_subcluster_res_0.4, nrow = 3, scales = 'free_y') +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif', method = 'wilcox.test', size = 8) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 1)) +
    scale_x_discrete(labels = c('Healthy' = 'Healthy', 'Tumor' = 'Diseased')) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Antigen Presentation Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_9clusters_AntigenPresentScore_Boxplot.pdf'), device = 'pdf', width = 12, height = 10, bg = 'transparent')

#### Antigen_process_Score plot

In [ ]:
samples_APC_integrated[[]] %>% colnames()

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10)
FetchData(samples_APC_integrated, vars = c('APC_subcluster_res_0.4', 'Age_type', 'sample', 'lung_condition', 'Antigen_process_Score2')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(APC_subcluster_res_0.4 %in% c('FABP4_Macro', 'PPARG_Macro', 'LGMN_Macro', 'SPP1_Macro', 'MIF_Macro')) %>%
    mutate(APC_subcluster_res_0.4 = factor(APC_subcluster_res_0.4,
                                           levels = c('FABP4_Macro', 'PPARG_Macro', 'LGMN_Macro', 'SPP1_Macro', 'MIF_Macro'))) %>%
    ggplot(mapping = aes(x = lung_condition, y = Antigen_process_Score2, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    facet_wrap(~APC_subcluster_res_0.4, nrow = 1, ) +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif', method = 'wilcox.test', size = 8) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 1)) +
    scale_x_discrete(labels = c('Healthy' = 'Healthy', 'Tumor' = 'Diseased')) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Antigen Process Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_Macro_AntigenProcessScore_Boxplot.pdf'), device = 'pdf', width = 18, height = 7, bg = 'transparent')

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 14)
apc_subclusters_selected <- c('CD14_Mono', 'CD16_Mono', 'PPARG_Mono', 'cDC1', 'cDC2', 'LAMP3_DC', 'pDC', 'Naive_B', 'Memory_B')

FetchData(samples_APC_integrated, vars = c('APC_subcluster_res_0.4', 'Age_type', 'sample', 'lung_condition', 'Antigen_process_Score2')) %>%
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    filter(APC_subcluster_res_0.4 %in% apc_subclusters_selected) %>%
    mutate(APC_subcluster_res_0.4 = factor(APC_subcluster_res_0.4, levels = apc_subclusters_selected)) %>%
    ggplot(mapping = aes(x = lung_condition, y = Antigen_process_Score2, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=F, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    facet_wrap(~APC_subcluster_res_0.4, nrow = 3, scales = 'free_y') +
    stat_compare_means(mapping = aes(group = Age_type), label = 'p.signif', method = 'wilcox.test', size = 8) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = element_blank(), nrow = 1)) +
    scale_x_discrete(labels = c('Healthy' = 'Healthy', 'Tumor' = 'Diseased')) +
    geom_hline(yintercept = 0, linetype='dashed') +
    xlab(label = NULL) +
    ylab(label = 'Antigen Process Score') +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.text.x = element_text(angle = 0),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'APC_9clusters_AntigenProcessScore_Boxplot.pdf'), device = 'pdf', width = 13, height = 10, bg = 'transparent')

### Flow cytometry analysis

In [ ]:
#enter the Flow cytometry results
HLADR_MFI_DC_df <- as.data.frame(list('MFI' = c(17840, 23685, 31624, 58034, 45035, 35687),
                                      'Age_type' = factor(c(rep('Young',3), rep('Old',3)), levels = c('Young', 'Old')),
                                      'lung_condition' = 'Diseased',
                                      'Celltype' = 'DC'))

HLADR_MFI_Macro_df <- as.data.frame(list('MFI' = c(29870, 27329, 30180, 42540, 36870, 33586),
                                      'Age_type' = factor(c(rep('Young',3), rep('Old',3)), levels = c('Young', 'Old')),
                                      'lung_condition' = 'Diseased',
                                      'Celltype' = 'Macrophage'))

In [ ]:
# F test for variance
var.test(MFI ~ Age_type, HLADR_MFI_Macro_df)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)
ggplot(HLADR_MFI_DC_df, mapping = aes(x = Age_type, y = MFI, fill = Age_type)) +
    stat_boxplot(geom = 'errorbar', width = 0.3, linewidth=1, position = position_dodge(0.9)) +
    geom_boxplot(notch=F, width=0.5, linewidth=1, position = position_dodge(0.85)) +
    geom_jitter(size = 4, shape = 21, position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75)) +
    geom_signif(test = 't.test', test.args = list(var.equal = T), size = 1, 
                comparisons = list(c('Young', 'Old')), textsize = 5.88,
                map_signif_level = function(p) sprintf("p = %.4f", p)) +
    scale_fill_manual(values = age_group_color, guide = NULL) +
    xlab(label = NULL) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
          axis.text = element_text(colour = 'black'),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'FCM_HLADR_DC_MFI_Boxplot.pdf'), device = 'pdf', width = 8, height = 10, bg = 'transparent')

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)
ggplot(HLADR_MFI_Macro_df, mapping = aes(x = Age_type, y = MFI, fill = Age_type)) +
    stat_boxplot(geom = 'errorbar', width = 0.3, linewidth=1, position = position_dodge(0.9)) +
    geom_boxplot(notch=F, width=0.5, linewidth=1, position = position_dodge(0.85)) +
    geom_jitter(size = 4, shape = 21, position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75)) +
    geom_signif(test = 't.test', test.args = list(var.equal = T), size = 1, 
                comparisons = list(c('Young', 'Old')), textsize = 5.88,
                map_signif_level = function(p) sprintf("p = %.4f", p)) +
    scale_fill_manual(values = age_group_color, guide = NULL) +
    xlab(label = NULL) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(legend.position = 'none',
          axis.text = element_text(colour = 'black'),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'FCM_HLADR_Macro_MFI_Boxplot.pdf'), device = 'pdf', width = 8, height = 10, bg = 'transparent')

## Samples_Nonimm_analysis(Integration:Harmony)

### subset

In [ ]:
Idents(samples_all_integrated) <- samples_all_integrated$`main_cell_type_res_0.4`
levels(samples_all_integrated)

In [ ]:
# subset immune cells
Idents(samples_all_integrated) <- samples_all_integrated$`main_cell_type_res_0.4`
samples_Nonimm_integrated <- subset(samples_all_integrated, idents = c('Epi', 'Endo', 'Fibro'))
samples_Nonimm_integrated

### Harmony integration

In [ ]:
#normalization
samples_Nonimm_integrated <- samples_Nonimm_integrated %>%
    NormalizeData(verbose = FALSE)

In [ ]:
# find variable genes in each sample
fvf_collection <- split(row.names(samples_Nonimm_integrated@meta.data), samples_Nonimm_integrated@meta.data$sample) %>%
    lapply(function(cells_use) {
    samples_Nonimm_integrated[,cells_use] %>%
        FindVariableFeatures(selection.method = "vst", nfeatures = 2000) %>% 
        VariableFeatures()
    }) %>% unlist

fvf_genes <- table(fvf_collection) %>% as.data.frame() %>% slice_max(order_by = Freq, n = 3000)
fvf_genes <- fvf_genes[['fvf_collection']]

VariableFeatures(samples_Nonimm_integrated) <- fvf_genes

In [ ]:
samples_Nonimm_integrated

In [ ]:
#run scale_data

samples_Nonimm_integrated <- ScaleData(samples_Nonimm_integrated, verbose = FALSE)

In [ ]:
# run pca
samples_Nonimm_integrated <- RunPCA(object = samples_Nonimm_integrated, features = VariableFeatures(samples_Nonimm_integrated), npcs = 50, verbose = FALSE)

In [ ]:
# run harmony
samples_Nonimm_integrated <- RunHarmony(object = samples_Nonimm_integrated, group.by.vars = 'sample', plot_convergence = T, max.iter.harmony = 25)

### Clustring and annotation analysis

In [ ]:
# UMAP
samples_Nonimm_integrated <- RunUMAP(samples_Nonimm_integrated, dims = 1:50, reduction = "harmony", verbose = F )


In [ ]:
# FindNeighbors
samples_Nonimm_integrated <- FindNeighbors(samples_Nonimm_integrated, dims = 1:50, reduction = "harmony", verbose = F)

In [ ]:
# FindClusters
for (i in c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)) {
  samples_Nonimm_integrated <- FindClusters(samples_Nonimm_integrated, resolution = i, verbose = F)
  print(DimPlot(samples_Nonimm_integrated, reduction = "umap", label = T) + labs(title = paste0("resolution: ", i)))
}

In [ ]:
#save data
saveRDS(samples_Nonimm_integrated, paste0(obj, "/", "samples_Nonimm_integrated.rds"))

In [ ]:
options(repr.plot.height =24, repr.plot.width = 24)

NonImm_markers <- c('EPCAM', 'KRT7', 'KRT19', 'PECAM1', 'CCL21', 'PTPRC', 'LUM', 'ACTA2', 'PDGFRB')

FeaturePlot(object = samples_Nonimm_integrated, features = NonImm_markers, order = T)

In [ ]:
DefaultAssay(samples_Nonimm_integrated) <- "RNA"


##endo: "PECAM1", "VWF", "CDH5"
##pericyte: P2RY14
##Lymphatic: "CCL21", "PROX1"
##fibro: "LUM", "TCF21",'PDGFRA', 'DCN', 'DPT'
##smc: 'PDGFRB', "ACTA2", 'MYLK', 'MYH11', 'TAGLN'
##endo: An Integrated Gene Expression Landscape Profiling Approach to Identify Lung Tumor Endothelial Cell Heterogeneity and Angiogenic Candidates
##fibro: Single-cell analysis reveals prognostic fibroblast subpopulations linked to molecular and immunological subtypes of lung cancer


#NonImm_markers_expression_dotplot_res_1
NonImm_markers <- c("PECAM1", "VWF", "CDH5", "CCL21", "PROX1",
                    "LUM", 'DCN', 'DPT', "TCF21",'PDGFRA', 'PDGFRB', "ACTA2", 'MYLK', 'MYH11', 'TAGLN','UPK3B', 'WT1', 'P2RY14',
                    "EPCAM", "CDH1", "KRT7", "KRT19", "SFTPB", "SFTPC", "AGER", "FOXJ1", "SCGB1A1", "SCGB3A2",
                    "FN1", "TGFBI", "COL1A1", "MKI67",
                    "JCHAIN", "IGKC", "PTPRC", "TRAC")

options(repr.plot.height =15, repr.plot.width = 18)
DotPlot(samples_Nonimm_integrated, features = NonImm_markers, group.by = "RNA_snn_res.1") * theme(axis.text = element_text(size = 20, face = "bold")) +
  coord_flip()


In [ ]:
options(repr.plot.height =12, repr.plot.width = 15)
DimPlot(samples_Nonimm_integrated, group.by = 'RNA_snn_res.1', label = T, label.size = 5)

In [ ]:
#find NonImmMarkers_res1_presto with wilcox test in presto
NonImmMarkers_res1_presto <- presto::wilcoxauc(X = samples_Nonimm_integrated, group_by = 'RNA_snn_res.1')

In [ ]:
NonImmMarkers_res1_presto %>% filter(group == '2') %>% arrange(-logFC) %>% head(60)

In [ ]:
##Nonimm mainclass annotation_res0.5
Idents(samples_Nonimm_integrated) <- samples_Nonimm_integrated$RNA_snn_res.1

Nonimm_maincluster_anno <- c('0' = 'Epi', '1' = 'Epi', '2' = 'Epi','3' = 'Epi', '4' = 'Fibro', '5' = 'Epi', 
                             '6' = 'Endo', '7' = 'Epi', '8' = 'Epi', '9' = 'Epi', '10' = 'Endo', '11' = 'Epi',
                             '12' = 'Endo', '13' = 'Endo', '14' = 'Mural', '15' = 'Epi', '16' = 'Epi',
                             '17' = 'Epi', '18' = 'Epi', '19' = 'Contamination', '20' = 'Epi', '21' = 'Contamination',
                             '22' = 'Contamination', '23' = 'Epi', '24' = 'Endo', '25' = 'Epi', '26' = 'Contamination',
                             '27' = 'Epi', '28' = 'Epi', '29' ='Epi', '30' = 'Contamination', '31' = 'Contamination',
                             '32' = 'Epi', '33' = 'Epi', '34' = 'Epi', '35' = 'Epi', '36' = 'Epi', '37' = 'Epi')



samples_Nonimm_integrated <- RenameIdents(samples_Nonimm_integrated, Nonimm_maincluster_anno)
samples_Nonimm_integrated$Nonimm_maincluster_res1 <- Idents(samples_Nonimm_integrated)

DimPlot(samples_Nonimm_integrated, label = T, pt.size = 1, label.size = 7, repel = T) +
    theme(plot.title = element_text(size = 30),
          legend.text = element_text(size = 20),
          legend.key.size = unit(0.5, "inches")) +
    guides(colour = guide_legend(override.aes = list(size = 5)))

ggsave(paste0(figures, '/', "Nonimm_maincluster_annotation_res1.png"), width = 18, height = 18)

In [ ]:
#filter Contamination cells
samples_Nonimm_integrated <- subset(samples_Nonimm_integrated, subset = Nonimm_maincluster_res1 != "Contamination")

In [ ]:
saveRDS(samples_Nonimm_integrated, file = paste0(obj, "/", "samples_Nonimm_integrated.rds"))

In [ ]:
samples_Nonimm_integrated

### InferCNV

In [ ]:
FetchData(samples_Nonimm_integrated, vars = c('Nonimm_maincluster_res1', 'RNA_snn_res.1')) 

In [ ]:
# downsample the Nonimm for infercnv
annotation_NonImm <- FetchData(samples_Nonimm_integrated, vars = c('Nonimm_maincluster_res1', 'RNA_snn_res.1')) %>%
    transmute("idents_for_infercnv" = ifelse(Nonimm_maincluster_res1 == 'Epi',
                                      paste0(Nonimm_maincluster_res1, '_c', RNA_snn_res.1),
                                      paste0(Nonimm_maincluster_res1)))

samples_Nonimm_integrated <- AddMetaData(object = samples_Nonimm_integrated, metadata = annotation_NonImm)

#downsample
Idents(samples_Nonimm_integrated) <- samples_Nonimm_integrated$`idents_for_infercnv`
samples_Nonimm_integrated_ds <- subset(samples_Nonimm_integrated, downsample = 1000)

In [ ]:
samples_Nonimm_integrated_ds

In [ ]:
#get the raw count matrix from RNA assay

DefaultAssay(samples_Nonimm_integrated_ds) <- "RNA"
raw_count_matrix_NonImm <- GetAssayData(samples_Nonimm_integrated_ds, slot="counts", assay = "RNA")

In [ ]:
## prepare the Gene ordering file 
## check the genome reference for alignment: UMIs were quantified using Cellranger 3.0.2 (10x Genomics) with reference transcriptome GRCh38. 
## download the Gene ordering file from  TrinityCTAT : hg38_gencode_v27.txt

In [ ]:
##prepare the cell annotation file
annotation_file_NonImm <- FetchData(samples_Nonimm_integrated_ds, vars = c('idents_for_infercnv')) %>%
    mutate("cells" = rownames(.), .before = idents_for_infercnv)

write_tsv(annotation_file_NonImm,paste0(obj, '/Infercnv/', "annotation_file_NonImm.tsv"), col_names = F)

In [ ]:
##create infercnv objects
infercnv_NonImm_object <- infercnv::CreateInfercnvObject(raw_counts_matrix = raw_count_matrix_NonImm,
                                               gene_order_file = paste0(obj, '/Infercnv/', "hg38_gencode_v27.txt"),   ### load the file using function rather than manually load
                                               annotations_file = paste0(obj, '/Infercnv/', "annotation_file_NonImm.tsv"),
                                               ref_group_names = c('Endo', 'Mural', 'Fibro'))

In [ ]:
# run infercnv
infercnv_NonImm_object <- infercnv::run(infercnv_NonImm_object,  
                                        cutoff = 0.05, 
                                        out_dir = paste0(obj, '/', 'Infercnv'), 
                                        cluster_by_groups = T, HMM = F, HMM_type = "i6",
                                        denoise = T, sd_amplifier = 1.5, plot_steps = F,
                                        cluster_references = T, leiden_resolution = 0.001, num_threads = 16)            #The final arguments

In [ ]:
options(repr.plot.height =12, repr.plot.width = 15)
DimPlot(samples_Nonimm_integrated, group.by = 'RNA_snn_res.1', label = T, label.size = 5)

In [ ]:
options(repr.plot.height =12, repr.plot.width = 20)
DimPlot(samples_Nonimm_integrated, group.by = 'sample', label = F, label.size = 5)

In [ ]:
DefaultAssay(samples_Nonimm_integrated) <- "RNA"


##endo: "PECAM1", "VWF", "CDH5"
##pericyte: P2RY14
##Lymphatic: "CCL21", "PROX1"
##fibro: "LUM", "TCF21",'PDGFRA', 'DCN', 'DPT'
##smc: 'PDGFRB', "ACTA2", 'MYLK', 'MYH11', 'TAGLN'
##endo: An Integrated Gene Expression Landscape Profiling Approach to Identify Lung Tumor Endothelial Cell Heterogeneity and Angiogenic Candidates
##fibro: Single-cell analysis reveals prognostic fibroblast subpopulations linked to molecular and immunological subtypes of lung cancer


#NonImm_markers_expression_dotplot_res_1
NonImm_markers <- c("PECAM1", "VWF", "CDH5", "CCL21", "PROX1",
                    "LUM", 'DCN', 'DPT', "TCF21",'PDGFRA', 'PDGFRB', "ACTA2", 'MYLK', 'MYH11', 'TAGLN','UPK3B', 'WT1', 'P2RY14',
                    "EPCAM", "CDH1", "KRT7", "KRT19", "SFTPB", "SFTPC", "AGER", "FOXJ1", "SCGB1A1", "SCGB3A2",
                    "FN1", "TGFBI", "COL1A1", "MKI67",
                    "JCHAIN", "IGKC", "PTPRC", "TRAC")

options(repr.plot.height =15, repr.plot.width = 18)
DotPlot(samples_Nonimm_integrated, features = NonImm_markers, group.by = "RNA_snn_res.1") * theme(axis.text = element_text(size = 20, face = "bold")) +
  coord_flip()

In [ ]:
##Nonimm mainclass annotation_res1_wTumor
Idents(samples_Nonimm_integrated) <- samples_Nonimm_integrated$RNA_snn_res.1

Nonimm_maincluster_wTumor_anno <- c('0' = 'Epi', '1' = 'Epi', '2' = 'Epi','3' = 'Epi', '4' = 'Fibro', '5' = 'Epi', 
                             '6' = 'Endo', '7' = 'Tumor', '8' = 'Epi', '9' = 'Epi', '10' = 'Endo', '11' = 'Tumor',
                             '12' = 'Endo', '13' = 'Endo', '14' = 'Mural', '15' = 'Tumor', '16' = 'Epi',
                             '17' = 'Epi', '18' = 'Epi', '20' = 'Tumor', 
                             '23' = 'Tumor', '24' = 'Endo', '25' = 'Tumor',
                             '27' = 'Tumor', '28' = 'Tumor', '29' ='Epi',
                             '32' = 'Tumor', '33' = 'Tumor', '34' = 'Tumor', '35' = 'Tumor', '36' = 'Tumor', '37' = 'Tumor')




samples_Nonimm_integrated <- RenameIdents(samples_Nonimm_integrated, Nonimm_maincluster_wTumor_anno)
samples_Nonimm_integrated$Nonimm_maincluster_wTumor_res1 <- Idents(samples_Nonimm_integrated)

options(repr.plot.height =12, repr.plot.width = 15)
DimPlot(samples_Nonimm_integrated, label = T, pt.size = 1, label.size = 7, repel = T) +
    theme(plot.title = element_text(size = 30),
          legend.text = element_text(size = 20),
          legend.key.size = unit(0.5, "inches")) +
    guides(colour = guide_legend(override.aes = list(size = 5)))

ggsave(paste0(figures, '/', "Nonimm_maincluster_wTumor_annotation_res_1.png"), width = 18, height = 18)

In [ ]:
#save data
saveRDS(samples_Nonimm_integrated, paste0(obj, "/", "samples_Nonimm_integrated.rds"))

## Samples_all_annotation

### Merge annotated objects

In [ ]:
#load data
samples_Nonimm_integrated <- readRDS(paste0(obj, "/", "samples_Nonimm_integrated.rds"))
samples_imm_integrated <- readRDS(paste0(obj, "/", "samples_imm_integrated.rds"))
samples_T_Cell_integrated <- readRDS(paste0(obj, "/", "samples_T_Cell_integrated.rds"))
samples_APC_integrated <- readRDS(paste0(obj, "/", "samples_APC_integrated.rds"))

In [ ]:
samples_imm_integrated$`ImmMaincluster_res0.9` %>% unique()

In [ ]:
#subset other Imms
other_imms <- c('NK', 'Plasma', 'Mast', 'Neutrophil')
samples_NK_Plasma_Mast_Neu_integrated <- subset(samples_imm_integrated, subset = ImmMaincluster_res0.9 %in% other_imms)

In [ ]:
samples_NK_Plasma_Mast_Neu_integrated$`ImmMaincluster_res0.9` %>% unique()

In [ ]:
#concat all the annotated objects: main anno
samples_Nonimm_integrated$'main_anno' <- samples_Nonimm_integrated$`Nonimm_maincluster_wTumor_res1`
samples_T_Cell_integrated$'main_anno' <- samples_T_Cell_integrated$`ImmMaincluster_res0.9`
samples_APC_integrated$'main_anno' <- samples_APC_integrated$`ImmMaincluster_res0.9`
samples_NK_Plasma_Mast_Neu_integrated $'main_anno' <- samples_NK_Plasma_Mast_Neu_integrated$`ImmMaincluster_res0.9`

#concat
samples_all_annotation <- merge(x = samples_Nonimm_integrated, y = list(samples_T_Cell_integrated, samples_APC_integrated, samples_NK_Plasma_Mast_Neu_integrated))

In [ ]:
samples_all_annotation

### Harmony integration

In [ ]:
#normalization
samples_all_annotation <- samples_all_annotation %>%
    NormalizeData(verbose = FALSE)

In [ ]:
# find variable genes in each sample
fvf_collection <- split(row.names(samples_all_annotation@meta.data), samples_all_annotation@meta.data$sample) %>%
    lapply(function(cells_use) {
    samples_all_annotation[,cells_use] %>%
        FindVariableFeatures(selection.method = "vst", nfeatures = 2000) %>% 
        VariableFeatures()
    }) %>% unlist

fvf_genes <- table(fvf_collection) %>% as.data.frame() %>% slice_max(order_by = Freq, n = 3000)
fvf_genes <- fvf_genes[['fvf_collection']]

VariableFeatures(samples_all_annotation) <- fvf_genes

In [ ]:
samples_all_annotation

In [ ]:
#run scale_data pca
samples_all_annotation <- samples_all_annotation %>% 
    ScaleData(verbose = FALSE) %>% 
    RunPCA(features = VariableFeatures(samples_all_annotation), npcs = 50, verbose = FALSE)

In [ ]:
# run harmony
samples_all_annotation <- RunHarmony(object = samples_all_annotation, group.by.vars = 'sample', plot_convergence = T)

In [ ]:
# UMAP
samples_all_annotation <- RunUMAP(samples_all_annotation, dims = 1:50, reduction = "harmony", verbose = F)


In [ ]:
options(repr.plot.height =10, repr.plot.width = 10)
DimPlot(samples_all_annotation, group.by = 'main_anno', label = F, pt.size = 0.5, raster=FALSE, shuffle=T) +
    scale_color_manual(values = main_celltype_colors) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank(),
          legend.key.height = unit(x = 0.5, units = 'in'),
          legend.key.size = unit(x = 0.5, units = 'in'))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'All_cell_UMAP.pdf'), device = 'pdf', width = 10, height = 10, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'All_cell_UMAP.png'), device = 'png', width = 10, height = 10, dpi = 300, bg = 'transparent')

In [ ]:
options(repr.plot.height =10, repr.plot.width = 10)
DimPlot(samples_all_annotation, group.by = 'main_cell_type_res_0.4', label = F, pt.size = 0.5, raster=FALSE, shuffle=T) +
    scale_color_manual(values = main_celltype_colors) +
    labs(title = NULL) +
    theme_classic(base_size = 12) +
    transparent_bg +
    theme(legend.position = 'none',
          axis.text = element_blank(),
          axis.line = element_blank(),
          axis.ticks = element_blank(),
          axis.title = element_blank())

#ggsave(filename = paste0(figures, '/Figures_raw', '/', 'All_cell_UMAP_maincluster.pdf'), device = 'pdf', width = 10, height = 10, bg = 'transparent')
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'All_cell_UMAP_maincluster.png'), device = 'png', width = 10, height = 10, dpi = 300, bg = 'transparent')

In [ ]:
all_clusters_markers <- c('EPCAM', 'KRT7', 'SFTPB', 'SFTPC', 'SCGB3A2', 'SCGB1A1', 'AGER','FOXJ1', 'CAPS', 'UPK3B',
                          'PECAM1', 'VWF', 'LUM', 'DCN', 'ACTA2', 'TAGLN',
                          'PTPRC', 'NKG7', 'TRAC', 'CD3E', 'CD68', 'C1QA', 'CD14', 'FCN1', 'CSF3R', 'FCGR3B', 'KIT', 'CPA3',
                          'CD79A', 'MS4A1', 'JCHAIN', 'IGKC')

In [ ]:
options(repr.plot.height =6, repr.plot.width = 13)
DotPlot(samples_all_annotation, features = all_clusters_markers, group.by = 'main_cell_type_res_0.4', scale = T) + 
    scale_y_discrete(limits=c('Epi', 'Endo', 'Fibro', 'Imm')) +
    scale_color_gradient(low = '#E2E1EF', high = '#C23339', guide = guide_colorbar(order = 1), limits = c(-2,2), oob = scales::squish) +
    scale_size_area(max_size = 12, guide = guide_legend(order = 2)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(panel.border = element_rect(linewidth = 1, fill = NA, color = 'black'),
          legend.title = element_text(size = 20),
          legend.position = 'top',
          axis.text = element_text(colour = 'black'),
          axis.line = element_blank(),
          axis.text.x = element_text(angle = 90),
          axis.title.x = element_blank(),
          axis.title.y = element_blank())

#C23339
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'All_Cell_maincluster_markers_Dotplot.pdf'), device = 'pdf', width = 13, height = 8, bg = 'transparent')

### Adding the T_Cell and APC_Cell Sub_Anno

In [ ]:
#T_subcluster_res_0.5
#APC_subcluster_res_0.4

In [ ]:
#merge the metadata
all_meta <- FetchData(samples_all_annotation, vars = c('main_anno')) %>%
    mutate(cell = rownames(.), sub_anno = main_anno) %>% 
    select(c('cell', 'sub_anno'))

T_meta <- FetchData(samples_T_Cell_integrated, vars = c('T_subcluster_res_0.5')) %>%
    mutate(cell = rownames(.), T_subcluster_res_0.5 = as.character(T_subcluster_res_0.5))

APC_meta <- FetchData(samples_APC_integrated, vars = c('APC_subcluster_res_0.4')) %>%
    mutate(cell = rownames(.), APC_subcluster_res_0.4 = as.character(APC_subcluster_res_0.4))

#left_join
all_meta <- left_join(x = all_meta, y = T_meta, by = 'cell') %>%
    left_join(y = APC_meta, by = 'cell')


#fill the sub_anno
all_meta <- all_meta %>% 
    mutate(sub_anno = ifelse(!is.na(T_subcluster_res_0.5),
                                    T_subcluster_res_0.5,
                                    sub_anno)) %>%
    mutate(sub_anno = ifelse(!is.na(APC_subcluster_res_0.4),
                                    APC_subcluster_res_0.4,
                                    sub_anno)) %>%
    select(c('cell', 'sub_anno')) %>%
    column_to_rownames(var = 'cell')

#add metadata
samples_all_annotation <- AddMetaData(samples_all_annotation, metadata = all_meta)

In [ ]:
samples_all_annotation[[]] %>% colnames()

In [ ]:
table(samples_all_annotation$sub_anno, samples_all_annotation$Age_type, samples_all_annotation$lung_condition)

In [ ]:
# convert the samples_all_annotation_diseased(tumor) into anndata
Idents(samples_all_annotation) <- samples_all_annotation$`lung_condition`
samples_all_annotation_diseased <- subset(samples_all_annotation, idents = 'Tumor')

sceasy::convertFormat(samples_all_annotation_diseased, from="seurat", to="anndata",
                      main_layer = 'counts', drop_single_values = FALSE,
                       outFile=paste0(obj, '/', 'samples_all_annotation_diseased.h5ad'))

In [ ]:
# convert the samples_all_annotation_healthy(healthy) into anndata
Idents(samples_all_annotation) <- samples_all_annotation$`lung_condition`
samples_all_annotation_healthy <- subset(samples_all_annotation, idents = 'Healthy')

sceasy::convertFormat(samples_all_annotation_healthy, from="seurat", to="anndata",
                      main_layer = 'counts', drop_single_values = FALSE,
                       outFile=paste0(obj, '/', 'samples_all_annotation_healthy.h5ad'))

In [ ]:
112485 + 110709 + 22162

In [ ]:
#save
saveRDS(samples_all_annotation, file = paste0(obj, '/', 'samples_all_annotation.rds'))

In [ ]:
#read
samples_all_annotation <- readRDS(paste0(obj, '/', 'samples_all_annotation.rds'))

## Samples metadata plot

### Diseased

In [ ]:
Diseased_clinical_data <- read.csv(paste0(obj, '/', 'Diseased_clinical_data2.csv'))

In [ ]:
#TilePlot
Diseased_clinical_tileplot <- Diseased_clinical_data  %>%
    select(id, gender, condition, smoking, stage, age_group) %>%
    mutate(id = str_replace(id, pattern = 'T', replacement = 'D')) %>%
    mutate(id = factor(id, levels = rev(id))) %>%
    gather(Category,Value,gender:condition:smoking:stage:age_group)

In [ ]:
options(repr.plot.height =12, repr.plot.width = 10)

#TilePlot
color_palette <- brewer.pal(12, "Set3")

# Define color values for each unique 'Plot_Value'
color_values <- c(
  'Young' = color_palette[1],
  'Old' = color_palette[2],
  'No' = color_palette[3],
  'NA.' = color_palette[4], 
  'Yes' = color_palette[5],
  'Diseased' = color_palette[6],
  'Healthy' = color_palette[7],
  'Male' = color_palette[9],
  'Female' = color_palette[8],
  'IAC' = color_palette[10],
  'MIA' = color_palette[11],
  'AIS' = color_palette[12]
)



# 继续你原有的ggplot代码
tile_plot <- ggplot(Diseased_clinical_tileplot, aes(x = Category, y = id, fill = Value)) + 
  geom_tile(width = 0.8, height = 0.8) + # 调整方块大小
  scale_fill_manual(values = color_values) +
  theme_minimal() +
  labs(x = "ID", y = "Category", fill = "Legend Title") +
  transparent_bg +
  theme(
    axis.text.x = element_text(hjust = 1, size = 16, colour = 'black', angle = 25),  # 调整x轴标签的大小和角度
    axis.text.y = element_text(size = 16, colour = 'black'),  # 调整y轴标签的大小
#    legend.position = "none",
    strip.text = element_text(size = 8),  # 调整分类标签的大小
    strip.background = element_blank(),  # 移除分类标签背景
    panel.border = element_blank(), # 移除面板边框
    panel.grid.major = element_blank(), # 移除主要网格线
    panel.grid.minor = element_blank(), # 移除次要网格线
    panel.spacing.y = unit(0.01, "lines"),  # 减小不同分类变量之间的间距
    plot.margin = unit(c(1, 1, 1, 1), "lines") # 可以尝试调整边缘空间
  )+
  coord_fixed(ratio = 1)+  # 固定x轴和y轴的比例 
  scale_x_discrete(expand = c(0, 0))  # 减少分类变量之间的间距

tile_plot
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Diseased_Samples_Metadata_TilePlot.pdf'), device = 'pdf', width = 5, height = 10, bg = 'transparent')

In [ ]:
options(repr.plot.height =6, repr.plot.width = 12)
# 计算比例
stage_counts <- table(Diseased_clinical_data$age_group, Diseased_clinical_data$stage)
stage_proportions <- prop.table(stage_counts, 1)

# 转换为长格式
stage_proportions_long <- as.data.frame(as.table(stage_proportions))
names(stage_proportions_long) <- c("Group", "Stage", "Proportion")

# 设置颜色
color_palette <- brewer.pal(12, "Set3")
stage_colors <- c('IAC' = color_palette[10],
                  'MIA' = color_palette[11],
                  'AIS' = color_palette[12])

# 绘制横置的堆叠条形图
p <-ggplot(stage_proportions_long, aes(x = Group, y = Proportion, fill = Stage, label = scales::percent(Proportion))) +
  geom_bar(stat = "identity", position = "fill", width = 0.5) +
  geom_text(position = position_fill(vjust = 0.5), color = "white", fontface = "bold", size = 6.5) +
  scale_fill_manual(values = stage_colors) +
  labs(x = "Group", y = "Proportion", fill = "Stage") +
  theme_minimal() +
  coord_flip()
Stagebar <-p + theme_void()

Stagebar
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Diseased_Samples_Metadata_Stage_BarPlot.pdf'), device = 'pdf', width = 10, height = 5, bg = 'transparent')

In [ ]:
#吸烟比例

# 计算比例
smoking_counts <- table(Diseased_clinical_data$age_group, Diseased_clinical_data$smoking)
smoking_proportions <- prop.table(smoking_counts, 1)

# 转换为长格式
smoking_proportions_long <- as.data.frame(as.table(smoking_proportions))
names(smoking_proportions_long) <- c("Group", "Smoking", "Proportion")

# 设置颜色
color_palette <- brewer.pal(12, "Set3")
smoking_colors <- c('No' = color_palette[3],
                    'NA.' = color_palette[4],
                    'Yes' = color_palette[5])

# 绘制横置的堆叠条形图
p2 <-ggplot(smoking_proportions_long, aes(x = Group, y = Proportion, fill = Smoking, label = scales::percent(Proportion))) +
  geom_bar(stat = "identity", position = "fill", width = 0.5) +
  geom_text(position = position_fill(vjust = 0.5), color = "white", fontface = "bold", size = 4.5) +
  scale_fill_manual(values = smoking_colors) +
  labs(x = "Group", y = "Proportion", fill = "Smoking") +
  theme_minimal() +
  coord_flip()
Smokingbar <-p2 + theme_void()
Smokingbar

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Diseased_Samples_Metadata_Smoking_BarPlot.pdf'), device = 'pdf', width = 10, height = 5, bg = 'transparent')

In [ ]:
#性别比例
sex_counts <- table(Diseased_clinical_data$age_group, Diseased_clinical_data$gender)
sex_proportions <- prop.table(sex_counts, 1)

# 转换为长格式
sex_proportions_long <- as.data.frame(as.table(sex_proportions))
names(sex_proportions_long) <- c("Group", "Gender", "Proportion")

# 设置颜色
color_palette <- brewer.pal(12, "Set3")
sex_colors <- c( 'Male' = color_palette[9],
  'Female' = color_palette[8])

# 绘制横置的堆叠条形图
p3 <-ggplot(sex_proportions_long, aes(x = Group, y = Proportion, fill = Gender, label = scales::percent(Proportion))) +
  geom_bar(stat = "identity", position = "fill", width = 0.5) +
  geom_text(position = position_fill(vjust = 0.5), color = "white", fontface = "bold", size = 4.5) +
  scale_fill_manual(values = sex_colors) +
  labs(x = "Group", y = "Proportion", fill = "Gender") +
  theme_minimal() +
  coord_flip()
Sexbar <-p3 + theme_void()
Sexbar

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Diseased_Samples_Metadata_Sex_BarPlot.pdf'), device = 'pdf', width = 10, height = 5, bg = 'transparent')

### Healthy

In [ ]:
Healthy_clinical_data <- read.csv(paste0(obj, '/', 'Healthy_clinical_data2.csv'))

In [ ]:
Healthy_clinical_data %>% colnames()

In [ ]:
head(Healthy_clinical_data)

In [ ]:
#长数据转化
Diseased_clinical_tileplot <- Healthy_clinical_data  %>%
    select(id, gender, condition, smoking, age_group) %>%
    mutate(id = factor(id, levels = rev(id))) %>%
    gather(Category,Value,gender:condition:smoking:age_group)

In [ ]:
# 首先定义颜色
color_palette <- brewer.pal(12, "Set3")

# Define color values for each unique 'Plot_Value'
color_values <- c(
  'Young' = color_palette[1],
  'Old' = color_palette[2],
  'No' = color_palette[3],
  'NA.' = color_palette[4], 
  'Yes' = color_palette[5],
  'Diseased' = color_palette[6],
  'Healthy' = color_palette[7],
  'Male' = color_palette[9],
  'Female' = color_palette[8],
  'IAC' = color_palette[10],
  'MIA' = color_palette[11],
  'AIS' = color_palette[12]
)



# 继续你原有的ggplot代码
tile_plot <- ggplot(Diseased_clinical_tileplot, aes(x = Category, y = id, fill = Value)) + 
  geom_tile(width = 0.8, height = 0.8) + # 调整方块大小
  scale_fill_manual(values = color_values) +
  theme_minimal() +
  labs(x = "ID", y = "Category", fill = "Legend Title") +
  transparent_bg +
  theme(
    axis.text.x = element_text(hjust = 1, size = 16, colour = 'black', angle = 25),  # 调整x轴标签的大小和角度
    axis.text.y = element_text(size = 16, colour = 'black'),  # 调整y轴标签的大小
#    legend.position = "none",
    strip.text = element_text(size = 8),  # 调整分类标签的大小
    strip.background = element_blank(),  # 移除分类标签背景
    panel.border = element_blank(), # 移除面板边框
    panel.grid.major = element_blank(), # 移除主要网格线
    panel.grid.minor = element_blank(), # 移除次要网格线
    panel.spacing.y = unit(0.01, "lines"),  # 减小不同分类变量之间的间距
    plot.margin = unit(c(1, 1, 1, 1), "lines") # 可以尝试调整边缘空间
  )+
  coord_fixed(ratio = 1)+  # 固定x轴和y轴的比例 
  scale_x_discrete(expand = c(0, 0))  # 减少分类变量之间的间距

tile_plot
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Healthy_Samples_Metadata_TilePlot.pdf'), device = 'pdf', width = 5, height = 10, bg = 'transparent')

In [ ]:
#吸烟比例

# 计算比例
smoking_counts <- table(Healthy_clinical_data$age_group, Healthy_clinical_data$smoking)
smoking_proportions <- prop.table(smoking_counts, 1)

# 转换为长格式
smoking_proportions_long <- as.data.frame(as.table(smoking_proportions))
names(smoking_proportions_long) <- c("Group", "Smoking", "Proportion")

# 设置颜色
color_palette <- brewer.pal(12, "Set3")
smoking_colors <- c('No' = color_palette[3],
                    'NA.' = color_palette[4],
                    'Yes' = color_palette[5])

# 绘制横置的堆叠条形图
p2 <-ggplot(smoking_proportions_long, aes(x = Group, y = Proportion, fill = Smoking, label = scales::percent(Proportion))) +
  geom_bar(stat = "identity", position = "fill", width = 0.5) +
  geom_text(position = position_fill(vjust = 0.5), color = "white", fontface = "bold", size = 4.5) +
  scale_fill_manual(values = smoking_colors) +
  labs(x = "Group", y = "Proportion", fill = "Smoking") +
  theme_minimal() +
  coord_flip()
Smokingbar <-p2 + theme_void()
Smokingbar
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Healthy_Samples_Metadata_Smoking_BarPlot.pdf'), device = 'pdf', width = 10, height = 5, bg = 'transparent')

In [ ]:
#性别比例
sex_counts <- table(Healthy_clinical_data$age_group, Healthy_clinical_data$gender)
sex_proportions <- prop.table(sex_counts, 1)

# 转换为长格式
sex_proportions_long <- as.data.frame(as.table(sex_proportions))
names(sex_proportions_long) <- c("Group", "Gender", "Proportion")

# 设置颜色
color_palette <- brewer.pal(12, "Set3")
sex_colors <- c( 'Male' = color_palette[9],
  'Female' = color_palette[8])

# 绘制横置的堆叠条形图
p3 <-ggplot(sex_proportions_long, aes(x = Group, y = Proportion, fill = Gender, label = scales::percent(Proportion))) +
  geom_bar(stat = "identity", position = "fill", width = 0.5) +
  geom_text(position = position_fill(vjust = 0.5), color = "white", fontface = "bold", size = 4.5) +
  scale_fill_manual(values = sex_colors) +
  labs(x = "Group", y = "Proportion", fill = "Gender") +
  theme_minimal() +
  coord_flip()
Sexbar <-p3 + theme_void()
Sexbar
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Healthy_Samples_Metadata_Sex_BarPlot.pdf'), device = 'pdf', width = 10, height = 5, bg = 'transparent')

# CellphoneDB (R part)

In [ ]:
library(ktplots)

In [ ]:
cellphoneDB_obj <- paste0(obj, '/', 'CellphoneDB_db')

## Prepare input

In [ ]:
combine_cpdb_debuged <- function(...) {
    output <- list(...)
    anames <- c("id_cp_interaction", "interacting_pair", "partner_a", "partner_b",
        "gene_a", "gene_b", "secreted", "receptor_a", "receptor_b", "annotation_strategy",
        "is_integrin", 'directionality', 'classification')
    bnames <- c("gene_name", "uniprot", "is_complex", "protein_name", "complex_name",
        "id_cp_interaction")
    if (all(colnames(output[[1]])[1:13] == anames)) {
        out <- output %>%
            reduce(full_join, by = anames)
    } else if (all(colnames(output[[1]])[1:6] == bnames)) {
        out <- output %>%
            reduce(full_join, by = bnames)
    }
    return(out)
}

In [ ]:
#Prepare the DEGs files for method3
T_B_clusters <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                'IFITM3_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex', 
                'Naive_B', 'Memory_B')

#diseased
degs_diseased_T_B_cluster <- as.data.frame(NULL)
for ( i in T_B_clusters) {
    
    if ( i %in% c('Naive_B', 'Memory_B')) {
        deg_df <- readRDS( paste0(obj, '/', 'DEGs_B_Cell_tumor/', paste0('DEGs_B_Cell_', i, '_young_old_tumor_wilcox.rds')))
    } else {
        deg_df <- readRDS( paste0(obj, '/', 'DEGs_T_Cell_tumor/', paste0('DEGs_T_Cell_', i, '_young_old_tumor_wilcox.rds')))
    }
    
    deg_df <- deg_df %>%
        mutate(gene = rownames(.), young_cell_cluster = paste0('Young_',i), old_cell_cluster = paste0('Old_',i)) %>%
        filter(p_val_adj < 0.05, abs(avg_log2FC) > 0.5) %>%
        select(c('young_cell_cluster', 'old_cell_cluster', 'gene'))

    degs_diseased_T_B_cluster <- rbind(degs_diseased_T_B_cluster, deg_df)
}

degs_diseased_T_B_cluster_young <- degs_diseased_T_B_cluster[c('young_cell_cluster', 'gene')]
degs_diseased_T_B_cluster_old <- degs_diseased_T_B_cluster[c('old_cell_cluster', 'gene')]

write_tsv(degs_diseased_T_B_cluster_young, paste0(cellphoneDB_obj, '/', 'diseased_T_B_young_subAnno_DEGs_file.tsv'), col_names = F)
write_tsv(degs_diseased_T_B_cluster_old, paste0(cellphoneDB_obj, '/', 'diseased_T_B_old_subAnno_DEGs_file.tsv'), col_names = F)


#Prepare the DEGs files for method3
T_B_clusters <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                'IFITM3_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex', 
                'Naive_B', 'Memory_B')

#healthy
degs_healthy_T_B_cluster <- as.data.frame(NULL)
for ( i in T_B_clusters) {
    
    if ( i %in% c('Naive_B', 'Memory_B')) {
        deg_df <- readRDS( paste0(obj, '/', 'DEGs_B_Cell_healthy/', paste0('DEGs_B_Cell_', i, '_young_old_healthy_wilcox.rds')))
    } else {
        deg_df <- readRDS( paste0(obj, '/', 'DEGs_T_Cell_healthy/', paste0('DEGs_T_Cell_', i, '_young_old_healthy_wilcox.rds')))
    }
    
    deg_df <- deg_df %>%
        mutate(gene = rownames(.), young_cell_cluster = paste0('Young_',i), old_cell_cluster = paste0('Old_',i)) %>%
        filter(p_val_adj < 0.05, abs(avg_log2FC) > 0.5) %>%
        select(c('young_cell_cluster', 'old_cell_cluster', 'gene'))

    degs_healthy_T_B_cluster <- rbind(degs_healthy_T_B_cluster, deg_df)
}

degs_healthy_T_B_cluster_young <- degs_healthy_T_B_cluster[c('young_cell_cluster', 'gene')]
degs_healthy_T_B_cluster_old <- degs_healthy_T_B_cluster[c('old_cell_cluster', 'gene')]

write_tsv(degs_healthy_T_B_cluster_young, paste0(cellphoneDB_obj, '/', 'healthy_T_B_young_subAnno_DEGs_file.tsv'), col_names = F)
write_tsv(degs_healthy_T_B_cluster_old, paste0(cellphoneDB_obj, '/', 'healthy_T_B_old_subAnno_DEGs_file.tsv'), col_names = F)


## Run CellphoneDB in Python 

## Result analysis

In [ ]:
# seurat objects for cellphoneDB analysis
T_B_clusters <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                'IFITM3_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex', 
                'Naive_B', 'Memory_B')


##split the samples_all_annotation into healthy and tumor
Idents(samples_all_annotation) <- samples_all_annotation$`lung_condition`
samples_all_annotation_diseased <- subset(samples_all_annotation, idents = 'Tumor')
samples_all_annotation_healthy <- subset(samples_all_annotation, idents = 'Healthy')

## subset T_B clusters
Idents(samples_all_annotation_diseased) <- samples_all_annotation_diseased$sub_anno
samples_all_diseased_T_B <- subset(samples_all_annotation_diseased, idents = T_B_clusters)

Idents(samples_all_annotation_healthy) <- samples_all_annotation_healthy$sub_anno
samples_all_healthy_T_B <- subset(samples_all_annotation_healthy, idents = T_B_clusters)

>NOTE: Should use the `readr::read_tsv` to read the cellphoneDB txt file instead of `utils::read.tsv` or `utils::read.table`  
> 
>NOTE: The data loaded by `readr::read_tsv` should be converted into `data.frame` format. Otherwise, the ktplots plotting function will report error

In [ ]:
# method3_diseased_T_B_subAnno: means, rel_int_degs
cellphoneDB_method3_diseased_T_B_young_subAnno_means_df <- read_tsv(paste0(cellphoneDB_obj, '/', 'degs_analysis_means_samples_all_diseased_T_B_young_subAnno_cpdb_method3.txt'), num_threads = 20) %>% as.data.frame()
cellphoneDB_method3_diseased_T_B_young_subAnno_rel_df <- read_tsv(paste0(cellphoneDB_obj, '/', 'degs_analysis_relevant_interactions_samples_all_diseased_T_B_young_subAnno_cpdb_method3.txt'), num_threads = 20) %>% as.data.frame()

cellphoneDB_method3_diseased_T_B_old_subAnno_means_df <- read_tsv(paste0(cellphoneDB_obj, '/', 'degs_analysis_means_samples_all_diseased_T_B_old_subAnno_cpdb_method3.txt'), num_threads = 20) %>% as.data.frame()
cellphoneDB_method3_diseased_T_B_old_subAnno_rel_df <- read_tsv(paste0(cellphoneDB_obj, '/', 'degs_analysis_relevant_interactions_samples_all_diseased_T_B_old_subAnno_cpdb_method3.txt'), num_threads = 20) %>% as.data.frame()


# method3_healthy_T_B_subAnno: means, rel_int_degs
cellphoneDB_method3_healthy_T_B_young_subAnno_means_df <- read_tsv(paste0(cellphoneDB_obj, '/', 'degs_analysis_means_samples_all_healthy_T_B_young_subAnno_cpdb_method3.txt'), num_threads = 20) %>% as.data.frame()
cellphoneDB_method3_healthy_T_B_young_subAnno_rel_df <- read_tsv(paste0(cellphoneDB_obj, '/', 'degs_analysis_relevant_interactions_samples_all_healthy_T_B_young_subAnno_cpdb_method3.txt'), num_threads = 20) %>% as.data.frame()

cellphoneDB_method3_healthy_T_B_old_subAnno_means_df <- read_tsv(paste0(cellphoneDB_obj, '/', 'degs_analysis_means_samples_all_healthy_T_B_old_subAnno_cpdb_method3.txt'), num_threads = 20) %>% as.data.frame()
cellphoneDB_method3_healthy_T_B_old_subAnno_rel_df <- read_tsv(paste0(cellphoneDB_obj, '/', 'degs_analysis_relevant_interactions_samples_all_healthy_T_B_old_subAnno_cpdb_method3.txt'), num_threads = 20) %>% as.data.frame()


In [ ]:
# method3_diseased_T_B_subAnno: combined means, rel_int_degs files
cellphoneDB_method3_combined_diseased_T_B_subAnno_rel_df <- combine_cpdb_debuged(cellphoneDB_method3_diseased_T_B_young_subAnno_rel_df, cellphoneDB_method3_diseased_T_B_old_subAnno_rel_df)
cellphoneDB_method3_combined_diseased_T_B_subAnno_means_df <- combine_cpdb_debuged(cellphoneDB_method3_diseased_T_B_young_subAnno_means_df, cellphoneDB_method3_diseased_T_B_old_subAnno_means_df)

# method3_healthy_T_B_subAnno: combined means, rel_int_degs files
cellphoneDB_method3_combined_healthy_T_B_subAnno_rel_df <- combine_cpdb_debuged(cellphoneDB_method3_healthy_T_B_young_subAnno_rel_df, cellphoneDB_method3_healthy_T_B_old_subAnno_rel_df)
cellphoneDB_method3_combined_healthy_T_B_subAnno_means_df <- combine_cpdb_debuged(cellphoneDB_method3_healthy_T_B_young_subAnno_means_df, cellphoneDB_method3_healthy_T_B_old_subAnno_means_df)


## Plotting

In [ ]:
B_subcluster <- c('Naive_B', 'Memory_B')
CD4_subcluster <- c('CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg')
CD8_subcluster <- c('IFITM3_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex')

In [ ]:
# get the CD4_T and B celltype pairs for plotting
CD4T_B_celltype_pairs = vector(mode = 'character')

for (groups in list(c('CD4_subcluster', 'B_subcluster'), c('B_subcluster', 'CD4_subcluster'))) {
    for (i in get(groups[1])) {
        for (j in get(groups[2])) {
            for (k in c('Young', 'Old')) {
                pairs <- paste0(i, ' > ', j, '_', k)
                CD4T_B_celltype_pairs = c(CD4T_B_celltype_pairs, pairs)
            }
        }
    }
}

In [ ]:
# get the CD8_T and B celltype pairs for plotting
CD8T_B_celltype_pairs = vector(mode = 'character')

for (groups in list(c('CD8_subcluster', 'B_subcluster'), c('B_subcluster', 'CD8_subcluster'))) {
    for (i in get(groups[1])) {
        for (j in get(groups[2])) {
            for (k in c('Young', 'Old')) {
                pairs <- paste0(i, ' > ', j, '_', k)
                CD8T_B_celltype_pairs = c(CD8T_B_celltype_pairs, pairs)
            }
        }
    }
}

In [ ]:
plot_cpdb_manual <- function(plot_cpdb_df, vline_position, max_size=8){
    DEFAULT_SEP <- ">@<"
    SPECIAL_SEP <- paste0(rep(DEFAULT_SEP, 3), collapse = "")
    
    #manipulate the df from ktplot::plot_cpdb
    df <- plot_cpdb_df
    
    df$Var1 <- gsub(paste0(".*", SPECIAL_SEP), "", df$Var1)
    df$Age_type <- str_split_i(df$Var2, pattern = '_', i = 1) #Get the age_type
    df$Var2 <- gsub("Old_|Young_", "", df$Var2)
    df$Var2 <- gsub("-", " > ", df$Var2)
    df <- unite(df, col = 'Var2_Age_Group', Var2, Age_type, remove = F)

    #set the scaled_means as zero if no significant
    #df$scaled_means[df$significant == 'no'] <- 0.25
    
    #remove the interactions invoving integrin
    df <- filter(df, !str_detect(string = df$Var1, pattern = 'integrin'))
    
    #plot
    g <- ggplot(df, aes(x = Var2_Age_Group, y = Var1, fill = Age_type, color = significant, size = scaled_means))

    g <- g +
        geom_point(na.rm = TRUE, stroke = 1.5, shape = 21) +
        geom_vline(xintercept = vline_position, linetype='dashed') +
#        scale_x_discrete(position = "top") +
        #max_size:8
        scale_radius(range = c(0,max_size)) +
        scale_color_manual(values = c('yes' = '#B31A2C', 'no' = 'white')) +
        scale_fill_manual(values = age_group_color, na.value = NA, na.translate = FALSE, limits = c('Young', 'Old')) +
        guides(
            size = guide_legend(title = 'Scaled_Means', reverse = TRUE, order = 2),
            color = guide_legend(reverse = TRUE,order = 3)) +
        theme_bw(base_size = 20) +
        theme(text = element_text(color = 'black'),
              axis.text.x = element_text(angle = 41, hjust = 0, color = "#000000"),
              axis.text.y = element_text(color = "#000000"),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              panel.grid=element_blank(),
              legend.direction = "vertical",
              legend.box = "vertical",
              legend.title = element_text(size = 15))
    
    return(g)
}

### B_CD4+T

#### Immune gene_family

In [ ]:
# get the df from ktplot::plot_cpdb; plot_cpdb_diseased_B_CD4_df
plot_cpdb_diseased_B_CD4_T_Th_df <- plot_cpdb(
    scdata=samples_all_diseased_T_B,
    cell_type1="Naive_B|Memory_B",
    cell_type2='CCR7_CD4_Tnaive|CXCR6_CD4_Trm|CXCL13_CD4_Tex|FOXP3_CD4_Treg',  # this means all cell-types
    celltype_key="sub_anno",
    splitby_key = "Age_type",
    means=cellphoneDB_method3_combined_diseased_T_B_subAnno_means_df,
    pvals=cellphoneDB_method3_combined_diseased_T_B_subAnno_rel_df,
    keep_significant_only = T,
    degs_analysis = T,
    gene_family = c("Th1", 'Th2', 'Th17', 'Treg', 'niche', 'chemokines', 'costimulatory', 'coinhibitory'),
#    gene_family = c('chemokines'),
    return_table = T
)

# get the df from ktplot::plot_cpdb; plot_cpdb_healthy_B_CD4_df
plot_cpdb_healthy_B_CD4_T_Th_df <- plot_cpdb(
    scdata=samples_all_healthy_T_B,
    cell_type1="Naive_B|Memory_B",
    cell_type2='CCR7_CD4_Tnaive|CXCR6_CD4_Trm|CXCL13_CD4_Tex|FOXP3_CD4_Treg',  # this means all cell-types
    celltype_key="sub_anno",
    splitby_key = "Age_type",
    means=cellphoneDB_method3_combined_healthy_T_B_subAnno_means_df,
    pvals=cellphoneDB_method3_combined_healthy_T_B_subAnno_rel_df,
    keep_significant_only = T,
    degs_analysis = T,
    gene_family = c("Th1", 'Th2', 'Th17', 'Treg', 'niche', 'chemokines', 'costimulatory', 'coinhibitory'),
#    gene_family = c('chemokines'),
    return_table = T
)

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15)
plot_cpdb_manual(plot_cpdb_df = plot_cpdb_diseased_B_CD4_T_Th_df, vline_position = seq(from = 2.5, to = 40, by = 2)) +
    scale_x_discrete(limits = CD4T_B_celltype_pairs) +
    coord_flip() +
    transparent_bg +
    theme(axis.text.x = element_text(angle = 90, hjust = 0, color = "#000000"))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Diseased_B_CD4T_Cells_Interaction_ImmReceptor_DotPlot.pdf'), device = 'pdf', width = 10, height = 16, bg = 'transparent')

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15)
plot_cpdb_manual(plot_cpdb_df = plot_cpdb_healthy_B_CD4_T_Th_df, vline_position = seq(from = 2.5, to = 40, by = 2)) +
    scale_x_discrete(limits = CD4T_B_celltype_pairs) +
    coord_flip() +
    transparent_bg +
    theme(axis.text.x = element_text(angle = 90, hjust = 0, color = "#000000"))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Healthy_B_CD4T_Cells_Interaction_ImmReceptor_DotPlot.pdf'), device = 'pdf', width = 10, height = 16, bg = 'transparent')

#### All gene_family

In [ ]:
# get the df from ktplot::plot_cpdb; plot_cpdb_diseased_B_CD4_df
plot_cpdb_diseased_B_CD4_T_Th_df <- plot_cpdb(
    scdata=samples_all_diseased_T_B,
    cell_type1="Naive_B|Memory_B",
    cell_type2='CCR7_CD4_Tnaive|CXCR6_CD4_Trm|CXCL13_CD4_Tex|FOXP3_CD4_Treg',  # this means all cell-types
    celltype_key="sub_anno",
    splitby_key = "Age_type",
    means=cellphoneDB_method3_combined_diseased_T_B_subAnno_means_df,
    pvals=cellphoneDB_method3_combined_diseased_T_B_subAnno_rel_df,
    keep_significant_only = T,
    degs_analysis = T,
#    gene_family = c("Th1", 'Th2', 'Th17', 'Treg', 'niche', 'chemokines', 'costimulatory', 'coinhibitory'),
#    gene_family = c('chemokines'),
    return_table = T
)

# get the df from ktplot::plot_cpdb; plot_cpdb_healthy_B_CD4_df
plot_cpdb_healthy_B_CD4_T_Th_df <- plot_cpdb(
    scdata=samples_all_healthy_T_B,
    cell_type1="Naive_B|Memory_B",
    cell_type2='CCR7_CD4_Tnaive|CXCR6_CD4_Trm|CXCL13_CD4_Tex|FOXP3_CD4_Treg',  # this means all cell-types
    celltype_key="sub_anno",
    splitby_key = "Age_type",
    means=cellphoneDB_method3_combined_healthy_T_B_subAnno_means_df,
    pvals=cellphoneDB_method3_combined_healthy_T_B_subAnno_rel_df,
    keep_significant_only = T,
    degs_analysis = T,
#    gene_family = c("Th1", 'Th2', 'Th17', 'Treg', 'niche', 'chemokines', 'costimulatory', 'coinhibitory'),
#    gene_family = c('chemokines'),
    return_table = T
)

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15)
plot_cpdb_manual(plot_cpdb_df = plot_cpdb_diseased_B_CD4_T_Th_df, vline_position = seq(from = 2.5, to = 40, by = 2)) +
    scale_x_discrete(limits = CD4T_B_celltype_pairs) +
    coord_flip() +
    transparent_bg +
    theme(axis.text.x = element_text(angle = 90, hjust = 0, color = "#000000"))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Diseased_B_CD4T_Cells_Interaction_AllReceptor_DotPlot.pdf'), device = 'pdf', width = 11, height = 16, bg = 'transparent')

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15)
plot_cpdb_manual(plot_cpdb_df = plot_cpdb_healthy_B_CD4_T_Th_df, vline_position = seq(from = 2.5, to = 40, by = 2)) +
    scale_x_discrete(limits = CD4T_B_celltype_pairs) +
    coord_flip() +
    transparent_bg +
    theme(axis.text.x = element_text(angle = 90, hjust = 0, color = "#000000"))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Healthy_B_CD4T_Cells_Interaction_AllReceptor_DotPlot.pdf'), device = 'pdf', width = 11, height = 16, bg = 'transparent')

### B_CD8+T

#### Immune gene_family

In [ ]:
# get the df from ktplot::plot_cpdb; plot_cpdb_diseased_B_CD8_df
plot_cpdb_diseased_B_CD8_T_Th_df <- plot_cpdb(
    scdata=samples_all_diseased_T_B,
    cell_type1="Naive_B|Memory_B",
    cell_type2='IFITM3_CD8_Teffector|GZMB_CD8_Teffector|GZMK_CD8_Tem|ZNF683_CD8_Trm|CXCL13_CD8_Tex',  # this means all cell-types
    celltype_key="sub_anno",
    splitby_key = "Age_type",
    means=cellphoneDB_method3_combined_diseased_T_B_subAnno_means_df,
    pvals=cellphoneDB_method3_combined_diseased_T_B_subAnno_rel_df,
    keep_significant_only = T,
    degs_analysis = T,
    gene_family = c("Th1", 'Th2', 'Th17', 'Treg', 'niche', 'chemokines', 'costimulatory', 'coinhibitory'),
#    gene_family = c('chemokines'),
    return_table = T
)

# get the df from ktplot::plot_cpdb; plot_cpdb_healthy_B_CD4_df
plot_cpdb_healthy_B_CD8_T_Th_df <- plot_cpdb(
    scdata=samples_all_healthy_T_B,
    cell_type1="Naive_B|Memory_B",
    cell_type2='IFITM3_CD8_Teffector|GZMB_CD8_Teffector|GZMK_CD8_Tem|ZNF683_CD8_Trm|CXCL13_CD8_Tex',  # this means all cell-types
    celltype_key="sub_anno",
    splitby_key = "Age_type",
    means=cellphoneDB_method3_combined_healthy_T_B_subAnno_means_df,
    pvals=cellphoneDB_method3_combined_healthy_T_B_subAnno_rel_df,
    keep_significant_only = T,
    degs_analysis = T,
    gene_family = c("Th1", 'Th2', 'Th17', 'Treg', 'niche', 'chemokines', 'costimulatory', 'coinhibitory'),
#    gene_family = c('chemokines'),
    return_table = T
)

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15)
plot_cpdb_manual(plot_cpdb_df = plot_cpdb_diseased_B_CD8_T_Th_df, vline_position = seq(from = 2.5, to = 40, by = 2)) +
    scale_x_discrete(limits = CD8T_B_celltype_pairs) +
    coord_flip() +
    transparent_bg +
    theme(axis.text.x = element_text(angle = 90, hjust = 0, color = "#000000"))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Diseased_B_CD8T_Cells_Interaction_ImmReceptor_DotPlot.pdf'), device = 'pdf', width = 11, height = 16, bg = 'transparent')

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15)
plot_cpdb_manual(plot_cpdb_df = plot_cpdb_healthy_B_CD8_T_Th_df, vline_position = seq(from = 2.5, to = 40, by = 2)) +
    scale_x_discrete(limits = CD8T_B_celltype_pairs) +
    coord_flip() +
    transparent_bg +
    theme(axis.text.x = element_text(angle = 90, hjust = 0, color = "#000000"))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Healthy_B_CD8T_Cells_Interaction_ImmReceptor_DotPlot.pdf'), device = 'pdf', width = 11, height = 16, bg = 'transparent')

#### All gene_family

In [ ]:
# get the df from ktplot::plot_cpdb; plot_cpdb_diseased_B_CD8_df
plot_cpdb_diseased_B_CD8_T_Th_df <- plot_cpdb(
    scdata=samples_all_diseased_T_B,
    cell_type1="Naive_B|Memory_B",
    cell_type2='IFITM3_CD8_Teffector|GZMB_CD8_Teffector|GZMK_CD8_Tem|ZNF683_CD8_Trm|CXCL13_CD8_Tex',  # this means all cell-types
    celltype_key="sub_anno",
    splitby_key = "Age_type",
    means=cellphoneDB_method3_combined_diseased_T_B_subAnno_means_df,
    pvals=cellphoneDB_method3_combined_diseased_T_B_subAnno_rel_df,
    keep_significant_only = T,
    degs_analysis = T,
#    gene_family = c("Th1", 'Th2', 'Th17', 'Treg', 'niche', 'chemokines', 'costimulatory', 'coinhibitory'),
#    gene_family = c('chemokines'),
    return_table = T
)

# get the df from ktplot::plot_cpdb; plot_cpdb_healthy_B_CD4_df
plot_cpdb_healthy_B_CD8_T_Th_df <- plot_cpdb(
    scdata=samples_all_healthy_T_B,
    cell_type1="Naive_B|Memory_B",
    cell_type2='IFITM3_CD8_Teffector|GZMB_CD8_Teffector|GZMK_CD8_Tem|ZNF683_CD8_Trm|CXCL13_CD8_Tex',  # this means all cell-types
    celltype_key="sub_anno",
    splitby_key = "Age_type",
    means=cellphoneDB_method3_combined_healthy_T_B_subAnno_means_df,
    pvals=cellphoneDB_method3_combined_healthy_T_B_subAnno_rel_df,
    keep_significant_only = T,
    degs_analysis = T,
#    gene_family = c("Th1", 'Th2', 'Th17', 'Treg', 'niche', 'chemokines', 'costimulatory', 'coinhibitory'),
#    gene_family = c('chemokines'),
    return_table = T
)

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15)
plot_cpdb_manual(plot_cpdb_df = plot_cpdb_diseased_B_CD8_T_Th_df, vline_position = seq(from = 2.5, to = 40, by = 2)) +
    scale_x_discrete(limits = CD8T_B_celltype_pairs) +
    coord_flip() +
    transparent_bg +
    theme(axis.text.x = element_text(angle = 90, hjust = 0, color = "#000000"))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Diseased_B_CD8T_Cells_Interaction_AllReceptor_DotPlot.pdf'), device = 'pdf', width = 13, height = 16, bg = 'transparent')

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15)
plot_cpdb_manual(plot_cpdb_df = plot_cpdb_healthy_B_CD8_T_Th_df, vline_position = seq(from = 2.5, to = 40, by = 2)) +
    scale_x_discrete(limits = CD8T_B_celltype_pairs) +
    coord_flip() +
    transparent_bg +
    theme(axis.text.x = element_text(angle = 90, hjust = 0, color = "#000000"))

ggsave(filename = paste0(figures, '/Figures_raw', '/', 'Healthy_B_CD8T_Cells_Interaction_AllReceptor_DotPlot.pdf'), device = 'pdf', width = 13, height = 16, bg = 'transparent')

# All_8samples_st_analysis

## Data load and preprocess

In [ ]:
# tissue_position_function
get_tissue_position <- function(coord_dir, filter.matrix = TRUE) {
    coord_file_path <- Sys.glob(paths = file.path(coord_dir, 'tissue_positions*'))
    tissue.positions <- read.csv(file = coord_file_path, 
                                 col.names = c('barcodes', 'tissue', 'row', 'col', 'imagerow', 'imagecol'),
                                 header = ifelse(test = basename(coord_file_path) == "tissue_positions.csv",yes = TRUE,no = FALSE),
                                 as.is = TRUE,
                                 row.names = 1)
    if (filter.matrix) {
        tissue.positions <- tissue.positions[which(x = tissue.positions$tissue == 1), , drop = FALSE]
    }
    
    tissue.positions['Cell'] <- rownames(tissue.positions)
    
    return(tissue.positions)
}

In [ ]:
# Load luad_dataset
luad_a <- Load10X_Spatial('/public/home/liwang/project/lung_cancer_ST/4FFPE/LUAD-A/outs', slice = 'luad_a')
luad_b <- Load10X_Spatial('/public/home/liwang/project/lung_cancer_ST/4FFPE/LUAD-B/outs', slice = 'luad_b')
luad_c <- Load10X_Spatial('/public/home/liwang/project/lung_cancer_ST/4FFPE/LUAD-C/outs', slice = 'luad_c')

In [ ]:
## complete the metadata
luad_a$orig.ident <- 'LUAD_A'
luad_a$Age <- 33
luad_a$Age_type <- 'Young'
luad_b$orig.ident <- 'LUAD_B'
luad_b$Age <- 30
luad_b$Age_type <- 'Young'
luad_c$orig.ident <- 'LUAD_C'
luad_c$Age <- 32
luad_c$Age_type <- 'Young'

In [ ]:
## Load emm_2022_taojiang_ST dataset
for (emm_samples in c('TD1', 'TD3', 'TD5', 'TD6', 'TD8')) {
    assign(x = paste0(str_to_lower(emm_samples), '_matrix'),
           value = Read10X(data.dir = paste0('/public/home/liwang/project/lung_cancer_ST/ST_data/emm_2022_taojiang/expression_matrix_ST/', emm_samples)))
    
    assign(x = str_to_lower(emm_samples),
           value = CreateSeuratObject(counts = get(paste0(str_to_lower(emm_samples), '_matrix')), project = emm_samples, assay = 'Spatial'))
}

In [ ]:
## complete the metadata
td1$Age <- 57
td1$Age_type <- 'Old'

td3$Age <- 37
td3$Age_type <- 'Young'

td5$Age <- 56
td5$Age_type <- 'Old'

td6$Age <- 56
td6$Age_type <- 'Old'

td8$Age <- 69
td8$Age_type <- 'Old'

In [ ]:
## Load emm_2022_taojiang_ST dataset (coordinate)
for (emm_samples in c('TD1', 'TD3', 'TD5', 'TD6', 'TD8')) {
    assign(x = paste0(str_to_lower(emm_samples), '_coord'), 
           value = get_tissue_position(coord_dir = paste0('/public/home/liwang/project/lung_cancer_ST/ST_data/emm_2022_taojiang/expression_matrix_ST/', emm_samples)))
}

## Load luad_ST dataset (coordinate)
for (luad_samples in c('LUAD-A', 'LUAD-B', 'LUAD-C')) {
    luad_names <- str_replace(string = luad_samples, pattern = '-', replacement = '_')
    assign(x = paste0(str_to_lower(luad_names), '_coord'), 
           value = get_tissue_position(coord_dir = paste0('/public/home/liwang/project/lung_cancer_ST/4FFPE/', luad_samples, '/outs/spatial'))) 
}

In [ ]:
# add the coord into metadata
for (ST_sample in c('luad_a', 'luad_b', 'luad_c', 'td1', 'td3', 'td5', 'td6', 'td8')) {
    assign(x = ST_sample,
           value = AddMetaData(object = get(ST_sample), metadata = get(paste0(ST_sample, '_coord'))))
}

In [ ]:
## subset the intersected genes

ST_samples_list <- list('luad_a' = luad_a, 'luad_b' = luad_b, 'luad_c' = luad_c,
                        'td1' = td1, 'td3' = td3, 'td5' = td5, 'td6' = td6, 'td8' = td8)

shared_genes <- rownames(luad_a)
for (sample in ST_samples_list) {
    shared_genes <- intersect(x = shared_genes, y = rownames(sample))
}

ST_samples_list <- lapply(ST_samples_list, FUN = function(x) subset(x, features = shared_genes))


In [ ]:
lapply(ST_samples_list, FUN = function(x) NCOL(x))

In [ ]:
## Filter spots based on nUMI (nCount_Spatial > 500)

ST_samples_list <- lapply(X = ST_samples_list, FUN = function(x) {
    x <- PercentageFeatureSet(x, pattern = "^MT-", col.name = "pMT")
    x <- PercentageFeatureSet(x, pattern = "^HBA|^HBB", col.name = "pHB")
})

ST_samples_list <- lapply(ST_samples_list , FUN = function(x) subset(x, subset = nCount_Spatial > 500  & pMT < 12 & pHB < 5))


In [ ]:
lapply(ST_samples_list, FUN = function(x) NCOL(x))

In [ ]:
## SCTransform
ST_samples_list <- lapply(ST_samples_list , FUN = function(x) {
    x <- SCTransform(object = x, assay = "Spatial", vars.to.regress = c('nCount_Spatial'))
})


In [ ]:
## merge objects
all_8samples_st <- merge(x = ST_samples_list[[1]], y = ST_samples_list[2:8])

In [ ]:
## Recorrect the UMI based on the same sequencing depth in SCT model
all_8samples_st <- PrepSCTFindMarkers(all_8samples_st)

In [ ]:
saveRDS(all_8samples_st, paste0(obj, '/', 'all_8samples_st_merged.rds'))

In [ ]:
all_8samples_st <- readRDS(paste0(obj, '/', 'all_8samples_st_merged.rds'))

In [ ]:
all_8samples_st

## TLS Region Analysis

### ssGSEA

In [ ]:
# prepare the counts matrix as input
all_8sampels_st_counts <- GetAssayData(object = all_8samples_st, slot = 'counts', assay = 'Spatial')

In [ ]:
## prepare the predefined gene set
msigdb_c2_kegg <- msigdbr(species = 'Homo sapiens', category = "C2",subcategory = "KEGG")
msigdb_c2_kegg_list <- msigdb_c2_kegg %>% split(x = .$gene_symbol, f = .$gs_description)

msigdb_c5_GOBP <- msigdbr(species = 'Homo sapiens', category = "C5",subcategory = "GO:BP")
msigdb_c5_GOBP_list <- msigdb_c5_GOBP %>% split(x = .$gene_symbol, f = .$gs_name)

In [ ]:
## prepare TLS_signature genesets
TLS_signature_genesets <- list(
                           'TLS_signature_from_Cho_Science_2026' = c('MS4A1', 'CD19', 'CD22', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'CD74', 'CD79A', 'IL7R', 'ITGAE', 'CD1D', 'CD52', 'CD79B', 'FCER2', 'CR2', 'PDCD1', 'PTGDS', 'TRBC2', 'CXCL13', 'CXCR5'),     
                           'TLS_signature_from_Rita_Nature_2020' = c('CD79B', 'CD1D', 'CCR6', 'LAT', 'SKAP1', 'CETP', 'EIF1AY', 'RBP5', 'PTGDS'),
                           'TLS_signature_from_Dieu_Trends_Immunol_2014' = c('CCL19', 'CCL21', 'CXCL13', 'CCR7', 'CXCR5', 'SELL', 'LAMP3'), 
                           'TLS_signature_from_Luc_Cancer_Res_2011' = c('CCL19', 'CXCL13', 'CCL21', 'IL16', 'CCL22', 'CCL17', 'ITGAL', 'ITGAD', 'ITGA4', 'ICAM3', 'VCAM1', 'MADCAM1'), 
                           'Antigen_presentation_from_KEGG' = unique(msigdb_c2_kegg_list$'Antigen processing and presentation'),
                           'GOBP_POSITIVE_REGULATION_OF_RESPONSE_TO_TUMOR_CELL' = unique(msigdb_c5_GOBP_list$`GOBP_POSITIVE_REGULATION_OF_RESPONSE_TO_TUMOR_CELL`)
                          )



- B Cell activation signature
    - c('LYN', 'CD22', 'CD81', 'TNFRSF13C', 'BTLA') from `B-cell-specific checkpoint molecules that regulate anti-tumour immunity`
    - c('CD69', 'CD83', 'IER2', 'DUSP2', 'IL6', 'NR4A2', 'JUN', 'CCR7', 'GPR183') from `Single-cell analysis of human B cell maturation predicts how antibody class switching shapes selection dynamics`
    - c('CD83', 'CD81', 'IL21', 'IL21R', 'IL4R', 'BCL6') from `INTEGRATED SINGLE-CELL TRANSCRIPTOMICS AND EPIGENOMICS REVEALS STRONG GERMINAL CENTER–ASSOCIATED ETIOLOGY OF AUTOIMMUNE RISK LOCI`
    - c('NME1', 'ITGAE', 'ITGA1', 'TNFRSF4', 'TNF', 'HLA-DRA', 'ICOS', 'IL2RA') from `Autoreactive T cells target peripheral nerves in Guillain–Barré syndrome`
    - c('FOXMI', 'PLK1', 'BUB1', 'CCNB1', 'TOP2A', 'MKI67') from `Autoreactive T cells target peripheral nerves in Guillain–Barré syndrome`
    
- B Cell inhibition signature
    - c('FCRLA', 'FCRL2', 'FCRL3', 'CBLB', 'CD72', 'SIGLEC10') from `Single-cell analysis of human B cell maturation predicts how antibody class switching shapes selection dynamics`

In [ ]:
# Run ssgsea
all_8sampels_st_ssgsea <- gsva(expr = all_8sampels_st_counts, gset.idx.list = TLS_signature_genesets, method='ssgsea', kcdf='Poisson', ssgsea.norm=T)

In [ ]:
## reshape the all_8sampels_st_ssgsea_df for plotting
all_8samples_st_metadata <- FetchData(all_8samples_st, vars = c('orig.ident', 'Age', 'Age_type')) %>% mutate(barcode = rownames(.), PatientID = orig.ident)

all_8sampels_st_ssgsea_df <- all_8sampels_st_ssgsea %>% as.matrix() %>% t() %>% as.data.frame() %>% mutate(barcode = rownames(.))

all_8samples_st_metadata <- left_join(x = all_8samples_st_metadata, y = all_8sampels_st_ssgsea_df, by = 'barcode')

In [ ]:
## add ssGSEA score into metadata for spatial plot

rownames(all_8samples_st_metadata) <- all_8samples_st_metadata$barcode
all_8samples_st <- AddMetaData(object = all_8samples_st, metadata = all_8samples_st_metadata)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)
ggplot(subset(all_8samples_st, subset = PatientID == 'TD8')[[]], aes(x = imagecol, y = imagerow, color = TLS_signature_from_Dieu_Trends_Immunol_2014)) +
  geom_point(size = 3, stroke = NA) +           
  scale_color_gradientn(
    colors = c("#440154", "#3B528B", "#21918C", "#5EC962", "#FDE725"),  # viridis 配色
    name = "TLS\nscore"
  ) +
  scale_y_reverse() +                              
  coord_fixed() +                                  
  theme_void() +                                   
  theme(
    legend.position = "right",
    plot.margin = margin(5, 10, 5, 5)
  )

In [ ]:
all_8samples_st

In [ ]:
## Determine the TLS region based on TLS score (top 2%)
all_8samples_st$'TLS_Region_ssGSEA' <- ifelse(all_8samples_st$TLS_signature_from_Dieu_Trends_Immunol_2014 >= quantile(x = all_8samples_st$TLS_signature_from_Dieu_Trends_Immunol_2014, probs = 0.98),
                                              yes = 'TLS_Region', no = 'NonTLS_Region')

all_8samples_st_TLS_Region_ssGSEA <- subset(all_8samples_st, subset = TLS_Region_ssGSEA == 'TLS_Region')

In [ ]:
options(repr.plot.width = 24, repr.plot.height = 12)

# 1. 
plot_df <- all_8samples_st[[]]

# 2. 
sample_order <- c('LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3', 'TD1', 'TD5', 'TD6', 'TD8')
plot_df$PatientID <- factor(plot_df$PatientID, levels = sample_order)

# 3. 
ggplot(subset(plot_df, PatientID %in% sample_order),
       aes(x = imagecol, y = imagerow, color = TLS_Region_ssGSEA)) +
  geom_point(size = 1.8, stroke = NA) +
  scale_y_reverse() +
  coord_fixed() +
  facet_wrap(~ PatientID, nrow = 2) +            # 2 行 4 列，按 levels 顺序
  theme_void() +
  theme(
    legend.position = "right",
    strip.text = element_text(size = 12, face = "bold"),
    plot.margin = margin(5, 10, 5, 5)
  )

In [ ]:
# convert the all_8samples_st with ssGSEA score to h5ad

sceasy::convertFormat(all_8samples_st, from="seurat", to="anndata", assay = 'Spatial',
                      main_layer = 'counts', drop_single_values = FALSE,
                      outFile=paste0(obj, '/', 'all_8samples_st_ssGSEA_Spatial_counts.h5ad'))

sceasy::convertFormat(all_8samples_st, from="seurat", to="anndata", assay = 'SCT',
                      main_layer = 'data', drop_single_values = FALSE,
                      outFile=paste0(obj, '/', 'all_8samples_st_ssGSEA_SCT_data.h5ad'))

In [ ]:
saveRDS(all_8samples_st, paste0(obj, '/', 'all_8samples_st_ssGSEA.rds'))

In [ ]:
all_8samples_st <- readRDS(paste0(obj, '/', 'all_8samples_st_ssGSEA.rds'))

### DEGs analysis

In [ ]:
## Recorrect the UMI based on the same sequencing depth in SCT model
all_8samples_st <- PrepSCTFindMarkers(all_8samples_st)

In [ ]:
#DEGs

Idents(all_8samples_st) <- all_8samples_st$TLS_Region_ssGSEA
DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox <- FindMarkers(object = all_8samples_st, assay = 'SCT', slot = 'data', test.use = "wilcox",
                                                          subset.ident = 'TLS_Region', logfc.threshold = 0, min.pct = 0.1, 
                                                          group.by = 'Age_type', `ident.1` = 'Young', `ident.2` = 'Old', recorrect_umi = FALSE)

In [ ]:
saveRDS(DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox, paste0(obj, '/', 'DEGs_ST_TLS_Region/', 'DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox.rds'))

In [ ]:

DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox <- readRDS(paste0(obj, '/', 'DEGs_ST_TLS_Region/', 'DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox.rds'))

In [ ]:
DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox %>% filter(p_val_adj < 0.05 & avg_log2FC < -0.5) %>% arrange(desc(avg_log2FC)) %>% mutate(gene = rownames(.)) %>% pull(gene) 

In [ ]:
DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox %>% mutate(gene = rownames(.)) %>% filter(gene %in% c("CD3D", "MS4A1", "CR2", "FCER2", "MKI67", "BCL6"))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
volcano_plot(DEGs_df = DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox, col_names_pval = 'p_val_adj', col_names_LFC = 'avg_log2FC',
             pval_cutoff = 0.01, logFC_cutoff = 0.5, label_logFC_cutoff = 0.75, label_pval_cutoff = 0.01,
             title = NULL) + transparent_bg

In [ ]:
volcano_plot(DEGs_df = DEGs_ST_TLS_Region_ssGSEA_young_old_wilcox, col_names_pval = 'p_val_adj', col_names_LFC = 'avg_log2FC',
             pval_cutoff = 0.01, logFC_cutoff = 0.5, label_logFC_cutoff = 0.75, label_pval_cutoff = 0.01,
             title = NULL) + transparent_bg
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'DEGs_ST_TLS_Regions_ssGSEA_wilcox_VolcanoPlot.pdf'), device = 'pdf', width = 10, height = 10, bg = 'transparent')

### Signature Score Plot

In [ ]:
unique(msigdb_c5_GOBP_list$`GOBP_POSITIVE_REGULATION_OF_RESPONSE_TO_TUMOR_CELL`)

In [ ]:
colnames(all_8samples_st_TLS_Region_ssGSEA[[]])

In [ ]:
#Antigen_presentation_from_KEGG
options(repr.plot.width = 12, repr.plot.height = 6)
p1 <- all_8samples_st_TLS_Region_ssGSEA[[]] %>% 
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = Age_type, y = Antigen_presentation_from_KEGG, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    stat_compare_means(comparisons = list(c('Young', 'Old')), label = 'p.format', bracket.size = 1, method = 'wilcox.test') +
    scale_fill_manual(values = age_group_color, guide = NULL) +
#    geom_hline(yintercept = 0, linetype='dashed') +
    ylab(label = 'Antigen Process and Presentation Score') +
    xlab(label = NULL) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(plot.title = element_text(hjust = 0.5),
          legend.position = 'top',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

#GOBP_POSITIVE_REGULATION_OF_RESPONSE_TO_TUMOR_CELL
p2 <- all_8samples_st_TLS_Region_ssGSEA[[]] %>% 
    mutate(Age_type = factor(Age_type, levels = c('Young', 'Old'))) %>%
    ggplot(mapping = aes(x = Age_type, y = GOBP_POSITIVE_REGULATION_OF_RESPONSE_TO_TUMOR_CELL, fill = Age_type)) +
    geom_violin(linewidth=1, position = position_dodge(0.85), alpha = 0.9) +
    geom_boxplot(notch=T, width=0.2, linewidth=1, position = position_dodge(0.85)) +
    stat_compare_means(comparisons = list(c('Young', 'Old')), label = 'p.format', bracket.size = 1, method = 'wilcox.test') +
    scale_fill_manual(values = age_group_color, guide = NULL) +
#    geom_hline(yintercept = 0, linetype='dashed') +
    ylab(label = 'Positive Regulation Response Score to Tumor Cell') +
    xlab(label = NULL) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(plot.title = element_text(hjust = 0.5),
          legend.position = 'top',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

p1 + p2
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'ST_TLS_region_SignatureScore_VlnPlot.pdf'), device = 'pdf', width = 10, height = 5, bg = 'transparent')

## Neighborhood_Analysis(RCTD_based)

In [ ]:
library('spacexr')

### Single-Cell Reference

In [ ]:
#read
samples_all_annotation <- readRDS(paste0(obj, '/', 'samples_all_annotation.rds'))

In [ ]:
Idents(samples_all_annotation) <- samples_all_annotation$`lung_condition`
samples_all_annotation_diseased <- subset(samples_all_annotation, idents = 'Tumor')
samples_all_annotation_diseased

In [ ]:
# get the counts for young and old groups
Idents(samples_all_annotation_diseased) <- samples_all_annotation_diseased$Age_type
samples_all_annotation_diseased_young <- subset(samples_all_annotation_diseased, idents = 'Young')
samples_all_annotation_diseased_old <- subset(samples_all_annotation_diseased, idents = 'Old')

#counts
samples_all_annotation_diseased_young_counts <- GetAssayData(samples_all_annotation_diseased_young, slot = 'counts', assay = 'RNA')
samples_all_annotation_diseased_old_counts <- GetAssayData(samples_all_annotation_diseased_old, slot = 'counts', assay = 'RNA')

In [ ]:
# get the cell_types for RCTD
samples_all_annotation_diseased_young_CellType_subcluster <- as.factor(samples_all_annotation_diseased_young$'sub_anno')
names(samples_all_annotation_diseased_young_CellType_subcluster) <- Cells(samples_all_annotation_diseased_young)

samples_all_annotation_diseased_old_CellType_subcluster <- as.factor(samples_all_annotation_diseased_old$'sub_anno')
names(samples_all_annotation_diseased_old_CellType_subcluster) <- Cells(samples_all_annotation_diseased_old)


samples_all_annotation_diseased_young_CellType_maincluster <- as.factor(samples_all_annotation_diseased_young$'main_anno')
names(samples_all_annotation_diseased_young_CellType_maincluster) <- Cells(samples_all_annotation_diseased_young)

samples_all_annotation_diseased_old_CellType_maincluster <- as.factor(samples_all_annotation_diseased_old$'main_anno')
names(samples_all_annotation_diseased_old_CellType_maincluster) <- Cells(samples_all_annotation_diseased_old)


In [ ]:
#Create the Reference object
RCTD_reference_diseased_young_subcluster <- Reference(samples_all_annotation_diseased_young_counts, samples_all_annotation_diseased_young_CellType_subcluster)
RCTD_reference_diseased_old_subcluster <- Reference(samples_all_annotation_diseased_old_counts, samples_all_annotation_diseased_old_CellType_subcluster)

RCTD_reference_diseased_young_maincluster <- Reference(samples_all_annotation_diseased_young_counts, samples_all_annotation_diseased_young_CellType_maincluster)
RCTD_reference_diseased_old_maincluster <- Reference(samples_all_annotation_diseased_old_counts, samples_all_annotation_diseased_old_CellType_maincluster)


### Spatial Transcriptomics data

In [ ]:
# get the counts for young and old groups
Idents(all_8samples_st) <- all_8samples_st$Age_type
all_8samples_st_young <- subset(all_8samples_st, idents = 'Young')
all_8samples_st_old <- subset(all_8samples_st, idents = 'Old')

#counts
all_8samples_st_young_counts <- GetAssayData(all_8samples_st_young, slot = 'counts', assay = 'Spatial')
all_8samples_st_old_counts <- GetAssayData(all_8samples_st_old, slot = 'counts', assay = 'Spatial')

In [ ]:
# coord
all_8samples_st_young_coords <- FetchData(all_8samples_st_young, vars = c('imagerow','imagecol')) %>% rename(c('x' = 'imagecol', 'y' = 'imagerow'))
all_8samples_st_old_coords <- FetchData(all_8samples_st_old, vars = c('imagerow','imagecol')) %>% rename(c('x' = 'imagecol', 'y' = 'imagerow'))

In [ ]:
### Create SpatialRNA object
RCTD_st_young <- SpatialRNA(coords = all_8samples_st_young_coords, counts = all_8samples_st_young_counts)
RCTD_st_old <- SpatialRNA(coords = all_8samples_st_old_coords, counts = all_8samples_st_old_counts)

### Running RCTD

#### Subcluster

In [ ]:
#create
RCTD_young_subcluster <- create.RCTD(spatialRNA = RCTD_st_young,
                                     reference = RCTD_reference_diseased_young_subcluster,
                                     max_cores = 8, CELL_MIN_INSTANCE = 10)

RCTD_old_subcluster <- create.RCTD(spatialRNA = RCTD_st_old,
                                   reference = RCTD_reference_diseased_old_subcluster,
                                   max_cores = 8, CELL_MIN_INSTANCE = 10)


In [ ]:
#run
RCTD_young_subcluster <- run.RCTD(RCTD_young_subcluster, doublet_mode = 'full')

RCTD_old_subcluster <- run.RCTD(RCTD_old_subcluster, doublet_mode = 'full')


In [ ]:
#save
saveRDS(RCTD_young_subcluster, paste0(obj, '/', 'RCTD_FullMode_diseased_young_subcluster.rds'))
saveRDS(RCTD_old_subcluster, paste0(obj, '/', 'RCTD_FullMode_diseased_old_subcluster.rds'))

In [ ]:
RCTD_young_subcluster <- readRDS(paste0(obj, '/', 'RCTD_FullMode_diseased_young_subcluster.rds'))
RCTD_old_subcluster <- readRDS(paste0(obj, '/', 'RCTD_FullMode_diseased_old_subcluster.rds'))

#### Maincluster

In [ ]:
#create
RCTD_young_maincluster <- create.RCTD(spatialRNA = RCTD_st_young,
                                     reference = RCTD_reference_diseased_young_maincluster,
                                     max_cores = 8, CELL_MIN_INSTANCE = 10)

RCTD_old_maincluster <- create.RCTD(spatialRNA = RCTD_st_old,
                                   reference = RCTD_reference_diseased_old_maincluster,
                                   max_cores = 8, CELL_MIN_INSTANCE = 10)


In [ ]:
#run
RCTD_young_maincluster <- run.RCTD(RCTD_young_maincluster, doublet_mode = 'full')

RCTD_old_maincluster <- run.RCTD(RCTD_old_maincluster, doublet_mode = 'full')

In [ ]:
#save
saveRDS(RCTD_young_maincluster, paste0(obj, '/', 'RCTD_FullMode_diseased_young_maincluster.rds'))
saveRDS(RCTD_old_maincluster, paste0(obj, '/', 'RCTD_FullMode_diseased_old_maincluster.rds'))

In [ ]:
RCTD_young_maincluster <- readRDS(paste0(obj, '/', 'RCTD_FullMode_diseased_young_maincluster.rds'))
RCTD_old_maincluster <- readRDS(paste0(obj, '/', 'RCTD_FullMode_diseased_old_maincluster.rds'))

### Exploring the full mode results

#### Subcluster

In [ ]:
RCTD_young_subcluster_weights <- RCTD_young_subcluster@results$weights
RCTD_old_subcluster_weights <- RCTD_old_subcluster@results$weights

In [ ]:
# normalize_weight 
RCTD_young_subcluster_weights <- as.data.frame(normalize_weights(RCTD_young_subcluster_weights))
RCTD_old_subcluster_weights <- as.data.frame(normalize_weights(RCTD_old_subcluster_weights))

In [ ]:
RCTD_young_subcluster_weights

In [ ]:
# concat the weight df and combine the metadata info
RCTD_subcluster_weights_df <- rbind(RCTD_young_subcluster_weights, RCTD_old_subcluster_weights) %>% mutate(barcode = rownames(.))

write_csv(RCTD_subcluster_weights_df,  paste0(obj, '/', 'RCTD_FullMode_diseased_subcluster_weights_df.csv'))


#### Maincluster

In [ ]:
RCTD_young_maincluster_weights <- RCTD_young_maincluster@results$weights
RCTD_old_maincluster_weights <- RCTD_old_maincluster@results$weights

In [ ]:
# normalize_weight 
RCTD_young_maincluster_weights <- as.data.frame(normalize_weights(RCTD_young_maincluster_weights))
RCTD_old_maincluster_weights <- as.data.frame(normalize_weights(RCTD_old_maincluster_weights))

In [ ]:
RCTD_young_maincluster_weights

In [ ]:
# concat the weight df and combine the metadata info
RCTD_maincluster_weights_df <- rbind(RCTD_young_maincluster_weights, RCTD_old_maincluster_weights) %>% mutate(barcode = rownames(.))

write_csv(RCTD_maincluster_weights_df,  paste0(obj, '/', 'RCTD_FullMode_diseased_maincluster_weights_df.csv'))


## Neighborhood_Analysis_Plotting(RCTD_based)

In [ ]:
cxcl13_cd4_based_neighbor_df <- read_csv(paste0(obj, '/', 'Neighborhood_Analysis', '/', 'cxcl13_cd4_based_neighbor_df.csv'))
cxcl13_cd8_based_neighbor_df <- read_csv(paste0(obj, '/', 'Neighborhood_Analysis', '/', 'cxcl13_cd8_based_neighbor_df.csv'))

In [ ]:
#plot
options(repr.plot.width = 12, repr.plot.height = 10)
p1 <- ggplot(data = cxcl13_cd4_based_neighbor_df,
       mapping = aes(x = Query_Cell_Type, y = Deconv_Fraction, fill = factor(Age_type, levels = c('Young', 'Old')))) + 
    stat_boxplot(geom = 'errorbar', width = 0.5, linewidth=1, position = position_dodge(0.9)) +
    geom_boxplot(notch = F, width = 0.8, linewidth=1, position = position_dodge(0.9), outlier.shape = NA) +
    stat_compare_means(mapping = aes(group = Age_type), method = 'wilcox', label = 'p.signif') +
    labs(x = 'Neighborhood Cell Type', y = 'Neighborhood Fraction', title = 'Center: CXCL13_CD4_Tex') +
    scale_x_discrete(limits = c('Naive_B', 'Memory_B')) +
    scale_fill_manual(values = age_group_color, guide = NULL) +
#    coord_cartesian(ylim = c(0,0.11)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(plot.title = element_text(hjust = 0.5),
          legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

p2 <- ggplot(data = cxcl13_cd8_based_neighbor_df,
       mapping = aes(x = Query_Cell_Type, y = Deconv_Fraction, fill = factor(Age_type, levels = c('Young', 'Old')))) +
    stat_boxplot(geom = 'errorbar', width = 0.5, linewidth=1, position = position_dodge(0.9)) +
    geom_boxplot(notch = F, width = 0.8, linewidth=1, position = position_dodge(0.9), outlier.shape = NA) +
    stat_compare_means(mapping = aes(group = Age_type), method = 'wilcox', label = 'p.signif') +
    labs(x = NULL, y = NULL,  title = 'Center: CXCL13_CD8_Tex') +
    scale_x_discrete(limits = c('Naive_B', 'Memory_B')) +
    scale_fill_manual(values = age_group_color, guide = guide_legend(title = 'Age Group', title.theme = element_text(size = 20))) +
    coord_cartesian(ylim = c(0,0.3)) +
    theme_classic(base_size = 25) +
    transparent_bg +
    theme(plot.title = element_text(hjust = 0.5),
          legend.position = 'none',
#          text = element_text(family = 'Arial'),
          axis.text = element_text(colour = 'black'),
          axis.title.x = element_text(margin = margin(15,0,0,0)),
          axis.title.y = element_text(margin = margin(0,15,0,0)))

p1 + p2 + transparent_bg
ggsave(filename = paste0(figures, '/Figures_raw', '/', 'CXCL13_CD4_CD8_T_Cells_Neighborhood_BoxPlot.pdf'), device = 'pdf', width = 12, height = 8)